In [10]:
from __future__ import annotations

WEB_DNB_M03_BASE_VERSION = "2026-09-13-M03-exact-full-support-feasible-v2-checkpoint-scope-fix"
WEB_DNB_M03_VERSION = "2026-10-01-M03-seam-safe-cprime-20m-v1"

def fit_webdnb_m03(
    *,
    mode="observed",
    y=None,
    group=None,
    gene_ids=None,
    sample_names=None,
    kappa_F=None,
    random_seed=None,
    numerical_settings=None,
    checkpoint_dir=None,
    fresh_start=False,
    resume=True,
    retry_failed_on_resume=True,
    n_jobs_exact=None,
    verbose=1,
    exact_mode="default",
    p_cutoff=None,
    fdr_cutoff=None,
    exact_top_n=None,
    exact_top_fraction=None,
    exact_gene_ids=None,
    refine_from=None,
):
    from datetime import datetime
    _started = datetime.now().astimezone()
    print(f"[start] {_started.strftime('%Y-%m-%d %I:%M:%S %p %Z')}", flush=True)
    # Capture outer call arguments under names that are never rebound inside
    # the nested runner. This prevents Python nested-scope UnboundLocalError
    # when stage variables are later rebound from checkpoints.
    _input_y = y
    _input_group = group
    _input_gene_ids = gene_ids
    _input_sample_names = sample_names
    _input_refine_from = refine_from

    # ------------------------------------------------------------------
    # Single-entry refinement resolver
    # ------------------------------------------------------------------
    # fit_webdnb_m03 handles both the initial M03 fit and every later exact
    # refinement.  When refine_from is supplied, immutable analysis settings
    # are inherited from that prior result and the existing checkpoints are
    # reused.  No second wrapper function is needed.
    if _input_refine_from is not None:
        import json as _json
        from pathlib import Path as _Path

        if not isinstance(_input_refine_from, dict):
            raise TypeError(
                "refine_from must be a result dict returned by fit_webdnb_m03."
            )

        _prior_state = _input_refine_from.get("refinement_state")
        if not isinstance(_prior_state, dict):
            raise ValueError(
                "refine_from does not contain a valid refinement_state. "
                "Use a result produced by this refinement-capable fit_webdnb_m03."
            )

        if str(mode).strip().lower() != "observed":
            raise ValueError(
                "refine_from can only be used with mode='observed'."
            )
        if bool(fresh_start):
            raise ValueError(
                "fresh_start=True cannot be combined with refine_from because "
                "refinement reuses the prior M03 checkpoints."
            )
        if not bool(resume):
            raise ValueError(
                "resume=False cannot be combined with refine_from because "
                "refinement requires the prior M03 checkpoints."
            )

        _prior_checkpoint_dir = _prior_state.get("checkpoint_dir")
        if not _prior_checkpoint_dir:
            raise ValueError(
                "refine_from does not contain a checkpoint_dir."
            )
        if checkpoint_dir is None:
            checkpoint_dir = _prior_checkpoint_dir
        else:
            _given = _Path(checkpoint_dir).expanduser().resolve()
            _prior = _Path(_prior_checkpoint_dir).expanduser().resolve()
            if _given != _prior:
                raise ValueError(
                    "checkpoint_dir does not match refine_from. Refinement must "
                    "continue from the checkpoint directory belonging to the prior fit."
                )

        _prior_random_seed = _prior_state.get("random_seed")
        if _prior_random_seed is None:
            raise ValueError(
                "refine_from does not contain random_seed."
            )
        if random_seed is None:
            random_seed = _prior_random_seed
        elif int(random_seed) != int(_prior_random_seed):
            raise ValueError(
                "random_seed does not match refine_from. The base M03 fit is frozen "
                "during recursive exact refinement."
            )

        _prior_numerical_settings = _prior_state.get("numerical_settings")
        if not isinstance(_prior_numerical_settings, dict):
            raise ValueError(
                "refine_from does not contain numerical_settings."
            )
        if numerical_settings is None:
            numerical_settings = _prior_numerical_settings
        else:
            _current_settings = _json.dumps(
                numerical_settings, sort_keys=True, default=str
            )
            _prior_settings = _json.dumps(
                _prior_numerical_settings, sort_keys=True, default=str
            )
            if _current_settings != _prior_settings:
                raise ValueError(
                    "numerical_settings do not match refine_from. The base M03 fit "
                    "cannot change during recursive exact refinement."
                )

        if n_jobs_exact is None:
            n_jobs_exact = int(_prior_state.get("n_jobs_exact", 6))
    elif n_jobs_exact is None:
        n_jobs_exact = 6

    if isinstance(n_jobs_exact, bool) or int(n_jobs_exact) < 1:
        raise ValueError("n_jobs_exact must be a positive integer.")
    n_jobs_exact = int(n_jobs_exact)

    try:
        def _run():
            """Canonical M03 plus optional selective exact normalized-DNB refinement.

            Initial call:
                supply y/group/gene_ids, numerical_settings, random_seed, and
                checkpoint_dir.

            Recursive refinement call:
                supply refine_from=<previous fit_webdnb_m03 result> plus the
                next exact_mode/options. The function inherits and verifies the
                immutable base-fit settings, reuses all completed M03 stages and
                exact-gene checkpoints, and computes only newly requested genes.

            The frozen M03 statistical base remains WEB-DP r + DEB theta + C'(1),
            mandatory exact full/null fallback for inadmissible R_F, and the
            support-feasible null geometry.  The C'(1) numerical backend now
            uses exact discrete evaluation whenever the explicit support-work
            cap permits it, reserving the continuum backend for support windows
            beyond that cap. Optional refinement modes reuse
            that frozen base and exact-profile additional genes without
            refitting any genome-wide M03 quantity.

            This function contains its own full statistical and numerical backend.
            It does not call another project fitting function.
            """
            nonlocal p_cutoff, fdr_cutoff, exact_top_n, exact_top_fraction, exact_gene_ids
            import numpy as np
            import pandas as pd
            import threading
            from scipy.interpolate import RegularGridInterpolator, LinearNDInterpolator
            from scipy.optimize import minimize, minimize_scalar, brentq
            from scipy.special import betaln, expit, gammaln, logit, logsumexp, roots_legendre, xlogy
            from scipy.stats import chi2, gamma as gamma_dist, nbinom
            from scipy.spatial import Delaunay

            method_name = "M03 final | r=WEB-DP | theta=DEB | C'(1) + exact-full fallback + support-feasible null"
            r_estimator = 'web_dp'
            theta_method = 'deb'
            geometry_method = 'cprime1'

            if mode == "describe":
                return {
                    "method_name": method_name,
                    "method_number": 3,
                    "r_estimator": r_estimator,
                    "theta_method": theta_method,
                    "geometry_method": geometry_method,
                    "geometry_label": "C'(1) with exact full fallback + support-feasible null",
                    "numerical_backend": "exact_discrete_when_feasible_seam_safe_v1",
                    "exact_refinement_modes": (
                        "default", "called", "called_buffer", "top_n",
                        "top_fraction", "all", "gene_list"
                    ),
                    "base_version": WEB_DNB_M03_BASE_VERSION,
                    "code_version": WEB_DNB_M03_VERSION,
                }


            # Every tunable numerical control is supplied explicitly by the caller.
            # Method identity is fixed by this function; numerical tuning is not.
            if not isinstance(numerical_settings, dict):
                raise TypeError("numerical_settings must be a dict supplied explicitly by the caller.")

            required_numerical_settings = (
                "use_tmm",
                "alpha_floor",
                "alpha_cap",
                "deb_dp_maxiter",
                "deb_dp_flex_gamma_bounds",
                "theta_floor",
                "theta_cap",
                "mean_fit_maxiter",
                "mean_fit_tol",
                "web_maxiter",
                "mixture_maxiter",
                "ash_n_scales",
                "ash_em_maxiter",
                "ash_em_tol",
                "ash_prior_pseudocount",
                "cprime_grid_n_mu",
                "cprime_grid_n_r",
                "cprime_grid_max_refinements",
                "cprime_grid_emergency_refinements",
                "cprime_audit_n",
                "cprime_audit_tol",
                "theta_cprime_order",
                "theta_cprime_audit_n",
                "theta_cprime_audit_tol",
                "theta_cprime_max_abs_delta",
                "cprime_tail_probability",
                "cprime_support_max",
                "cprime_expect_maxcount",
                "cprime_chunk_size",
                "cprime_grid_exact_width_max",
                "cprime_gamma_quad_n",
                "cprime_gamma_min_mu_over_r",
                "cprime_continuum_audit_quad_n",
                "cprime_continuum_audit_quad_n_max",
                "cprime_continuum_audit_min_mu",
                "cprime_continuum_audit_min_mu_over_r",
                "cprime_continuum_audit_selfcheck_tol",
                "corrected_score_tolerance",
                "deb_precision_maxiter",
                "exact_theta_search_min",
                "exact_theta_search_max",
                "exact_score_root_xtol",
                "exact_score_root_rtol",
                "exact_score_zero_tol",
                "exact_profile_local_step",
                "support_floor_tolerance",
            )
            missing_settings = [
                key for key in required_numerical_settings if key not in numerical_settings
            ]
            extra_settings = sorted(
                set(numerical_settings).difference(required_numerical_settings)
            )
            if missing_settings:
                raise ValueError(
                    "numerical_settings is missing required keys: "
                    + ", ".join(missing_settings)
                )
            if extra_settings:
                raise ValueError(
                    "numerical_settings contains unrecognized keys: "
                    + ", ".join(extra_settings)
                )

            use_tmm = bool(numerical_settings["use_tmm"])
            alpha_floor = float(numerical_settings["alpha_floor"])
            alpha_cap = float(numerical_settings["alpha_cap"])
            deb_dp_maxiter = int(numerical_settings["deb_dp_maxiter"])
            deb_dp_flex_gamma_bounds = tuple(
                float(x) for x in numerical_settings["deb_dp_flex_gamma_bounds"]
            )
            theta_floor = float(numerical_settings["theta_floor"])
            theta_cap = float(numerical_settings["theta_cap"])
            mean_fit_maxiter = int(numerical_settings["mean_fit_maxiter"])
            mean_fit_tol = float(numerical_settings["mean_fit_tol"])
            web_maxiter = int(numerical_settings["web_maxiter"])
            mixture_maxiter = int(numerical_settings["mixture_maxiter"])
            ash_n_scales = int(numerical_settings["ash_n_scales"])
            ash_em_maxiter = int(numerical_settings["ash_em_maxiter"])
            ash_em_tol = float(numerical_settings["ash_em_tol"])
            ash_prior_pseudocount = float(numerical_settings["ash_prior_pseudocount"])
            cprime_grid_n_mu = int(numerical_settings["cprime_grid_n_mu"])
            cprime_grid_n_r = int(numerical_settings["cprime_grid_n_r"])
            cprime_grid_max_refinements = int(
                numerical_settings["cprime_grid_max_refinements"]
            )
            cprime_grid_emergency_refinements = int(
                numerical_settings["cprime_grid_emergency_refinements"]
            )
            cprime_audit_n = int(numerical_settings["cprime_audit_n"])
            cprime_audit_tol = float(numerical_settings["cprime_audit_tol"])
            theta_cprime_order = int(numerical_settings["theta_cprime_order"])
            theta_cprime_audit_n = int(numerical_settings["theta_cprime_audit_n"])
            theta_cprime_audit_tol = float(numerical_settings["theta_cprime_audit_tol"])
            theta_cprime_max_abs_delta = float(
                numerical_settings["theta_cprime_max_abs_delta"]
            )
            cprime_tail_probability = float(numerical_settings["cprime_tail_probability"])
            cprime_support_max = int(numerical_settings["cprime_support_max"])
            cprime_expect_maxcount = int(numerical_settings["cprime_expect_maxcount"])
            cprime_chunk_size = int(numerical_settings["cprime_chunk_size"])
            cprime_grid_exact_width_max = int(
                numerical_settings["cprime_grid_exact_width_max"]
            )
            cprime_gamma_quad_n = int(numerical_settings["cprime_gamma_quad_n"])
            cprime_gamma_min_mu_over_r = float(
                numerical_settings["cprime_gamma_min_mu_over_r"]
            )
            cprime_continuum_audit_quad_n = int(
                numerical_settings["cprime_continuum_audit_quad_n"]
            )
            cprime_continuum_audit_quad_n_max = int(
                numerical_settings["cprime_continuum_audit_quad_n_max"]
            )
            cprime_continuum_audit_min_mu = float(
                numerical_settings["cprime_continuum_audit_min_mu"]
            )
            cprime_continuum_audit_min_mu_over_r = float(
                numerical_settings["cprime_continuum_audit_min_mu_over_r"]
            )
            cprime_continuum_audit_selfcheck_tol = float(
                numerical_settings["cprime_continuum_audit_selfcheck_tol"]
            )
            corrected_score_tolerance = float(
                numerical_settings["corrected_score_tolerance"]
            )
            deb_precision_maxiter = int(numerical_settings["deb_precision_maxiter"])
            exact_theta_search_min = float(numerical_settings["exact_theta_search_min"])
            exact_theta_search_max = float(numerical_settings["exact_theta_search_max"])
            exact_score_root_xtol = float(numerical_settings["exact_score_root_xtol"])
            exact_score_root_rtol = float(numerical_settings["exact_score_root_rtol"])
            exact_score_zero_tol = float(numerical_settings["exact_score_zero_tol"])
            exact_profile_local_step = float(numerical_settings["exact_profile_local_step"])
            support_floor_tolerance = float(numerical_settings["support_floor_tolerance"])

            if int(cprime_chunk_size) < 1_000:
                raise ValueError("cprime_chunk_size must be >= 1000.")
            if int(cprime_grid_exact_width_max) < 10_000:
                raise ValueError("cprime_grid_exact_width_max must be >= 10000.")
            if int(cprime_gamma_quad_n) < 32:
                raise ValueError("cprime_gamma_quad_n must be >= 32.")
            if int(cprime_continuum_audit_quad_n) < 64:
                raise ValueError("cprime_continuum_audit_quad_n must be >= 64.")
            if int(cprime_continuum_audit_quad_n_max) < int(cprime_continuum_audit_quad_n):
                raise ValueError(
                    "cprime_continuum_audit_quad_n_max must be >= "
                    "cprime_continuum_audit_quad_n."
                )
            if int(cprime_grid_emergency_refinements) < 0:
                raise ValueError("cprime_grid_emergency_refinements must be >= 0.")

            # ------------------------------------------------------------------
            # Basic utilities
            # ------------------------------------------------------------------

            def bh_adjust(p):
                p = np.asarray(p, dtype=float)
                q = np.full_like(p, np.nan)
                ok = np.isfinite(p) & (p >= 0) & (p <= 1)
                if not np.any(ok):
                    return q
                p0 = p[ok]
                order = np.argsort(p0, kind="stable")
                ranked = p0[order]
                n = len(ranked)
                adjusted = ranked * n / np.arange(1, n + 1)
                adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
                adjusted = np.minimum(adjusted, 1.0)
                restored = np.empty_like(adjusted)
                restored[order] = adjusted
                q[ok] = restored
                return q

            def half_poisson_deviance(y, mu):
                y = np.asarray(y, dtype=float)
                mu = np.maximum(np.asarray(mu, dtype=float), 1e-12)
                return np.maximum(xlogy(y, y / mu) - (y - mu), 0.0)

            def half_nb_deviance(y, mu, r):
                y = np.asarray(y, dtype=float)
                mu = np.maximum(np.asarray(mu, dtype=float), 1e-12)
                r = np.maximum(np.asarray(r, dtype=float), 1e-12)
                return np.maximum(
                    xlogy(y, y / mu) - (y + r) * np.log((y + r) / (mu + r)),
                    0.0,
                )

            def modified_web_statistic(S_F, S_0, theta_F, theta_0, n):
                S_F = np.asarray(S_F, dtype=float)
                S_0 = np.asarray(S_0, dtype=float)
                theta_F = np.asarray(theta_F, dtype=float)
                theta_0 = np.asarray(theta_0, dtype=float)
                T = 2.0 * (theta_0 * S_0 - theta_F * S_F) + float(n) * np.log(theta_F / theta_0)
                T = np.where(np.isfinite(T), T, np.nan)
                return np.maximum(T, 0.0)

            def calc_tmm_factors(y):
                y = np.asarray(y, dtype=float)
                lib = y.sum(axis=0)
                if np.any(lib <= 0):
                    raise ValueError("All selected samples must have positive library size.")

                uq = np.zeros(y.shape[1])
                for j in range(y.shape[1]):
                    pos = y[:, j] > 0
                    if np.any(pos):
                        uq[j] = np.quantile(y[pos, j] / lib[j], 0.75)

                positive_uq = uq > 0
                if np.any(positive_uq):
                    target = float(np.mean(uq[positive_uq]))
                    ref_index = int(np.argmin(np.abs(uq - target)))
                else:
                    ref_index = int(np.argmin(np.abs(lib - np.mean(lib))))

                ref = y[:, ref_index]
                ref_lib = lib[ref_index]
                factors = np.ones(y.shape[1])

                for j in range(y.shape[1]):
                    if j == ref_index:
                        continue
                    obs = y[:, j]
                    obs_lib = lib[j]
                    keep = (obs > 0) & (ref > 0)
                    if keep.sum() < 10:
                        continue
                    obs_k = obs[keep]
                    ref_k = ref[keep]
                    M = np.log2((obs_k / obs_lib) / (ref_k / ref_lib))
                    A = 0.5 * np.log2((obs_k / obs_lib) * (ref_k / ref_lib))
                    V = (
                        (obs_lib - obs_k) / (obs_lib * obs_k)
                        + (ref_lib - ref_k) / (ref_lib * ref_k)
                    )
                    finite = np.isfinite(M) & np.isfinite(A) & np.isfinite(V) & (V > 0)
                    M, A, V = M[finite], A[finite], V[finite]
                    if len(M) < 10:
                        continue
                    m_lo, m_hi = np.quantile(M, [0.30, 0.70])
                    a_lo, a_hi = np.quantile(A, [0.05, 0.95])
                    trim = (M >= m_lo) & (M <= m_hi) & (A >= a_lo) & (A <= a_hi)
                    if trim.sum() < 5:
                        continue
                    w = 1.0 / V[trim]
                    factors[j] = 2.0 ** (np.sum(w * M[trim]) / np.sum(w))

                factors /= np.exp(np.mean(np.log(np.maximum(factors, 1e-300))))
                return factors

            # ------------------------------------------------------------------
            # Mean fits
            # ------------------------------------------------------------------

            def fit_poisson_group_means(y, exposure, group):
                mu = np.zeros_like(y, dtype=float)
                group = np.asarray(group)
                for level in np.unique(group):
                    idx = np.flatnonzero(group == level)
                    q = y[:, idx].sum(axis=1) / exposure[idx].sum()
                    mu[:, idx] = q[:, None] * exposure[idx][None, :]
                return np.maximum(mu, 1e-12)

            def solve_nb_base_mean(y, exposure, alpha_g):
                y = np.asarray(y, dtype=float)
                exposure = np.asarray(exposure, dtype=float)
                alpha_g = np.asarray(alpha_g, dtype=float)
                q = np.maximum(y.sum(axis=1) / exposure.sum(), 1e-12)

                for _ in range(int(mean_fit_maxiter)):
                    mu = q[:, None] * exposure[None, :]
                    denom = 1.0 + alpha_g[:, None] * mu
                    score = np.sum((y - mu) / denom, axis=1)
                    deriv = -np.sum(
                        exposure[None, :] * (1.0 + alpha_g[:, None] * y) / denom**2,
                        axis=1,
                    )
                    deriv = np.minimum(deriv, -1e-14)
                    q_new = q - score / deriv
                    bad = ~np.isfinite(q_new) | (q_new <= 0)
                    q_new[bad] = q[bad] / 2.0
                    q_new = np.maximum(q_new, 1e-12)
                    change = np.max(np.abs(q_new - q) / np.maximum(np.abs(q), 1e-8))
                    q = q_new
                    if np.isfinite(change) and change < float(mean_fit_tol):
                        break
                return q

            def fit_nb_group_means(y, exposure, group, alpha_g):
                mu = np.zeros_like(y, dtype=float)
                q_by_group = {}
                group = np.asarray(group)
                for level in np.unique(group):
                    idx = np.flatnonzero(group == level)
                    q = solve_nb_base_mean(y[:, idx], exposure[idx], alpha_g)
                    q_by_group[str(level)] = q
                    mu[:, idx] = q[:, None] * exposure[idx][None, :]
                return np.maximum(mu, 1e-12), q_by_group

            # ------------------------------------------------------------------
            # WEB
            # ------------------------------------------------------------------

            def generalized_beta_logpdf(S, a, prior_shape, prior_rate):
                return (
                    (a - 1.0) * np.log(S)
                    - a * np.log(prior_rate)
                    - (a + prior_shape) * np.log1p(S / prior_rate)
                    - (gammaln(a) + gammaln(prior_shape) - gammaln(a + prior_shape))
                )

            def estimate_web_to_target(S, a, target_S):
                S = np.maximum(np.asarray(S, dtype=float), 1e-12)
                target_S = np.maximum(np.asarray(target_S, dtype=float), 1e-12)
                a_vec = np.full(len(S), float(a)) if np.isscalar(a) else np.asarray(a, dtype=float)

                def objective(weight):
                    if not 0.0 < weight < 1.0:
                        return np.inf
                    delta = weight / (1.0 - weight)
                    ll = generalized_beta_logpdf(S, a_vec, delta * a_vec, delta * target_S)
                    return -float(np.mean(ll)) if np.all(np.isfinite(ll)) else np.inf

                fit = minimize_scalar(
                    objective,
                    bounds=(1e-9, 1.0 - 1e-9),
                    method="bounded",
                    options={"xatol": 1e-10, "maxiter": int(web_maxiter)},
                )
                if not fit.success or not np.isfinite(fit.fun):
                    raise RuntimeError("WEB-to-target optimization failed.")
                weight = float(fit.x)
                if weight > 1.0 - 1e-6:
                    weight = 1.0
                return {"weight": weight, "S_shrunk": (1.0 - weight) * S + weight * target_S}

            def estimate_web(S, a):
                S = np.asarray(S, dtype=float)
                a_vec = np.full(len(S), float(a)) if np.isscalar(a) else np.asarray(a, dtype=float)
                ok = np.isfinite(S) & np.isfinite(a_vec) & (S > 0) & (a_vec > 0)
                if ok.sum() < 100:
                    raise RuntimeError("Too few usable genes for WEB EB.")
                theta0 = float(a_vec[ok].sum() / S[ok].sum())
                target_S = a_vec / theta0
                web = estimate_web_to_target(S, a_vec, target_S)
                theta = a_vec / np.maximum(web["S_shrunk"], 1e-12)
                theta = np.clip(theta, float(theta_floor), float(theta_cap))
                return {"theta": theta, "weight": web["weight"], "theta0": theta0}

            def estimate_web_masked(S, a, label):
                S = np.asarray(S, dtype=float)
                valid = np.isfinite(S) & (S > float(corrected_score_tolerance))
                if valid.sum() < 100:
                    raise RuntimeError(
                        f"{label}: only {int(valid.sum())} positive corrected scores are available."
                    )
                fit = estimate_web(S[valid], a)
                theta = np.full(len(S), np.nan)
                theta[valid] = fit["theta"]
                return {
                    "theta": theta,
                    "weight": fit["weight"],
                    "theta0": fit["theta0"],
                    "valid_mask": valid,
                    "n_valid": int(valid.sum()),
                    "n_invalid": int((~valid).sum()),
                    "invalid_fraction": float((~valid).mean()),
                }

            # ------------------------------------------------------------------
            # r_g = DP residual information + WEB
            # ------------------------------------------------------------------

            def fit_dp_trend(S, a, mean_mu):
                S = np.asarray(S, dtype=float)
                mean_mu = np.asarray(mean_mu, dtype=float)
                ok = np.isfinite(S) & np.isfinite(mean_mu) & (S > 0) & (mean_mu > 0)
                if ok.sum() < 100:
                    raise RuntimeError("Too few genes for DP trend fit.")
                S0 = S[ok]
                mu0 = mean_mu[ok]
                phi_raw = S0 / float(a)
                slope0 = max(
                    float(np.nanmedian(np.maximum(phi_raw - 1.0, 0.0) / np.maximum(mu0, 1e-8))),
                    1e-8,
                )
                intercept0 = max(
                    float(np.nanmedian(np.maximum(phi_raw - 1.0 - slope0 * mu0, 0.0))),
                    1e-8,
                )

                def objective(par):
                    intercept = np.exp(par[0])
                    alpha_inf = np.exp(par[1])
                    phi = 1.0 + intercept + alpha_inf * mu0
                    return float(np.sum(float(a) * np.log(phi) + S0 / phi))

                fit = minimize(
                    objective,
                    x0=np.log([intercept0, slope0]),
                    method="BFGS",
                    options={"maxiter": 500, "gtol": 1e-8},
                )
                intercept = float(np.exp(fit.x[0]))
                alpha_inf = float(np.exp(fit.x[1]))
                phi_trend = np.maximum(1.0 + intercept + alpha_inf * mean_mu, 1.0 + 1e-10)
                return {
                    "intercept": intercept,
                    "alpha_inf": alpha_inf,
                    "phi_trend": phi_trend,
                    "success": bool(fit.success),
                }

            def _dp_common_quantities(y, exposure, group):
                n = y.shape[1]
                d_F = n - len(np.unique(group))
                a_F = d_F / 2.0
                mu_P = fit_poisson_group_means(y, exposure, group)
                S_DP = half_poisson_deviance(y, mu_P).sum(axis=1)
                mean_mu = mu_P.mean(axis=1)
                phi_raw = S_DP / float(a_F)
                trend = fit_dp_trend(S_DP, a_F, mean_mu)
                return {
                    "a_F": float(a_F),
                    "mu_P": mu_P,
                    "S_DP": S_DP,
                    "mean_mu": mean_mu,
                    "phi_raw": phi_raw,
                    "trend": trend,
                }

            def _finish_r_from_phi(phi_hat, mean_mu):
                phi_hat = np.asarray(phi_hat, dtype=float)
                mean_mu = np.asarray(mean_mu, dtype=float)
                alpha_hat = (phi_hat - 1.0) / np.maximum(mean_mu, 1e-8)
                alpha_hat = np.clip(alpha_hat, float(alpha_floor), float(alpha_cap))
                r_hat = 1.0 / alpha_hat
                return alpha_hat, r_hat

            def estimate_r_dp_web(y, exposure, group):
                common = _dp_common_quantities(y, exposure, group)
                a_F = common["a_F"]
                S_DP = common["S_DP"]
                mean_mu = common["mean_mu"]
                trend = common["trend"]
                phi_trend = trend["phi_trend"]
                target = a_F * phi_trend
                web = estimate_web_to_target(S_DP, a_F, target)
                phi_hat = web["S_shrunk"] / a_F
                alpha_hat, r_hat = _finish_r_from_phi(phi_hat, mean_mu)
                return {
                    "r_estimator": "web_dp",
                    "r_hat": r_hat,
                    "alpha_hat": alpha_hat,
                    "S_DP": S_DP,
                    "mean_mu_DP": mean_mu,
                    "phi_raw_DP": common["phi_raw"],
                    "phi_trend_DP": phi_trend,
                    "phi_hat_DP": phi_hat,
                    "WEB_weight_r": web["weight"],
                    "DEB_DP_alpha0": np.nan,
                    "DEB_DP_implied_weight": np.nan,
                    "flex_intercept": np.nan,
                    "flex_a_ref": np.nan,
                    "flex_gamma": np.nan,
                    "flex_mu_ref": np.nan,
                    "trend_intercept": trend["intercept"],
                    "trend_alpha_inf": trend["alpha_inf"],
                    "trend_success": trend["success"],
                }

            def _fit_matched_deb_dp_shape(S_DP, a_F, phi_center):
                S_DP = np.asarray(S_DP, dtype=float)
                phi_center = np.asarray(phi_center, dtype=float)
                ok = (
                    np.isfinite(S_DP) & (S_DP > 0)
                    & np.isfinite(phi_center) & (phi_center > 0)
                )
                if ok.sum() < 100:
                    raise RuntimeError("Too few usable genes for matched DEB-DP.")
                x = S_DP[ok]
                center = phi_center[ok]

                def objective(log_alpha0):
                    alpha0 = float(np.exp(log_alpha0))
                    beta = alpha0 * center
                    ll = (
                        (float(a_F) - 1.0) * np.log(x)
                        - float(a_F) * np.log(beta)
                        - (float(a_F) + alpha0) * np.log1p(x / beta)
                        - betaln(float(a_F), alpha0)
                    )
                    return -float(np.mean(ll)) if np.all(np.isfinite(ll)) else np.inf

                fit = minimize_scalar(
                    objective,
                    bounds=(np.log(1e-8), np.log(1e9)),
                    method="bounded",
                    options={"xatol": 1e-10, "maxiter": int(deb_dp_maxiter)},
                )
                if not fit.success or not np.isfinite(fit.fun):
                    raise RuntimeError("Matched trend-DEB-DP marginal-likelihood fit failed.")
                alpha0 = float(np.exp(fit.x))
                beta = alpha0 * phi_center
                theta_post = (float(a_F) + alpha0) / (S_DP + beta)
                phi_hat = 1.0 / np.maximum(theta_post, 1e-300)
                implied_weight = alpha0 / (float(a_F) + alpha0)
                return {
                    "alpha0": alpha0,
                    "beta": beta,
                    "theta_post": theta_post,
                    "phi_hat": phi_hat,
                    "implied_weight": float(implied_weight),
                    "objective": float(fit.fun),
                }

            def estimate_r_dp_deb_matched(y, exposure, group):
                common = _dp_common_quantities(y, exposure, group)
                a_F = common["a_F"]
                trend = common["trend"]
                phi_trend = trend["phi_trend"]
                deb = _fit_matched_deb_dp_shape(common["S_DP"], a_F, phi_trend)
                phi_hat = deb["phi_hat"]
                alpha_hat, r_hat = _finish_r_from_phi(phi_hat, common["mean_mu"])
                return {
                    "r_estimator": "deb_dp_matched",
                    "r_hat": r_hat,
                    "alpha_hat": alpha_hat,
                    "S_DP": common["S_DP"],
                    "mean_mu_DP": common["mean_mu"],
                    "phi_raw_DP": common["phi_raw"],
                    "phi_trend_DP": phi_trend,
                    "phi_hat_DP": phi_hat,
                    "WEB_weight_r": np.nan,
                    "DEB_DP_alpha0": deb["alpha0"],
                    "DEB_DP_implied_weight": deb["implied_weight"],
                    "DEB_DP_objective": deb["objective"],
                    "flex_intercept": np.nan,
                    "flex_a_ref": np.nan,
                    "flex_gamma": np.nan,
                    "flex_mu_ref": np.nan,
                    "trend_intercept": trend["intercept"],
                    "trend_alpha_inf": trend["alpha_inf"],
                    "trend_success": trend["success"],
                }

            def estimate_r_dp_deb_flexible(y, exposure, group):
                """
                Flexible trend-DEB-DP.

                The DEB prior mean Fano factor is learned as

                    phi_center(mu) = 1 + b + a_ref * (mu / mu_ref)**gamma,

                rather than being forced to the NB-like linear mean-Fano trend.
                gamma=1 recovers a linear-in-mean excess Fano component (up to the
                reference-scale reparameterization); gamma near zero approaches a
                nearly mean-independent extra Fano component.  Prior strength alpha0
                and all trend parameters are estimated jointly by marginal likelihood.
                """
                common = _dp_common_quantities(y, exposure, group)
                a_F = float(common["a_F"])
                S_DP = np.asarray(common["S_DP"], dtype=float)
                mean_mu = np.asarray(common["mean_mu"], dtype=float)
                base_trend = common["trend"]
                ok = np.isfinite(S_DP) & (S_DP > 0) & np.isfinite(mean_mu) & (mean_mu > 0)
                if ok.sum() < 100:
                    raise RuntimeError("Too few usable genes for flexible trend-DEB-DP.")
                xS = S_DP[ok]
                xmu = mean_mu[ok]
                mu_ref = float(np.exp(np.mean(np.log(np.maximum(xmu, 1e-12)))))
                mu_scaled = np.maximum(xmu / mu_ref, 1e-12)

                b0 = max(float(base_trend["intercept"]), 1e-8)
                a_ref0 = max(float(base_trend["alpha_inf"]) * mu_ref, 1e-8)

                gamma_lo, gamma_hi = map(float, deb_dp_flex_gamma_bounds)
                if not (0 < gamma_lo < gamma_hi):
                    raise ValueError("deb_dp_flex_gamma_bounds must satisfy 0 < low < high.")

                def unpack(par):
                    alpha0 = float(np.exp(par[0]))
                    b = float(np.exp(par[1]))
                    a_ref = float(np.exp(par[2]))
                    gamma = float(np.exp(par[3]))
                    return alpha0, b, a_ref, gamma

                def objective(par):
                    alpha0, b, a_ref, gamma = unpack(par)
                    phi_center = 1.0 + b + a_ref * mu_scaled**gamma
                    beta = alpha0 * phi_center
                    ll = (
                        (a_F - 1.0) * np.log(xS)
                        - a_F * np.log(beta)
                        - (a_F + alpha0) * np.log1p(xS / beta)
                        - betaln(a_F, alpha0)
                    )
                    return -float(np.mean(ll)) if np.all(np.isfinite(ll)) else 1e100

                bounds = [
                    (np.log(1e-8), np.log(1e9)),
                    (np.log(1e-10), np.log(1e4)),
                    (np.log(1e-10), np.log(1e6)),
                    (np.log(gamma_lo), np.log(gamma_hi)),
                ]
                starts = []
                for alpha0_start in (0.2, 1.0, 5.0, 25.0, 100.0):
                    for gamma_start in (0.5, 1.0, 1.5):
                        gamma_start = float(np.clip(gamma_start, gamma_lo * 1.001, gamma_hi * 0.999))
                        starts.append(np.log([alpha0_start, b0, a_ref0, gamma_start]))

                fits = []
                for start in starts:
                    fit = minimize(
                        objective,
                        x0=start,
                        method="L-BFGS-B",
                        bounds=bounds,
                        options={"maxiter": int(deb_dp_maxiter), "ftol": 1e-12},
                    )
                    if np.isfinite(fit.fun):
                        fits.append(fit)
                if not fits:
                    raise RuntimeError("Flexible trend-DEB-DP marginal-likelihood fit failed.")
                best = min(fits, key=lambda f: f.fun)
                alpha0, b, a_ref, gamma = unpack(best.x)

                phi_center_all = 1.0 + b + a_ref * np.maximum(mean_mu / mu_ref, 1e-12)**gamma
                beta_all = alpha0 * phi_center_all
                theta_post = (a_F + alpha0) / (S_DP + beta_all)
                phi_hat = 1.0 / np.maximum(theta_post, 1e-300)
                alpha_hat, r_hat = _finish_r_from_phi(phi_hat, mean_mu)
                implied_weight = alpha0 / (a_F + alpha0)

                return {
                    "r_estimator": "deb_dp_flexible",
                    "r_hat": r_hat,
                    "alpha_hat": alpha_hat,
                    "S_DP": S_DP,
                    "mean_mu_DP": mean_mu,
                    "phi_raw_DP": common["phi_raw"],
                    "phi_trend_DP": phi_center_all,
                    "phi_hat_DP": phi_hat,
                    "WEB_weight_r": np.nan,
                    "DEB_DP_alpha0": alpha0,
                    "DEB_DP_implied_weight": float(implied_weight),
                    "DEB_DP_objective": float(best.fun),
                    "flex_intercept": b,
                    "flex_a_ref": a_ref,
                    "flex_gamma": gamma,
                    "flex_mu_ref": mu_ref,
                    "flex_gamma_at_lower_bound": bool(gamma <= gamma_lo * 1.001),
                    "flex_gamma_at_upper_bound": bool(gamma >= gamma_hi * 0.999),
                    "trend_intercept": base_trend["intercept"],
                    "trend_alpha_inf": base_trend["alpha_inf"],
                    "trend_success": bool(best.success),
                }

            def estimate_r_dp(y, exposure, group):
                method = str(r_estimator).strip().lower()
                aliases = {
                    "web": "web_dp",
                    "web-dp": "web_dp",
                    "matched": "deb_dp_matched",
                    "deb_matched": "deb_dp_matched",
                    "deb-dp-matched": "deb_dp_matched",
                    "flexible": "deb_dp_flexible",
                    "deb_flexible": "deb_dp_flexible",
                    "deb-dp-flexible": "deb_dp_flexible",
                }
                method = aliases.get(method, method)
                if method == "web_dp":
                    return estimate_r_dp_web(y, exposure, group)
                if method == "deb_dp_matched":
                    return estimate_r_dp_deb_matched(y, exposure, group)
                if method == "deb_dp_flexible":
                    return estimate_r_dp_deb_flexible(y, exposure, group)
                raise ValueError(
                    "r_estimator must be one of: 'web_dp', 'deb_dp_matched', "
                    "or 'deb_dp_flexible'."
                )

            # ------------------------------------------------------------------
            # Effect, SE, signed Z
            # ------------------------------------------------------------------

            def nb_effect_and_se(mu_F, q_by_group, group, alpha_g, theta_F):
                levels = [str(x) for x in np.unique(group)]
                if len(levels) != 2:
                    raise RuntimeError("This benchmark function requires exactly two groups.")
                q_A = np.maximum(q_by_group[levels[0]], 1e-12)
                q_B = np.maximum(q_by_group[levels[1]], 1e-12)
                beta = np.log(q_B) - np.log(q_A)
                group_string = np.asarray(group).astype(str)
                idx_A = np.flatnonzero(group_string == levels[0])
                idx_B = np.flatnonzero(group_string == levels[1])
                info = mu_F / (1.0 + alpha_g[:, None] * mu_F)
                I_A = np.maximum(info[:, idx_A].sum(axis=1), 1e-12)
                I_B = np.maximum(info[:, idx_B].sum(axis=1), 1e-12)
                var0 = 1.0 / I_A + 1.0 / I_B
                theta_F = np.asarray(theta_F, dtype=float)
                se = np.sqrt(var0 / theta_F)
                z = beta / se
                return {"beta": beta, "se": se, "z": np.where(np.isfinite(z), z, np.nan)}

            # ------------------------------------------------------------------
            # Symmetric 2-Normal
            # ------------------------------------------------------------------

            def fit_symmetric_2normal(z):
                z = np.asarray(z, dtype=float)
                ok = np.isfinite(z)
                z0 = z[ok]
                if len(z0) < 200:
                    raise RuntimeError("Too few finite Z values for symmetric 2-Normal.")
                median = float(np.median(z0))
                mad = float(np.median(np.abs(z0 - median)) / 0.6744897501960817)
                if not np.isfinite(mad) or mad <= 1e-4:
                    mad = max(float(np.std(z0)), 1.0)
                sd_all = max(float(np.std(z0)), mad, 1e-3)
                mu_lo, mu_hi = np.quantile(z0, [0.01, 0.99])
                if mu_hi <= mu_lo:
                    mu_lo, mu_hi = median - 2.0, median + 2.0

                def unpack(par):
                    mu0 = float(par[0])
                    sigma0 = float(np.exp(par[1]))
                    sigma1 = float(sigma0 + np.exp(par[2]))
                    pi0 = float(expit(par[3]))
                    return mu0, sigma0, sigma1, pi0

                def objective(par):
                    mu0, sigma0, sigma1, pi0 = unpack(par)
                    log0 = np.log(pi0) - np.log(sigma0) - 0.5 * ((z0 - mu0) / sigma0) ** 2
                    log1 = np.log1p(-pi0) - np.log(sigma1) - 0.5 * ((z0 - mu0) / sigma1) ** 2
                    ll = logsumexp(np.vstack([log0, log1]), axis=0)
                    return -float(np.mean(ll)) if np.all(np.isfinite(ll)) else 1e100

                starts = [
                    (median, max(0.5 * mad, 0.2), max(sd_all, 2.0 * mad), 0.90),
                    (median, max(mad, 0.3), max(2.0 * sd_all, 3.0 * mad), 0.95),
                    (0.0, max(mad, 0.5), max(1.5 * sd_all, 2.0 * mad), 0.80),
                ]
                bounds = [
                    (float(mu_lo), float(mu_hi)),
                    (np.log(0.05), np.log(max(10.0, 4.0 * sd_all))),
                    (np.log(0.01), np.log(max(20.0, 8.0 * sd_all))),
                    (logit(0.05), logit(0.9999)),
                ]
                fits = []
                for mu_start, s0_start, s1_start, pi_start in starts:
                    gap = max(s1_start - s0_start, 0.1)
                    x0 = np.array([mu_start, np.log(s0_start), np.log(gap), logit(pi_start)])
                    fit = minimize(
                        objective,
                        x0=x0,
                        method="L-BFGS-B",
                        bounds=bounds,
                        options={"maxiter": int(mixture_maxiter), "ftol": 1e-12},
                    )
                    if np.isfinite(fit.fun):
                        fits.append(fit)
                if not fits:
                    raise RuntimeError("Symmetric 2-Normal fit failed.")
                best = min(fits, key=lambda f: f.fun)
                mu0, sigma0, sigma1, pi0 = unpack(best.x)
                log0 = np.log(pi0) - np.log(sigma0) - 0.5 * ((z0 - mu0) / sigma0) ** 2
                log1 = np.log1p(-pi0) - np.log(sigma1) - 0.5 * ((z0 - mu0) / sigma1) ** 2
                logden = logsumexp(np.vstack([log0, log1]), axis=0)
                posterior = np.full(len(z), np.nan)
                posterior[ok] = np.exp(log0 - logden)
                return {
                    "posterior_null": posterior,
                    "mu0": mu0,
                    "sigma0": sigma0,
                    "sigma1": sigma1,
                    "pi0": pi0,
                    "sigma0_at_lower_bound": bool(sigma0 <= 0.0500001),
                }

            # ------------------------------------------------------------------
            # ASH-style centered normal mixture on beta_hat, SE
            # ------------------------------------------------------------------

            def fit_ash_normal_mixture(beta, se):
                beta = np.asarray(beta, dtype=float)
                se = np.asarray(se, dtype=float)
                ok = np.isfinite(beta) & np.isfinite(se) & (se > 0)
                b = beta[ok]
                s = se[ok]
                if len(b) < 200:
                    raise RuntimeError("Too few finite beta/SE values for ASH.")
                median_se = float(np.median(s))
                beta_scale = float(np.quantile(np.abs(b), 0.995))
                min_scale = max(0.05 * median_se, 1e-6)
                max_scale = max(8.0 * median_se, 2.0 * beta_scale, 10.0 * min_scale)
                scales = np.concatenate([[0.0], np.geomspace(min_scale, max_scale, int(ash_n_scales))])
                variance = s[:, None] ** 2 + scales[None, :] ** 2
                log_density = -0.5 * (np.log(variance) + b[:, None] ** 2 / variance)
                K = len(scales)
                pi = np.full(K, 0.50 / (K - 1))
                pi[0] = 0.50
                previous_ll = -np.inf
                for _ in range(int(ash_em_maxiter)):
                    log_joint = log_density + np.log(np.maximum(pi[None, :], 1e-300))
                    log_denom = logsumexp(log_joint, axis=1)
                    responsibility = np.exp(log_joint - log_denom[:, None])
                    pi_new = responsibility.sum(axis=0) + float(ash_prior_pseudocount)
                    pi_new /= pi_new.sum()
                    ll = float(log_denom.sum())
                    if np.isfinite(previous_ll) and abs(ll - previous_ll) <= float(ash_em_tol) * max(1.0, abs(previous_ll)):
                        pi = pi_new
                        break
                    pi = pi_new
                    previous_ll = ll
                log_joint = log_density + np.log(np.maximum(pi[None, :], 1e-300))
                log_denom = logsumexp(log_joint, axis=1)
                responsibility = np.exp(log_joint - log_denom[:, None])
                posterior_null = np.full(len(beta), np.nan)
                posterior_null[ok] = responsibility[:, 0]
                return {
                    "posterior_null": posterior_null,
                    "pi": pi,
                    "scales": scales,
                    "pi0": float(pi[0]),
                    "n_active_components": int(np.sum(pi > 1e-4)),
                    "max_scale": float(scales[-1]),
                    "median_se": median_se,
                }

            # ------------------------------------------------------------------
            # Robust normalizer numerical backend
            # ------------------------------------------------------------------

            def log_nb_pmf(y, mu, r):
                y = np.asarray(y, dtype=float)
                mu = float(mu)
                r = float(r)
                if mu <= 0 or r <= 0:
                    return np.full_like(y, -np.inf, dtype=float)
                p = r / (r + mu)
                return (
                    gammaln(y + r)
                    - gammaln(r)
                    - gammaln(y + 1.0)
                    + r * np.log(p)
                    + xlogy(y, 1.0 - p)
                )

            def nb_quantile_bounds(mu, r, tail_probability=None):
                mu = float(mu)
                r = float(r)
                if tail_probability is None:
                    tail_probability = float(cprime_tail_probability)
                tail_probability = float(np.clip(tail_probability, 1e-15, 1e-4))
                side = 0.5 * tail_probability
                p = r / (r + mu)
                lo_q = nbinom.ppf(side, r, p)
                hi_q = nbinom.ppf(1.0 - side, r, p)
                if not np.isfinite(lo_q):
                    lo_q = 0.0
                if not np.isfinite(hi_q):
                    sd = np.sqrt(mu + mu * mu / r)
                    hi_q = mu + 14.0 * sd + 50.0
                lo = max(0, int(np.floor(lo_q)))
                hi = max(lo, int(np.ceil(hi_q)))
                return lo, hi

            def exact_nb_s_moments_chunked(mu, r, tail_probability=None):
                """Exact discrete NB moments of s=D/2 over a quantile-centered window."""
                mu = float(mu)
                r = float(r)
                if not np.isfinite(mu) or not np.isfinite(r) or mu <= 0 or r <= 0:
                    return (np.nan,) * 7
                lo, hi = nb_quantile_bounds(mu, r, tail_probability)
                width = hi - lo + 1
                if width > int(cprime_support_max):
                    raise RuntimeError(
                        f"Exact C'(1) discrete window requires {width:,} points for "
                        f"mu={mu:.6g}, r={r:.6g}, exceeding cprime_support_max="
                        f"{int(cprime_support_max):,}."
                    )

                mass = 0.0
                raw1 = raw2 = raw3 = raw4 = 0.0
                chunk = int(cprime_chunk_size)
                for start in range(lo, hi + 1, chunk):
                    stop = min(hi + 1, start + chunk)
                    y = np.arange(start, stop, dtype=float)
                    logp = log_nb_pmf(y, mu, r)
                    prob = np.exp(logp)
                    s = half_nb_deviance(y, mu, r)
                    mass += float(np.sum(prob))
                    raw1 += float(np.sum(prob * s))
                    s2 = s * s
                    raw2 += float(np.sum(prob * s2))
                    raw3 += float(np.sum(prob * s2 * s))
                    raw4 += float(np.sum(prob * s2 * s2))

                if not np.isfinite(mass) or mass <= 0:
                    raise RuntimeError(f"C'(1) exact mass failed for mu={mu}, r={r}.")

                mean = raw1 / mass
                m2 = raw2 / mass
                m3 = raw3 / mass
                m4 = raw4 / mass
                var = max(m2 - mean * mean, 0.0)
                third = m3 - 3.0 * mean * m2 + 2.0 * mean**3
                fourth = m4 - 4.0 * mean * m3 + 6.0 * mean * mean * m2 - 3.0 * mean**4
                fourth_cumulant = fourth - 3.0 * var * var

                return (
                    mean - 0.5,
                    0.5 - var,
                    -1.0 + third,
                    3.0 - fourth_cumulant,
                    "centered_support_sum",
                    hi,
                    mass,
                )

            gamma_quad_cache = {}

            def get_gamma_quantile_quadrature(n_quad):
                n_quad = int(n_quad)
                if n_quad < 32:
                    raise ValueError("gamma quadrature order must be at least 32.")
                if n_quad not in gamma_quad_cache:
                    nodes, weights = roots_legendre(n_quad)
                    quad_eps = max(min(float(cprime_tail_probability), 1e-10), 1e-14)
                    u = 0.5 * (nodes + 1.0) * (1.0 - 2.0 * quad_eps) + quad_eps
                    w = 0.5 * (1.0 - 2.0 * quad_eps) * weights
                    w = w / np.sum(w)
                    gamma_quad_cache[n_quad] = (u, w)
                return gamma_quad_cache[n_quad]

            def gamma_continuum_s_moments_order(mu, r, n_quad):
                """
                High-count continuum reference for the NB half-deviance moments.

                As mu/r becomes large, Y/mu converges to Gamma(shape=r, scale=1/r).
                We still evaluate the finite-mu NB half deviance at y=mu*x.  The only
                approximation is replacing the integer-valued scaled NB distribution by
                its continuous gamma limit.
                """
                mu = float(mu)
                r = float(r)
                u, w = get_gamma_quantile_quadrature(n_quad)
                x = gamma_dist.ppf(u, a=r, scale=1.0 / r)
                y = mu * x
                s = half_nb_deviance(y, mu, r)
                mean = float(np.sum(w * s))
                centered = s - mean
                var = float(np.sum(w * centered**2))
                third = float(np.sum(w * centered**3))
                fourth = float(np.sum(w * centered**4))
                fourth_cumulant = fourth - 3.0 * var * var
                return np.array([
                    mean - 0.5,
                    0.5 - var,
                    -1.0 + third,
                    3.0 - fourth_cumulant,
                ], dtype=float)

            def gamma_continuum_s_moments(mu, r):
                values = gamma_continuum_s_moments_order(
                    mu, r, int(cprime_gamma_quad_n)
                )
                return (
                    float(values[0]), float(values[1]), float(values[2]), float(values[3]),
                    "gamma_continuum", np.nan, 1.0,
                )

            def continuum_audit_allowed(mu, r):
                mu = float(mu)
                r = float(r)
                return bool(
                    np.isfinite(mu) and np.isfinite(r)
                    and mu >= float(cprime_continuum_audit_min_mu)
                    and r > 0.0
                    and (mu / r) >= float(cprime_continuum_audit_min_mu_over_r)
                )

            def gamma_continuum_s_moments_audit(mu, r):
                """
                Independent high-order audit reference for support windows too wide to
                enumerate discretely.  The quadrature order is increased adaptively
                until the frozen self-check tolerance is met or the caller-supplied
                maximum order is exhausted.  The tolerance itself is never relaxed.
                """
                n_fine = int(cprime_continuum_audit_quad_n)
                n_max = int(cprime_continuum_audit_quad_n_max)
                tol = float(cprime_continuum_audit_selfcheck_tol)

                while True:
                    n_coarse = max(64, n_fine // 2)
                    coarse = gamma_continuum_s_moments_order(mu, r, n_coarse)
                    fine = gamma_continuum_s_moments_order(mu, r, n_fine)
                    selfcheck = float(abs(fine[0] - coarse[0]))

                    if (
                        np.all(np.isfinite(fine))
                        and np.isfinite(selfcheck)
                        and selfcheck <= tol
                    ):
                        return (
                            float(fine[0]), float(fine[1]), float(fine[2]), float(fine[3]),
                            "gamma_continuum_high_order_audit", float(n_fine), 1.0,
                        )

                    if n_fine >= n_max:
                        raise RuntimeError(
                            "High-count continuum C'(1) audit reference failed its internal "
                            f"quadrature check after adaptive refinement for "
                            f"mu={float(mu):.6g}, r={float(r):.6g}: "
                            f"order={n_fine}, |C1_fine-C1_coarse|={selfcheck:.6g} > "
                            f"{tol:.6g}; max_order={n_max}."
                        )

                    n_fine = min(2 * n_fine, n_max)

            def cprime_moments_grid_point(mu, r):
                """
                Production C'(1) grid-point backend: exact whenever feasible.

                Numerical policy
                ----------------
                The production lookup now uses the exact discrete NB support sum
                whenever the central support window fits inside the explicit
                computational work cap:

                    width <= cprime_support_max  -> exact discrete C'(1)

                The gamma-continuum approximation is used only when the exact
                discrete window itself exceeds that work cap AND the point lies
                inside the already-frozen trusted high-count continuum region:

                    width > cprime_support_max
                    and continuum_audit_allowed(mu, r)
                        -> gamma continuum C'(1)

                If the exact window exceeds cprime_support_max but the point is
                not continuum-eligible, the function fails explicitly rather than
                silently substituting an approximation whose accuracy has not been
                established for that numerical regime.

                This deliberately makes the production point target identical to
                the independent C'(1) audit target everywhere exact evaluation is
                computationally feasible.

                The returned final element is the central NB support width used
                for work/runtime diagnostics.
                """
                mu = float(mu)
                r = float(r)

                if (
                    not np.isfinite(mu)
                    or not np.isfinite(r)
                    or mu <= 0
                    or r <= 0
                ):
                    return (
                        np.nan,
                        np.nan,
                        np.nan,
                        np.nan,
                        "invalid",
                        np.nan,
                        np.nan,
                        np.nan,
                    )

                lo, hi = nb_quantile_bounds(mu, r)
                width = hi - lo + 1

                # Exact whenever the explicit work cap permits it.
                if width <= int(cprime_support_max):
                    out = exact_nb_s_moments_chunked(mu, r)
                    return (*out, int(width))

                # Only beyond the exact-work cap may the continuum backend be
                # substituted, and only in its already-audited high-count region.
                if continuum_audit_allowed(mu, r):
                    out = gamma_continuum_s_moments(mu, r)
                    return (*out, int(width))

                raise RuntimeError(
                    "C'(1) production grid point requires "
                    f"{int(width):,} discrete support points for "
                    f"mu={mu:.6g}, r={r:.6g}, exceeding "
                    f"cprime_support_max={int(cprime_support_max):,}, "
                    "and the point does not satisfy the frozen "
                    "continuum-audit eligibility conditions."
                )

            def cprime_moments_exact_point(mu, r):
                """
                Audit reference for C'(1).

                Use the exact discrete centered sum whenever it fits within the explicit
                brute-force work cap.  If the central NB quantile window itself spans
                more than cprime_support_max integers, switch only in the high-count
                asymptotic regime to an independent high-order continuum reference.
                """
                mu = float(mu)
                r = float(r)
                lo, hi = nb_quantile_bounds(mu, r)
                width = hi - lo + 1
                if width <= int(cprime_support_max):
                    return exact_nb_s_moments_chunked(mu, r)
                if continuum_audit_allowed(mu, r):
                    return gamma_continuum_s_moments_audit(mu, r)
                raise RuntimeError(
                    f"C'(1) audit window requires {width:,} discrete points for "
                    f"mu={mu:.6g}, r={r:.6g}, exceeding cprime_support_max="
                    f"{int(cprime_support_max):,}, and the point does not satisfy the "
                    "pre-specified high-count continuum-audit conditions."
                )

            def gamma_continuum_theta_cprime_order(mu, r, theta, n_quad):
                mu = float(mu)
                r = float(r)
                theta = float(theta)
                u, w = get_gamma_quantile_quadrature(n_quad)
                x = gamma_dist.ppf(u, a=r, scale=1.0 / r)
                y = mu * x
                s = half_nb_deviance(y, mu, r)
                logw = np.log(w) - (theta - 1.0) * s
                logZ = float(logsumexp(logw))
                q = np.exp(logw - logZ)
                mean_s = float(np.sum(q * s))
                return mean_s - 0.5 / theta

            def gamma_continuum_theta_cprime_audit(mu, r, theta):
                """
                High-count C'(theta) audit reference with adaptive Gauss-Legendre
                quadrature refinement.  A failed 1024-vs-512 check is therefore treated
                as insufficient reference resolution, not as statistical-method failure.
                The frozen self-check tolerance is preserved exactly.
                """
                n_fine = int(cprime_continuum_audit_quad_n)
                n_max = int(cprime_continuum_audit_quad_n_max)
                tol = float(cprime_continuum_audit_selfcheck_tol)

                while True:
                    n_coarse = max(64, n_fine // 2)
                    coarse = gamma_continuum_theta_cprime_order(
                        mu, r, theta, n_coarse
                    )
                    fine = gamma_continuum_theta_cprime_order(
                        mu, r, theta, n_fine
                    )
                    selfcheck = float(abs(fine - coarse))

                    if (
                        np.isfinite(fine)
                        and np.isfinite(selfcheck)
                        and selfcheck <= tol
                    ):
                        return (
                            float(fine),
                            "gamma_continuum_high_order_audit",
                            int(n_fine),
                        )

                    if n_fine >= n_max:
                        raise RuntimeError(
                            "High-count continuum C'(theta) audit reference failed its internal "
                            f"quadrature check after adaptive refinement for "
                            f"mu={float(mu):.6g}, r={float(r):.6g}, "
                            f"theta={float(theta):.6g}: order={n_fine}, "
                            f"|fine-coarse|={selfcheck:.6g} > {tol:.6g}; "
                            f"max_order={n_max}."
                        )

                    n_fine = min(2 * n_fine, n_max)

            def combine_logweighted_chunks(state, logw, s):
                chunk_logZ = float(logsumexp(logw))
                if not np.isfinite(chunk_logZ):
                    return state
                q = np.exp(logw - chunk_logZ)
                chunk_mean = float(np.sum(q * s))
                if state is None:
                    return [chunk_logZ, chunk_mean]
                old_logZ, old_mean = state
                new_logZ = float(np.logaddexp(old_logZ, chunk_logZ))
                a = float(np.exp(old_logZ - new_logZ))
                b = float(np.exp(chunk_logZ - new_logZ))
                return [new_logZ, a * old_mean + b * chunk_mean]

            def theta_window_sum(mu, r, theta, lo, hi):
                state = None
                chunk = int(cprime_chunk_size)
                for start in range(int(lo), int(hi) + 1, chunk):
                    stop = min(int(hi) + 1, start + chunk)
                    y = np.arange(start, stop, dtype=float)
                    log_nb = log_nb_pmf(y, mu, r)
                    s = half_nb_deviance(y, mu, r)
                    logw = log_nb - (theta - 1.0) * s
                    state = combine_logweighted_chunks(state, logw, s)
                return state

            def cprime_exact_theta_point(mu, r, theta):
                """
                Audit reference for C'(theta), centered on the distribution mass.

                Exact discrete summation is retained whenever its initial central NB
                window fits within cprime_support_max.  If the *window width itself* is
                larger than that cap, a high-order gamma-continuum audit reference is
                used only when the explicit high-count asymptotic criteria are met.

                For theta<1 the discrete target may broaden beyond the NB window.  The
                discrete path expands both tails adaptively.  If that expansion reaches
                the brute-force cap, it may likewise switch to the audited high-count
                continuum reference rather than fail solely because millions of nearly
                continuous lattice points would need enumeration.
                """
                mu = float(mu)
                r = float(r)
                theta = float(theta)
                if not np.isfinite(mu) or not np.isfinite(r) or not np.isfinite(theta):
                    return np.nan, "invalid", np.nan
                if mu <= 0 or r <= 0 or theta <= 0:
                    return np.nan, "invalid", np.nan

                base_tail = min(max(float(cprime_tail_probability), 1e-14), 1e-8)
                lo, hi = nb_quantile_bounds(mu, r, base_tail)
                initial_width = hi - lo + 1

                if initial_width > int(cprime_support_max):
                    if continuum_audit_allowed(mu, r):
                        return gamma_continuum_theta_cprime_audit(mu, r, theta)
                    raise RuntimeError(
                        f"theta-aware C' initial centered window requires "
                        f"{initial_width:,} points for mu={mu:.6g}, r={r:.6g}, "
                        f"theta={theta:.6g}, exceeding cprime_support_max="
                        f"{int(cprime_support_max):,}, and the point does not satisfy "
                        "the high-count continuum-audit conditions."
                    )

                state = theta_window_sum(mu, r, theta, lo, hi)
                if state is None:
                    raise RuntimeError(
                        f"theta-aware C' initial sum failed for mu={mu}, r={r}, theta={theta}."
                    )

                if theta >= 1.0:
                    return state[1] - 0.5 / theta, "centered_support_sum", hi - lo + 1

                total_points = initial_width
                step = max(1024, min(initial_width // 2, 250_000))
                stable_rounds = 0
                mass_tol = max(float(cprime_tail_probability), 1e-12)
                mean_tol = max(10.0 * float(cprime_tail_probability), 1e-10)

                while stable_rounds < 2:
                    old_logZ, old_mean = state
                    added_state = None
                    added_points = 0

                    if lo > 0:
                        new_lo = max(0, lo - step)
                        left_n = lo - new_lo
                        left = theta_window_sum(mu, r, theta, new_lo, lo - 1)
                        if left is not None:
                            added_state = left
                        lo = new_lo
                        added_points += left_n

                    new_hi = hi + step
                    right_n = new_hi - hi
                    right = theta_window_sum(mu, r, theta, hi + 1, new_hi)
                    if right is not None:
                        if added_state is None:
                            added_state = right
                        else:
                            lz, lm = added_state
                            rz, rm = right
                            nz = float(np.logaddexp(lz, rz))
                            added_state = [
                                nz,
                                float(np.exp(lz - nz)) * lm
                                + float(np.exp(rz - nz)) * rm,
                            ]
                    hi = new_hi
                    added_points += right_n

                    if added_state is None:
                        stable_rounds += 1
                        total_points += added_points
                        continue

                    add_logZ, add_mean = added_state
                    new_logZ = float(np.logaddexp(old_logZ, add_logZ))
                    old_weight = float(np.exp(old_logZ - new_logZ))
                    add_weight = float(np.exp(add_logZ - new_logZ))
                    new_mean = old_weight * old_mean + add_weight * add_mean
                    state = [new_logZ, new_mean]

                    relative_added_mass = add_weight
                    mean_change = abs(new_mean - old_mean)
                    if relative_added_mass <= mass_tol and mean_change <= mean_tol:
                        stable_rounds += 1
                    else:
                        stable_rounds = 0

                    total_points += added_points
                    if total_points > int(cprime_support_max):
                        if continuum_audit_allowed(mu, r):
                            return gamma_continuum_theta_cprime_audit(mu, r, theta)
                        raise RuntimeError(
                            f"theta-aware C' centered expansion did not converge for "
                            f"mu={mu:.6g}, r={r:.6g}, theta={theta:.6g} after "
                            f"{total_points:,} evaluated support points, and the point "
                            "does not satisfy the high-count continuum-audit conditions."
                        )
                    step = min(step * 2, 1_000_000)

                return state[1] - 0.5 / theta, "centered_support_sum", total_points

            def build_cprime_lookup(mu_F, mu_0, r_hat, replicate):
                """
                Build and audit the C-prime derivative lookup surfaces.

                Numerical policy
                ----------------
                Grid nodes retain the frozen exact-when-feasible rule:

                    support width <= cprime_support_max
                        -> exact discrete C'(1)

                    support width > cprime_support_max
                    and continuum_audit_allowed(mu, r)
                        -> gamma-continuum C'(1)

                The exact and gamma numerical targets are not forced to agree at the
                work-cap boundary.  Therefore an interpolator is NEVER allowed to blend
                across a grid cell whose four corners contain both backends.

                For ordinary cells, the existing rectangular linear/cubic interpolation
                is used exactly as before.

                For a mixed exact/gamma cell:
                  1. classify the query point by its own support width;
                  2. evaluate it with a branch-specific 2-D linear interpolator built
                     only from grid nodes belonging to that same backend;
                  3. if the branch interpolator cannot evaluate the point, fall back to
                     the direct production backend at that point.

                Thus no query is interpolated across the exact/gamma discontinuity.
                The SAME hybrid evaluator is used for the independent C'(1) audit and
                for the final full/null C1..C4 matrices.  Refinement therefore occurs
                only when the seam-safe production evaluator itself misses the frozen
                audit tolerance; the tolerance is never loosened.

                cprime_grid_max_refinements remains the normal refinement budget.
                cprime_grid_emergency_refinements supplies the existing numerical-only
                emergency budget.
                """
                n_samples = mu_F.shape[1]
                mu_all = np.concatenate([mu_F.ravel(), mu_0.ravel()])
                r_all = np.concatenate([
                    np.repeat(r_hat, n_samples),
                    np.repeat(r_hat, n_samples),
                ])
                ok = (
                    np.isfinite(mu_all)
                    & np.isfinite(r_all)
                    & (mu_all > 0)
                    & (r_all > 0)
                )
                mu_use = mu_all[ok]
                r_use = r_all[ok]
                if not len(mu_use):
                    raise RuntimeError(
                        f"replicate {replicate}: no usable mu/r values for cprime."
                    )

                lmu_min, lmu_max = np.min(np.log(mu_use)), np.max(np.log(mu_use))
                lr_min, lr_max = np.min(np.log(r_use)), np.max(np.log(r_use))
                mu_margin = max(0.02 * (lmu_max - lmu_min), 0.05)
                r_margin = max(0.02 * (lr_max - lr_min), 0.05)

                rng = np.random.default_rng(
                    int(random_seed) + 1_000_003 * int(replicate)
                )
                audit_n = min(int(cprime_audit_n), len(mu_use))
                if audit_n:
                    forced = list(dict.fromkeys([
                        int(np.argmin(mu_use)),
                        int(np.argmax(mu_use)),
                        int(np.argmin(r_use)),
                        int(np.argmax(r_use)),
                        int(np.argmin(mu_use / r_use)),
                        int(np.argmax(mu_use / r_use)),
                    ]))
                    if len(forced) > audit_n:
                        forced = forced[:audit_n]
                    remaining = max(0, audit_n - len(forced))
                    available = np.setdiff1d(
                        np.arange(len(mu_use)),
                        np.asarray(forced, dtype=int),
                    )
                    random_idx = (
                        rng.choice(
                            available,
                            size=min(remaining, len(available)),
                            replace=False,
                        )
                        if remaining and len(available)
                        else np.empty(0, dtype=int)
                    )
                    audit_idx = np.asarray(
                        forced + random_idx.tolist(),
                        dtype=int,
                    )
                    audit_mu = mu_use[audit_idx]
                    audit_r = r_use[audit_idx]
                    audit_exact_values = []
                    audit_reference_backends = []
                    for m, rr in zip(audit_mu, audit_r):
                        reference = cprime_moments_exact_point(m, rr)
                        audit_exact_values.append(reference[0])
                        audit_reference_backends.append(reference[4])
                    audit_exact = np.asarray(audit_exact_values, dtype=float)
                    audit_points = np.column_stack([
                        np.log(audit_mu),
                        np.log(audit_r),
                    ])
                else:
                    audit_mu = audit_r = audit_exact = np.empty(0)
                    audit_points = np.empty((0, 2), dtype=float)
                    audit_reference_backends = []

                def support_width_array(mu_values, r_values):
                    """Vectorized copy of nb_quantile_bounds(...)->window width."""
                    muv = np.asarray(mu_values, dtype=float)
                    rv = np.asarray(r_values, dtype=float)
                    if muv.shape != rv.shape:
                        raise ValueError("mu/r arrays must have the same shape.")

                    valid = (
                        np.isfinite(muv)
                        & np.isfinite(rv)
                        & (muv > 0)
                        & (rv > 0)
                    )
                    if not np.all(valid):
                        raise RuntimeError(
                            "Seam classification received invalid mu/r values."
                        )

                    side = 0.5 * float(
                        np.clip(
                            float(cprime_tail_probability),
                            1e-15,
                            1e-4,
                        )
                    )
                    p = rv / (rv + muv)
                    lo_q = nbinom.ppf(side, rv, p)
                    hi_q = nbinom.ppf(1.0 - side, rv, p)

                    lo_q = np.where(np.isfinite(lo_q), lo_q, 0.0)
                    bad_hi = ~np.isfinite(hi_q)
                    if np.any(bad_hi):
                        sd = np.sqrt(
                            muv[bad_hi]
                            + muv[bad_hi] * muv[bad_hi] / rv[bad_hi]
                        )
                        hi_q = np.asarray(hi_q, dtype=float)
                        hi_q[bad_hi] = (
                            muv[bad_hi]
                            + 14.0 * sd
                            + 50.0
                        )

                    lo = np.maximum(
                        0,
                        np.floor(lo_q).astype(np.int64),
                    )
                    hi = np.maximum(
                        lo,
                        np.ceil(hi_q).astype(np.int64),
                    )
                    return hi - lo + 1

                def production_backend_codes(mu_values, r_values):
                    """
                    0 = exact discrete
                    1 = gamma continuum
                    """
                    muv = np.asarray(mu_values, dtype=float)
                    rv = np.asarray(r_values, dtype=float)
                    widths = support_width_array(muv, rv)

                    exact = widths <= int(cprime_support_max)
                    continuum_ok = (
                        np.isfinite(muv)
                        & np.isfinite(rv)
                        & (muv >= float(cprime_continuum_audit_min_mu))
                        & (rv > 0)
                        & (
                            (muv / rv)
                            >= float(cprime_continuum_audit_min_mu_over_r)
                        )
                    )

                    invalid = (~exact) & (~continuum_ok)
                    if np.any(invalid):
                        first = int(np.flatnonzero(invalid)[0])
                        raise RuntimeError(
                            "Seam-safe C'(1) evaluator found a query point beyond "
                            "cprime_support_max and outside the frozen continuum region: "
                            f"mu={muv[first]:.6g}, r={rv[first]:.6g}, "
                            f"width={int(widths[first]):,}."
                        )

                    code = np.where(exact, 0, 1).astype(np.int8)
                    return code, widths

                def containing_cells(points, log_mu_grid, log_r_grid):
                    pts = np.asarray(points, dtype=float)
                    ii = (
                        np.searchsorted(
                            log_mu_grid,
                            pts[:, 0],
                            side="right",
                        )
                        - 1
                    )
                    jj = (
                        np.searchsorted(
                            log_r_grid,
                            pts[:, 1],
                            side="right",
                        )
                        - 1
                    )
                    ii = np.clip(ii, 0, len(log_mu_grid) - 2)
                    jj = np.clip(jj, 0, len(log_r_grid) - 2)
                    return ii.astype(int), jj.astype(int)

                def mixed_cell_mask(
                    points,
                    log_mu_grid,
                    log_r_grid,
                    backend_code,
                ):
                    if len(points) == 0:
                        return np.zeros(0, dtype=bool)

                    ii, jj = containing_cells(
                        points,
                        log_mu_grid,
                        log_r_grid,
                    )
                    c00 = backend_code[ii, jj]
                    c10 = backend_code[ii + 1, jj]
                    c01 = backend_code[ii, jj + 1]
                    c11 = backend_code[ii + 1, jj + 1]
                    same = (
                        (c00 == c10)
                        & (c00 == c01)
                        & (c00 == c11)
                    )
                    return ~same

                def build_branch_interpolators(
                    log_mu_grid,
                    log_r_grid,
                    backend_code,
                    values_by_suffix,
                ):
                    """
                    Build one Delaunay triangulation per numerical backend, then
                    reuse it for C1..C4.  Only same-backend values enter each
                    interpolator, so these interpolators cannot average exact and
                    gamma grid nodes together.
                    """
                    lm, lr = np.meshgrid(
                        log_mu_grid,
                        log_r_grid,
                        indexing="ij",
                    )
                    coords = np.column_stack([
                        lm.ravel(),
                        lr.ravel(),
                    ])
                    codes = backend_code.ravel()

                    branch = {}
                    branch_node_counts = {}

                    for code in (0, 1):
                        use = codes == code
                        branch_node_counts[int(code)] = int(np.sum(use))
                        if int(np.sum(use)) < 3:
                            continue

                        try:
                            tri = Delaunay(coords[use])
                        except Exception:
                            continue

                        for suffix, values in values_by_suffix.items():
                            try:
                                branch[(int(code), str(suffix))] = (
                                    LinearNDInterpolator(
                                        tri,
                                        np.asarray(values, dtype=float).ravel()[use],
                                        fill_value=np.nan,
                                    )
                                )
                            except Exception:
                                pass

                    return branch, branch_node_counts

                component_index = {
                    "c1": 0,
                    "c2": 1,
                    "c3": 2,
                    "c4": 3,
                }

                def evaluate_seam_safe(
                    *,
                    suffix,
                    regular_interp,
                    branch_interps,
                    points,
                    mu_values,
                    r_values,
                    log_mu_grid,
                    log_r_grid,
                    backend_code,
                ):
                    """
                    Evaluate one C-prime derivative surface without ever blending
                    exact and gamma nodes in a mixed cell.
                    """
                    pts = np.asarray(points, dtype=float)
                    muv = np.asarray(mu_values, dtype=float)
                    rv = np.asarray(r_values, dtype=float)

                    prediction = np.asarray(
                        regular_interp(pts),
                        dtype=float,
                    ).reshape(-1)

                    mixed = mixed_cell_mask(
                        pts,
                        log_mu_grid,
                        log_r_grid,
                        backend_code,
                    )

                    stats = {
                        "n_points": int(len(pts)),
                        "n_mixed_cell_points": int(np.sum(mixed)),
                        "n_branch_interp_points": 0,
                        "n_direct_fallback_points": 0,
                        "n_exact_branch_points": 0,
                        "n_gamma_branch_points": 0,
                    }

                    if not np.any(mixed):
                        return prediction, stats

                    seam_indices = np.flatnonzero(mixed)
                    seam_codes, _ = production_backend_codes(
                        muv[seam_indices],
                        rv[seam_indices],
                    )

                    for code in (0, 1):
                        local_mask = seam_codes == code
                        if not np.any(local_mask):
                            continue

                        idx = seam_indices[local_mask]
                        stats[
                            "n_exact_branch_points"
                            if code == 0
                            else "n_gamma_branch_points"
                        ] += int(len(idx))

                        branch_interp = branch_interps.get(
                            (int(code), str(suffix))
                        )

                        if branch_interp is None:
                            branch_values = np.full(
                                len(idx),
                                np.nan,
                                dtype=float,
                            )
                        else:
                            branch_values = np.asarray(
                                branch_interp(pts[idx]),
                                dtype=float,
                            ).reshape(-1)

                        good = np.isfinite(branch_values)
                        if np.any(good):
                            prediction[idx[good]] = branch_values[good]
                            stats["n_branch_interp_points"] += int(
                                np.sum(good)
                            )

                        bad_idx = idx[~good]
                        if len(bad_idx):
                            comp = int(component_index[str(suffix)])
                            for query_index in bad_idx:
                                direct = cprime_moments_grid_point(
                                    muv[query_index],
                                    rv[query_index],
                                )
                                value = float(direct[comp])
                                if not np.isfinite(value):
                                    raise RuntimeError(
                                        "Direct seam fallback returned a nonfinite "
                                        f"{suffix} value at mu="
                                        f"{muv[query_index]:.6g}, r="
                                        f"{rv[query_index]:.6g}."
                                    )
                                prediction[query_index] = value

                            stats["n_direct_fallback_points"] += int(
                                len(bad_idx)
                            )

                    return prediction, stats

                configured_max = int(cprime_grid_max_refinements)
                emergency_max = int(cprime_grid_emergency_refinements)
                hard_max = configured_max + emergency_max
                best = None
                selected = None

                for refinement in range(hard_max + 1):
                    n_mu = int(
                        np.ceil(
                            int(cprime_grid_n_mu)
                            * 1.5**refinement
                        )
                    )
                    n_r = int(
                        np.ceil(
                            int(cprime_grid_n_r)
                            * 1.5**refinement
                        )
                    )
                    log_mu_grid = np.linspace(
                        lmu_min - mu_margin,
                        lmu_max + mu_margin,
                        n_mu,
                    )
                    log_r_grid = np.linspace(
                        lr_min - r_margin,
                        lr_max + r_margin,
                        n_r,
                    )

                    values_c1 = np.empty((n_mu, n_r))
                    values_c2 = np.empty((n_mu, n_r))
                    values_c3 = np.empty((n_mu, n_r))
                    values_c4 = np.empty((n_mu, n_r))
                    backend_code = np.empty(
                        (n_mu, n_r),
                        dtype=np.int8,
                    )

                    support_count = 0
                    gamma_count = 0
                    wide_exact_fallback_count = 0
                    max_support_y = 0.0
                    max_discrete_support_width = 0

                    for i, lmu in enumerate(log_mu_grid):
                        mu = float(np.exp(lmu))
                        for j, lr in enumerate(log_r_grid):
                            rr = float(np.exp(lr))
                            (
                                c1,
                                c2,
                                c3,
                                c4,
                                backend,
                                y_max,
                                _,
                                support_width,
                            ) = cprime_moments_grid_point(mu, rr)

                            if not np.all(
                                np.isfinite([c1, c2, c3, c4])
                            ):
                                raise RuntimeError(
                                    f"replicate {replicate}: nonfinite C-prime "
                                    f"derivatives at mu={mu}, r={rr}."
                                )

                            values_c1[i, j] = c1
                            values_c2[i, j] = c2
                            values_c3[i, j] = c3
                            values_c4[i, j] = c4

                            is_discrete = (
                                backend == "centered_support_sum"
                            )
                            is_gamma = (
                                backend == "gamma_continuum"
                            )

                            if not (is_discrete or is_gamma):
                                raise RuntimeError(
                                    "C'(1) grid point returned an unexpected "
                                    f"backend {backend!r}."
                                )

                            backend_code[i, j] = (
                                0 if is_discrete else 1
                            )

                            support_count += int(is_discrete)
                            gamma_count += int(is_gamma)

                            if (
                                is_discrete
                                and np.isfinite(support_width)
                            ):
                                max_discrete_support_width = max(
                                    max_discrete_support_width,
                                    int(support_width),
                                )
                                wide_exact_fallback_count += int(
                                    int(support_width)
                                    > int(cprime_grid_exact_width_max)
                                )

                            if np.isfinite(y_max):
                                max_support_y = max(
                                    max_support_y,
                                    float(y_max),
                                )

                    values_by_suffix = {
                        "c1": values_c1,
                        "c2": values_c2,
                        "c3": values_c3,
                        "c4": values_c4,
                    }

                    branch_interps, branch_node_counts = (
                        build_branch_interpolators(
                            log_mu_grid,
                            log_r_grid,
                            backend_code,
                            values_by_suffix,
                        )
                    )

                    mixed_cells = (
                        (
                            backend_code[:-1, :-1]
                            != backend_code[1:, :-1]
                        )
                        | (
                            backend_code[:-1, :-1]
                            != backend_code[:-1, 1:]
                        )
                        | (
                            backend_code[:-1, :-1]
                            != backend_code[1:, 1:]
                        )
                    )
                    n_mixed_cells = int(np.sum(mixed_cells))

                    interpolation_methods = ["linear"]
                    if n_mu >= 4 and n_r >= 4:
                        interpolation_methods.append("cubic")

                    for interpolation_method in interpolation_methods:
                        try:
                            interp_c1 = RegularGridInterpolator(
                                (log_mu_grid, log_r_grid),
                                values_c1,
                                method=interpolation_method,
                                bounds_error=True,
                            )
                        except Exception:
                            continue

                        if audit_n:
                            audit_pred, audit_seam_stats = (
                                evaluate_seam_safe(
                                    suffix="c1",
                                    regular_interp=interp_c1,
                                    branch_interps=branch_interps,
                                    points=audit_points,
                                    mu_values=audit_mu,
                                    r_values=audit_r,
                                    log_mu_grid=log_mu_grid,
                                    log_r_grid=log_r_grid,
                                    backend_code=backend_code,
                                )
                            )
                            audit_error = (
                                audit_pred - audit_exact
                            )
                            max_abs = float(
                                np.max(np.abs(audit_error))
                            )
                            med_abs = float(
                                np.median(np.abs(audit_error))
                            )
                            q95_abs = float(
                                np.quantile(
                                    np.abs(audit_error),
                                    0.95,
                                )
                            )
                        else:
                            audit_pred = audit_error = np.empty(0)
                            max_abs = med_abs = q95_abs = 0.0
                            audit_seam_stats = {
                                "n_points": 0,
                                "n_mixed_cell_points": 0,
                                "n_branch_interp_points": 0,
                                "n_direct_fallback_points": 0,
                                "n_exact_branch_points": 0,
                                "n_gamma_branch_points": 0,
                            }

                        candidate = {
                            "interpolator_c1": interp_c1,
                            "branch_interpolators": branch_interps,
                            "backend_code": backend_code,
                            "log_mu_grid": log_mu_grid,
                            "log_r_grid": log_r_grid,
                            "n_mu": n_mu,
                            "n_r": n_r,
                            "refinement": refinement,
                            "configured_max_refinement": configured_max,
                            "hard_max_refinement": hard_max,
                            "emergency_refinements_used": max(
                                0,
                                refinement - configured_max,
                            ),
                            "interpolation_method": interpolation_method,
                            "support_sum_count": support_count,
                            "expect_count": gamma_count,
                            "gamma_continuum_count": gamma_count,
                            "wide_exact_fallback_count": (
                                wide_exact_fallback_count
                            ),
                            "max_discrete_support_width": int(
                                max_discrete_support_width
                            ),
                            "max_support_y": max_support_y,
                            "grid_continuum_policy": (
                                "exact_if_width<=cprime_support_max;"
                                "gamma_only_if_width>cprime_support_max"
                                " & continuum_audit_allowed(mu,r);"
                                "mixed_exact_gamma_cells_use_same_backend_"
                                "branch_interpolation_with_direct_fallback"
                            ),
                            "n_mixed_exact_gamma_cells": (
                                n_mixed_cells
                            ),
                            "exact_branch_grid_node_count": int(
                                branch_node_counts.get(0, 0)
                            ),
                            "gamma_branch_grid_node_count": int(
                                branch_node_counts.get(1, 0)
                            ),
                            "audit_n": len(audit_mu),
                            "audit_max_abs_error": max_abs,
                            "audit_median_abs_error": med_abs,
                            "audit_q95_abs_error": q95_abs,
                            "audit_mu": audit_mu,
                            "audit_r": audit_r,
                            "audit_exact": audit_exact,
                            "audit_interpolated": audit_pred,
                            "audit_error": audit_error,
                            "audit_mixed_cell_point_count": int(
                                audit_seam_stats[
                                    "n_mixed_cell_points"
                                ]
                            ),
                            "audit_seam_branch_interp_count": int(
                                audit_seam_stats[
                                    "n_branch_interp_points"
                                ]
                            ),
                            "audit_seam_direct_fallback_count": int(
                                audit_seam_stats[
                                    "n_direct_fallback_points"
                                ]
                            ),
                            "audit_seam_exact_branch_count": int(
                                audit_seam_stats[
                                    "n_exact_branch_points"
                                ]
                            ),
                            "audit_seam_gamma_branch_count": int(
                                audit_seam_stats[
                                    "n_gamma_branch_points"
                                ]
                            ),
                            "audit_discrete_reference_count": int(
                                sum(
                                    value
                                    == "centered_support_sum"
                                    for value in audit_reference_backends
                                )
                            ),
                            "audit_continuum_reference_count": int(
                                sum(
                                    value
                                    == "gamma_continuum_high_order_audit"
                                    for value in audit_reference_backends
                                )
                            ),
                        }

                        if (
                            best is None
                            or candidate[
                                "audit_max_abs_error"
                            ]
                            < best["audit_max_abs_error"]
                        ):
                            best = candidate

                        if max_abs <= float(cprime_audit_tol):
                            try:
                                interp_c2 = RegularGridInterpolator(
                                    (log_mu_grid, log_r_grid),
                                    values_c2,
                                    method=interpolation_method,
                                    bounds_error=True,
                                )
                                interp_c3 = RegularGridInterpolator(
                                    (log_mu_grid, log_r_grid),
                                    values_c3,
                                    method=interpolation_method,
                                    bounds_error=True,
                                )
                                interp_c4 = RegularGridInterpolator(
                                    (log_mu_grid, log_r_grid),
                                    values_c4,
                                    method=interpolation_method,
                                    bounds_error=True,
                                )
                            except Exception:
                                continue

                            candidate[
                                "interpolator_c2"
                            ] = interp_c2
                            candidate[
                                "interpolator_c3"
                            ] = interp_c3
                            candidate[
                                "interpolator_c4"
                            ] = interp_c4
                            selected = candidate
                            break

                    if selected is not None:
                        break

                if selected is None:
                    detail = (
                        "no audited interpolation candidate was constructed"
                        if best is None
                        else (
                            f"best method={best['interpolation_method']}, "
                            f"refinement={best['refinement']}, "
                            f"max abs error="
                            f"{best['audit_max_abs_error']:.6g}, "
                            f"mixed seam audit points="
                            f"{int(best.get('audit_mixed_cell_point_count', 0)):,}, "
                            f"seam direct fallbacks="
                            f"{int(best.get('audit_seam_direct_fallback_count', 0)):,}, "
                            f"max exact discrete width="
                            f"{int(best.get('max_discrete_support_width', 0)):,}, "
                            f"wide exact nodes="
                            f"{int(best.get('wide_exact_fallback_count', 0)):,}"
                        )
                    )
                    raise RuntimeError(
                        f"replicate {replicate}: cprime interpolation failed audit "
                        f"after normal+emergency refinement; {detail}; required "
                        f"tolerance={float(cprime_audit_tol):.6g}. "
                        "The audit tolerance was not loosened."
                    )

                best = selected

                r_points = np.repeat(r_hat, n_samples)
                mu_F_flat = mu_F.ravel()
                mu_0_flat = mu_0.ravel()

                points_F = np.column_stack([
                    np.log(mu_F_flat),
                    np.log(r_points),
                ])
                points_0 = np.column_stack([
                    np.log(mu_0_flat),
                    np.log(r_points),
                ])

                production_stats = {
                    "n_mixed_cell_points": 0,
                    "n_branch_interp_points": 0,
                    "n_direct_fallback_points": 0,
                    "n_exact_branch_points": 0,
                    "n_gamma_branch_points": 0,
                }

                for suffix in ("c1", "c2", "c3", "c4"):
                    interp = best[f"interpolator_{suffix}"]

                    values_F, stats_F = evaluate_seam_safe(
                        suffix=suffix,
                        regular_interp=interp,
                        branch_interps=best[
                            "branch_interpolators"
                        ],
                        points=points_F,
                        mu_values=mu_F_flat,
                        r_values=r_points,
                        log_mu_grid=best["log_mu_grid"],
                        log_r_grid=best["log_r_grid"],
                        backend_code=best["backend_code"],
                    )
                    values_0, stats_0 = evaluate_seam_safe(
                        suffix=suffix,
                        regular_interp=interp,
                        branch_interps=best[
                            "branch_interpolators"
                        ],
                        points=points_0,
                        mu_values=mu_0_flat,
                        r_values=r_points,
                        log_mu_grid=best["log_mu_grid"],
                        log_r_grid=best["log_r_grid"],
                        backend_code=best["backend_code"],
                    )

                    best[f"{suffix}_F_matrix"] = (
                        values_F.reshape(mu_F.shape)
                    )
                    best[f"{suffix}_0_matrix"] = (
                        values_0.reshape(mu_0.shape)
                    )

                    # Count production seam work once, using C1 only. C2..C4
                    # use the identical point classification.
                    if suffix == "c1":
                        for key in production_stats:
                            production_stats[key] = int(
                                stats_F[key]
                                + stats_0[key]
                            )

                best["production_mixed_cell_point_count"] = int(
                    production_stats["n_mixed_cell_points"]
                )
                best["production_seam_branch_interp_count"] = int(
                    production_stats["n_branch_interp_points"]
                )
                best["production_seam_direct_fallback_count"] = int(
                    production_stats["n_direct_fallback_points"]
                )
                best["production_seam_exact_branch_count"] = int(
                    production_stats["n_exact_branch_points"]
                )
                best["production_seam_gamma_branch_count"] = int(
                    production_stats["n_gamma_branch_points"]
                )

                best["C_F"] = best["c1_F_matrix"].sum(axis=1)
                best["C_0"] = best["c1_0_matrix"].sum(axis=1)

                # Large interpolation objects are no longer needed after the
                # full/null matrices are materialized.
                best.pop("branch_interpolators", None)
                best.pop("backend_code", None)
                best.pop("log_mu_grid", None)
                best.pop("log_r_grid", None)

                return best

            def evaluate_thetaaware_cprime(cprime, mu_F, mu_0, r_hat, theta_F, replicate):
                """
                Evaluate C'(theta_F) for the shared-score geometry with an audited
                hybrid strategy and an audit-triggered direct fallback.

                Numerical policy
                ----------------
                1. `theta_cprime_max_abs_delta` is the MAXIMUM Taylor radius, not an
                   admissibility bound on theta_F.
                2. Genes outside that radius are evaluated directly with
                   `cprime_exact_theta_point()`.
                3. Genes inside the radius are initially evaluated with the requested
                   Taylor expansion around theta=1.
                4. The Taylor region is audited at the frozen
                   `theta_cprime_audit_tol`.  If it fails, THE TOLERANCE IS NOT
                   LOOSENED.  Instead the effective Taylor radius is contracted to just
                   below the smallest |theta_F-1| among the audited failing points, and
                   every gene newly outside the contracted radius is recomputed
                   directly.  The reduced Taylor region is then re-audited.
                5. This repeats until the remaining Taylor region passes the frozen
                   audit, or until every valid gene is evaluated directly.

                Therefore a local Taylor audit failure is a trigger to use the more
                accurate direct numerical backend, not a reason to abort the method.
                The statistical target remains C'(theta_F) throughout.
                """
                theta_F = np.asarray(theta_F, dtype=float)
                r_hat = np.asarray(r_hat, dtype=float)

                valid_theta = np.isfinite(theta_F) & (theta_F > 0.0)
                n_valid_theta = int(np.sum(valid_theta))
                n_invalid_theta = int(len(theta_F) - n_valid_theta)

                if n_valid_theta < 100:
                    raise RuntimeError(
                        f"replicate {replicate}: only {n_valid_theta} valid theta_F values "
                        "are available for theta-aware C'."
                    )

                if int(theta_cprime_order) not in (1, 2, 3):
                    raise ValueError("theta_cprime_order must be 1, 2, or 3.")

                configured_cutoff = float(theta_cprime_max_abs_delta)
                if not np.isfinite(configured_cutoff) or configured_cutoff <= 0:
                    raise ValueError("theta_cprime_max_abs_delta must be finite and > 0.")

                abs_delta = np.abs(theta_F - 1.0)
                max_delta = float(np.max(abs_delta[valid_theta]))
                n_samples = int(mu_F.shape[1])

                cF = np.full_like(cprime["c1_F_matrix"], np.nan, dtype=float)
                c0 = np.full_like(cprime["c1_0_matrix"], np.nan, dtype=float)

                # Direct-evaluation accounting.  A gene is evaluated directly at most
                # once even if the Taylor radius contracts through several rounds.
                exact_evaluated = np.zeros(len(theta_F), dtype=bool)
                exact_support_sum_count = 0
                exact_continuum_count = 0
                exact_point_count = 0
                exact_max_support_work = 0.0

                def evaluate_exact_genes(gene_mask):
                    nonlocal exact_support_sum_count
                    nonlocal exact_continuum_count
                    nonlocal exact_point_count
                    nonlocal exact_max_support_work

                    gene_idx = np.flatnonzero(
                        np.asarray(gene_mask, dtype=bool) & valid_theta & (~exact_evaluated)
                    )
                    for g in gene_idx:
                        theta_g = float(theta_F[g])
                        r_g = float(r_hat[g])

                        for j in range(n_samples):
                            value_F, backend_F, support_F = cprime_exact_theta_point(
                                mu_F[g, j], r_g, theta_g
                            )
                            value_0, backend_0, support_0 = cprime_exact_theta_point(
                                mu_0[g, j], r_g, theta_g
                            )

                            if not np.isfinite(value_F) or not np.isfinite(value_0):
                                raise RuntimeError(
                                    f"replicate {replicate}: direct theta-aware C' fallback "
                                    f"returned a nonfinite value for gene index {g}, "
                                    f"sample {j}, theta_F={theta_g:.6g}, r={r_g:.6g}."
                                )

                            cF[g, j] = float(value_F)
                            c0[g, j] = float(value_0)
                            exact_point_count += 2

                            for backend, support_work in (
                                (backend_F, support_F),
                                (backend_0, support_0),
                            ):
                                exact_support_sum_count += int(
                                    backend == "centered_support_sum"
                                )
                                exact_continuum_count += int(
                                    backend == "gamma_continuum_high_order_audit"
                                )
                                if np.isfinite(support_work):
                                    exact_max_support_work = max(
                                        exact_max_support_work,
                                        float(support_work),
                                    )

                        exact_evaluated[g] = True

                def evaluate_taylor_genes(gene_mask):
                    gene_mask = np.asarray(gene_mask, dtype=bool)
                    if not np.any(gene_mask):
                        return

                    delta_local = theta_F[gene_mask, None] - 1.0
                    cF_local = cprime["c1_F_matrix"][gene_mask].copy()
                    c0_local = cprime["c1_0_matrix"][gene_mask].copy()

                    if int(theta_cprime_order) >= 1:
                        cF_local += cprime["c2_F_matrix"][gene_mask] * delta_local
                        c0_local += cprime["c2_0_matrix"][gene_mask] * delta_local
                    if int(theta_cprime_order) >= 2:
                        cF_local += (
                            0.5
                            * cprime["c3_F_matrix"][gene_mask]
                            * delta_local**2
                        )
                        c0_local += (
                            0.5
                            * cprime["c3_0_matrix"][gene_mask]
                            * delta_local**2
                        )
                    if int(theta_cprime_order) >= 3:
                        cF_local += (
                            (1.0 / 6.0)
                            * cprime["c4_F_matrix"][gene_mask]
                            * delta_local**3
                        )
                        c0_local += (
                            (1.0 / 6.0)
                            * cprime["c4_0_matrix"][gene_mask]
                            * delta_local**3
                        )

                    cF[gene_mask] = cF_local
                    c0[gene_mask] = c0_local

                # Start with the configured maximum Taylor radius.  The effective
                # radius may contract if the local Taylor audit fails.
                effective_cutoff = configured_cutoff
                initial_exact_mask = valid_theta & (abs_delta > effective_cutoff)
                n_initial_exact_genes = int(np.sum(initial_exact_mask))
                evaluate_exact_genes(initial_exact_mask)

                audit_fallback_triggered = False
                audit_fallback_rounds = 0
                initial_audit_max_abs_error = np.nan
                force_all_direct = False

                # Final-audit placeholders.  They are overwritten on each audit round
                # and therefore describe the Taylor region that is actually retained.
                audit_idx = np.empty(0, dtype=int)
                max_abs = 0.0
                median_abs = 0.0
                q95_abs = 0.0
                audit_passed = True
                audit_support_sum_count = 0
                audit_continuum_reference_count = 0
                audit_max_support_work = np.nan

                while True:
                    if force_all_direct:
                        local_mask = np.zeros(len(theta_F), dtype=bool)
                    else:
                        local_mask = valid_theta & (abs_delta <= effective_cutoff)

                    exact_mask = valid_theta & (~local_mask)
                    evaluate_exact_genes(exact_mask)
                    evaluate_taylor_genes(local_mask)

                    n_local_genes = int(np.sum(local_mask))
                    if n_local_genes == 0:
                        # Nothing is approximated, so there is no Taylor approximation
                        # left to audit.  The direct backend itself remains audited by
                        # its discrete/continuum machinery.
                        audit_idx = np.empty(0, dtype=int)
                        max_abs = 0.0
                        median_abs = 0.0
                        q95_abs = 0.0
                        audit_passed = True
                        audit_support_sum_count = 0
                        audit_continuum_reference_count = 0
                        audit_max_support_work = np.nan
                        break

                    theta_local = theta_F[local_mask]
                    mu_F_local = mu_F[local_mask]
                    mu_0_local = mu_0[local_mask]
                    r_local = r_hat[local_mask]

                    mu_all = np.concatenate([mu_F_local.ravel(), mu_0_local.ravel()])
                    r_all = np.concatenate([
                        np.repeat(r_local, n_samples),
                        np.repeat(r_local, n_samples),
                    ])
                    theta_all = np.concatenate([
                        np.repeat(theta_local, n_samples),
                        np.repeat(theta_local, n_samples),
                    ])
                    approx_all = np.concatenate([
                        cF[local_mask].ravel(),
                        c0[local_mask].ravel(),
                    ])

                    total = len(mu_all)
                    audit_n = min(int(theta_cprime_audit_n), total)
                    rng = np.random.default_rng(
                        int(random_seed) + 7_000_021 * int(replicate)
                    )

                    forced = []
                    if total:
                        forced.extend([
                            int(np.argmin(theta_all)),
                            int(np.argmax(theta_all)),
                            int(np.argmin(mu_all)),
                            int(np.argmax(mu_all)),
                            int(np.argmin(r_all)),
                            int(np.argmax(r_all)),
                            int(np.argmin(mu_all / r_all)),
                            int(np.argmax(mu_all / r_all)),
                            int(np.argmax(np.abs(theta_all - 1.0))),
                        ])

                    forced = list(dict.fromkeys(forced))
                    if len(forced) > audit_n:
                        forced = forced[:audit_n]

                    remaining_n = max(0, audit_n - len(forced))
                    available = np.setdiff1d(
                        np.arange(total),
                        np.asarray(forced, dtype=int),
                        assume_unique=False,
                    )
                    random_idx = (
                        rng.choice(
                            available,
                            size=min(remaining_n, len(available)),
                            replace=False,
                        )
                        if remaining_n and len(available)
                        else np.empty(0, dtype=int)
                    )
                    audit_idx = np.asarray(forced + random_idx.tolist(), dtype=int)

                    exact = np.empty(len(audit_idx), dtype=float)
                    audit_backends = []
                    audit_supports = []

                    for a, idx in enumerate(audit_idx):
                        value, backend, support_work = cprime_exact_theta_point(
                            mu_all[idx],
                            r_all[idx],
                            theta_all[idx],
                        )
                        exact[a] = value
                        audit_backends.append(backend)
                        audit_supports.append(support_work)

                    approximate = approx_all[audit_idx]
                    error = approximate - exact
                    abs_error = np.abs(error)
                    finite_error = np.isfinite(abs_error)

                    if len(error) and np.any(~finite_error):
                        max_abs = np.inf
                        median_abs = (
                            float(np.median(abs_error[finite_error]))
                            if np.any(finite_error)
                            else np.inf
                        )
                        q95_abs = (
                            float(np.quantile(abs_error[finite_error], 0.95))
                            if np.any(finite_error)
                            else np.inf
                        )
                    else:
                        max_abs = float(np.max(abs_error)) if len(error) else 0.0
                        median_abs = float(np.median(abs_error)) if len(error) else 0.0
                        q95_abs = (
                            float(np.quantile(abs_error, 0.95)) if len(error) else 0.0
                        )

                    if not np.isfinite(initial_audit_max_abs_error):
                        initial_audit_max_abs_error = float(max_abs)

                    audit_support_sum_count = int(sum(
                        value == "centered_support_sum"
                        for value in audit_backends
                    ))
                    audit_continuum_reference_count = int(sum(
                        value == "gamma_continuum_high_order_audit"
                        for value in audit_backends
                    ))
                    finite_supports = [
                        value for value in audit_supports if np.isfinite(value)
                    ]
                    audit_max_support_work = (
                        float(np.max(finite_supports))
                        if finite_supports
                        else np.nan
                    )

                    audit_passed = bool(
                        np.isfinite(max_abs)
                        and max_abs <= float(theta_cprime_audit_tol)
                    )
                    if audit_passed:
                        break

                    # Do NOT loosen the tolerance.  Contract the Taylor region to just
                    # below the smallest |theta-1| represented by an audited failure.
                    bad = (~finite_error) | (
                        abs_error > float(theta_cprime_audit_tol)
                    )
                    failing_deltas = np.abs(theta_all[audit_idx[bad]] - 1.0)

                    audit_fallback_triggered = True
                    audit_fallback_rounds += 1

                    if len(failing_deltas) == 0:
                        # Defensive fallback: if an audit is non-passing but no failing
                        # point can be identified, remove the approximation entirely.
                        force_all_direct = True
                        effective_cutoff = 0.0
                        continue

                    smallest_failed_delta = float(np.min(failing_deltas))
                    if (
                        not np.isfinite(smallest_failed_delta)
                        or smallest_failed_delta <= 0.0
                    ):
                        force_all_direct = True
                        effective_cutoff = 0.0
                        continue

                    new_cutoff = float(np.nextafter(smallest_failed_delta, 0.0))
                    if not np.isfinite(new_cutoff) or new_cutoff < 0.0:
                        new_cutoff = 0.0

                    # The contraction must be strict.  If floating-point spacing keeps
                    # it from shrinking, direct evaluation is the only safe fallback.
                    if not new_cutoff < effective_cutoff:
                        force_all_direct = True
                        effective_cutoff = 0.0
                        continue

                    effective_cutoff = new_cutoff

                final_local_mask = (
                    np.zeros(len(theta_F), dtype=bool)
                    if force_all_direct
                    else valid_theta & (abs_delta <= effective_cutoff)
                )
                final_exact_mask = valid_theta & (~final_local_mask)

                # Sanity check that every valid theta gene has been populated.
                if np.any(valid_theta & ~np.all(np.isfinite(cF), axis=1)):
                    raise RuntimeError(
                        f"replicate {replicate}: nonfinite full-model C'(theta_F) values "
                        "remain after audited Taylor/direct evaluation."
                    )
                if np.any(valid_theta & ~np.all(np.isfinite(c0), axis=1)):
                    raise RuntimeError(
                        f"replicate {replicate}: nonfinite null-model C'(theta_F) values "
                        "remain after audited Taylor/direct evaluation."
                    )

                n_local_genes = int(np.sum(final_local_mask))
                n_exact_genes = int(np.sum(final_exact_mask))
                exact_gene_fraction = float(n_exact_genes / n_valid_theta)
                n_audit_promoted_exact_genes = int(
                    max(0, n_exact_genes - n_initial_exact_genes)
                )
                max_local_delta = (
                    float(np.max(abs_delta[final_local_mask]))
                    if n_local_genes
                    else np.nan
                )
                min_exact_delta = (
                    float(np.min(abs_delta[final_exact_mask]))
                    if n_exact_genes
                    else np.nan
                )

                return {
                    "cprime_F_matrix": cF,
                    "cprime_0_matrix": c0,
                    "C_F": np.sum(cF, axis=1),
                    "C_0": np.sum(c0, axis=1),
                    "valid_theta_mask": valid_theta,
                    "local_taylor_mask": final_local_mask,
                    "exact_fallback_mask": final_exact_mask,
                    "n_valid_theta": n_valid_theta,
                    "n_invalid_theta": n_invalid_theta,
                    "invalid_theta_fraction": float(n_invalid_theta / len(theta_F)),
                    "n_taylor_genes": n_local_genes,
                    "n_initial_exact_fallback_genes": n_initial_exact_genes,
                    "n_exact_fallback_genes": n_exact_genes,
                    "n_audit_promoted_exact_genes": n_audit_promoted_exact_genes,
                    "exact_fallback_gene_fraction": exact_gene_fraction,
                    "exact_fallback_point_count": int(exact_point_count),
                    "exact_fallback_support_sum_count": int(exact_support_sum_count),
                    "exact_fallback_continuum_count": int(exact_continuum_count),
                    "exact_fallback_max_support_work": (
                        float(exact_max_support_work)
                        if exact_point_count
                        else np.nan
                    ),
                    "theta_taylor_cutoff": configured_cutoff,
                    "theta_taylor_effective_cutoff": float(effective_cutoff),
                    "theta_taylor_audit_fallback_triggered": bool(
                        audit_fallback_triggered
                    ),
                    "theta_taylor_audit_fallback_rounds": int(audit_fallback_rounds),
                    "theta_taylor_initial_audit_max_abs_error": float(
                        initial_audit_max_abs_error
                    ),
                    "max_abs_theta_delta": max_delta,
                    "max_abs_theta_delta_taylor": max_local_delta,
                    "min_abs_theta_delta_exact": min_exact_delta,
                    "audit_n": int(len(audit_idx)),
                    "audit_passed": bool(audit_passed),
                    "audit_max_abs_error": float(max_abs),
                    "audit_median_abs_error": float(median_abs),
                    "audit_q95_abs_error": float(q95_abs),
                    "audit_max_support_y": audit_max_support_work,
                    "audit_support_sum_count": int(audit_support_sum_count),
                    "audit_continuum_reference_count": int(
                        audit_continuum_reference_count
                    ),
                }

            # ------------------------------------------------------------------
            # Geometry
            # ------------------------------------------------------------------

            def geometry_from_shared_score(shared_score, S_F, S_0, theta_F, d_F, d_0, weights, label):
                shared_score = np.asarray(shared_score, dtype=float)
                S_F = np.asarray(S_F, dtype=float)
                S_0 = np.asarray(S_0, dtype=float)
                theta_F = np.asarray(theta_F, dtype=float)
                weights = np.asarray(weights, dtype=float)
                delta = S_0 - S_F
                tol = 1e-8 * max(1.0, float(np.nanmedian(np.abs(S_F))))
                if np.any(np.isfinite(delta) & (delta < -tol)):
                    raise RuntimeError(f"{label}: raw nesting failed; min S0-SF={float(np.nanmin(delta)):.6g}.")
                delta = np.maximum(delta, 0.0)
                ok = (
                    np.isfinite(shared_score) & np.isfinite(delta) & np.isfinite(theta_F)
                    & np.isfinite(weights) & (shared_score > float(corrected_score_tolerance))
                    & (theta_F > 0) & (weights >= 0)
                )
                if ok.sum() < 20:
                    raise RuntimeError(f"{label}: too few usable genes for geometry.")
                w = weights[ok]
                if w.sum() <= 0:
                    raise RuntimeError(f"{label}: geometry weights sum to zero.")
                Q_F = 2.0 * theta_F[ok] * shared_score[ok]
                Q_c = 2.0 * theta_F[ok] * delta[ok]
                Q_0 = Q_F + Q_c
                kappa_F = float(np.sum(w * Q_F) / (float(d_F) * np.sum(w)))
                kappa_0 = float(np.sum(w * Q_0) / (float(d_0) * np.sum(w)))
                kappa_c = float(float(d_0) * kappa_0 - float(d_F) * kappa_F)
                lambda_c = float(kappa_c / kappa_F)
                if min(kappa_F, kappa_0, kappa_c, lambda_c) <= 0 or not np.all(np.isfinite([kappa_F, kappa_0, kappa_c, lambda_c])):
                    raise RuntimeError(
                        f"{label}: invalid geometry kF={kappa_F}, k0={kappa_0}, kc={kappa_c}, lambda={lambda_c}."
                    )
                return {
                    "kappa_F": kappa_F, "kappa_0": kappa_0, "kappa_c": kappa_c,
                    "lambda_c": lambda_c, "n_genes": int(ok.sum()),
                    "weight_sum": float(w.sum()), "mean_weight": float(np.mean(w)),
                }

            def geometry_from_two_scores(score_F, score_0, theta_F, d_F, d_0, weights, label):
                score_F = np.asarray(score_F, dtype=float)
                score_0 = np.asarray(score_0, dtype=float)
                theta_F = np.asarray(theta_F, dtype=float)
                weights = np.asarray(weights, dtype=float)
                ok = (
                    np.isfinite(score_F) & np.isfinite(score_0) & np.isfinite(theta_F)
                    & np.isfinite(weights) & (score_F > float(corrected_score_tolerance))
                    & (score_0 > float(corrected_score_tolerance)) & (theta_F > 0) & (weights >= 0)
                )
                if ok.sum() < 20:
                    raise RuntimeError(f"{label}: too few usable genes for theta-aware geometry.")
                w = weights[ok]
                if w.sum() <= 0:
                    raise RuntimeError(f"{label}: theta-aware geometry weights sum to zero.")
                Q_F = 2.0 * theta_F[ok] * score_F[ok]
                Q_0 = 2.0 * theta_F[ok] * score_0[ok]
                kappa_F = float(np.sum(w * Q_F) / (float(d_F) * np.sum(w)))
                kappa_0 = float(np.sum(w * Q_0) / (float(d_0) * np.sum(w)))
                kappa_c = float(float(d_0) * kappa_0 - float(d_F) * kappa_F)
                lambda_c = float(kappa_c / kappa_F)
                if min(kappa_F, kappa_0, kappa_c, lambda_c) <= 0 or not np.all(np.isfinite([kappa_F, kappa_0, kappa_c, lambda_c])):
                    raise RuntimeError(
                        f"{label}: invalid theta-aware geometry kF={kappa_F}, k0={kappa_0}, kc={kappa_c}, lambda={lambda_c}."
                    )
                return {
                    "kappa_F": kappa_F, "kappa_0": kappa_0, "kappa_c": kappa_c,
                    "lambda_c": lambda_c, "n_genes": int(ok.sum()),
                    "weight_sum": float(w.sum()), "mean_weight": float(np.mean(w)),
                }

            def nested_S0(S_F, S_0, lambda_c, label):
                delta = S_0 - S_F
                tol = 1e-8 * max(1.0, float(np.nanmedian(np.abs(S_F))))
                if np.any(np.isfinite(delta) & (delta < -tol)):
                    raise RuntimeError(f"{label}: raw NB nesting failed.")
                return S_F + np.maximum(delta, 0.0) / float(lambda_c)

            def fit_raw_nested(S_F, S_0, theta_F, a_0, n, geom, label):
                S0d = nested_S0(S_F, S_0, geom["lambda_c"], label)
                web0 = estimate_web(S0d, a_0)
                T = modified_web_statistic(S_F, S0d, theta_F, web0["theta"], n)
                return {"S_0_dagger": S0d, "WEB_0": web0, "T": T, "p": chi2.sf(T, 1)}

            def normalized_statistic(S_F, S_0_used, C_F, C_0, theta_F, theta_0, n):
                R_F = S_F - C_F
                R_0 = S_0_used - C_0
                ok = (
                    np.isfinite(R_F) & np.isfinite(R_0) & np.isfinite(theta_F) & np.isfinite(theta_0)
                    & (R_F > float(corrected_score_tolerance)) & (R_0 > float(corrected_score_tolerance))
                    & (theta_F > 0) & (theta_0 > 0)
                )
                T_raw = np.full(len(S_F), np.nan)
                T_raw[ok] = (
                    2.0 * (theta_0[ok] * R_0[ok] - theta_F[ok] * R_F[ok])
                    + float(n) * np.log(theta_F[ok] / theta_0[ok])
                    + 2.0 * (C_0[ok] - C_F[ok])
                )
                T = np.full(len(S_F), np.nan)
                T[ok] = np.maximum(T_raw[ok], 0.0)
                p = np.full(len(S_F), np.nan)
                p[ok] = chi2.sf(T[ok], 1)
                return {
                    "R_F": R_F, "R_0": R_0, "valid_mask": ok,
                    "T_raw": T_raw, "T": T, "p": p,
                    "negative_fraction": float(np.mean(T_raw[ok] < 0)) if np.any(ok) else np.nan,
                }

            def fit_normalized_nested(S_F, S_0, C_F, C_0, theta_F, a_0, n, geom, label):
                S0d = nested_S0(S_F, S_0, geom["lambda_c"], label)
                R0d = S0d - C_0
                web0 = estimate_web_masked(R0d, a_0, f"{label} null")
                test = normalized_statistic(S_F, S0d, C_F, C_0, theta_F, web0["theta"], n)
                return {"S_0_dagger": S0d, "R_0_dagger": R0d, "WEB_0": web0, **test}


            # ------------------------------------------------------------------
            # Gamma-Gamma DEB residual-precision EB
            # ------------------------------------------------------------------

            def fit_deb_masked(S, a, label):
                S = np.asarray(S, dtype=float)
                valid = np.isfinite(S) & (S > float(corrected_score_tolerance))
                if valid.sum() < 100:
                    raise RuntimeError(f"{label}: too few positive corrected scores for DEB.")
                x = S[valid]
                a0 = float(a)
                theta_pool = float(np.clip(a0 * len(x) / np.sum(x), float(theta_floor), float(theta_cap)))

                def objective(par):
                    alpha0 = float(np.exp(par[0]))
                    theta_bar = float(np.exp(par[1]))
                    beta0 = alpha0 / theta_bar
                    ll = (
                        (a0 - 1.0) * np.log(x)
                        - a0 * np.log(beta0)
                        - (a0 + alpha0) * np.log1p(x / beta0)
                        - betaln(a0, alpha0)
                    )
                    return -float(np.mean(ll)) if np.all(np.isfinite(ll)) else 1e100

                fits = []
                theta_median = float(np.clip(np.median(a0 / x), float(theta_floor), float(theta_cap)))
                for alpha_start in (0.05, 0.2, 1.0, 5.0, 25.0, 100.0, 1000.0, 1e4):
                    for theta_start in (theta_pool, theta_median):
                        fit = minimize(
                            objective,
                            x0=np.log([alpha_start, theta_start]),
                            method="L-BFGS-B",
                            bounds=[
                                (np.log(1e-8), np.log(1e9)),
                                (np.log(float(theta_floor)), np.log(float(theta_cap))),
                            ],
                            options={"maxiter": int(deb_precision_maxiter), "ftol": 1e-12},
                        )
                        if np.isfinite(fit.fun):
                            fits.append(fit)
                if not fits:
                    raise RuntimeError(f"{label}: DEB fit failed.")
                best = min(fits, key=lambda x: x.fun)
                alpha0 = float(np.exp(best.x[0]))
                theta_bar = float(np.exp(best.x[1]))
                beta0 = alpha0 / theta_bar
                theta_valid = (a0 + alpha0) / (x + beta0)
                theta = np.full(len(S), np.nan)
                theta[valid] = np.clip(theta_valid, float(theta_floor), float(theta_cap))
                return {
                    "theta": theta,
                    "valid_mask": valid,
                    "alpha0": alpha0,
                    "beta0": beta0,
                    "prior_mean_theta": theta_bar,
                    "objective": float(best.fun),
                    "n_valid": int(valid.sum()),
                    "n_invalid": int((~valid).sum()),
                    "invalid_fraction": float((~valid).mean()),
                }


            # ------------------------------------------------------------------
            # Exact normalized-DNB precision backend.
            #
            # Canonical M03 invokes this backend mandatorily when the first-order
            # full residual score R_F <= 0. Optional exact-refinement modes reuse
            # the SAME backend for additional selected genes while leaving all
            # genome-wide M03 quantities frozen.
            # ------------------------------------------------------------------

            _exact_gamma_quad_cache = {}
            _exact_gamma_quad_lock = threading.Lock()

            def exact_log_nb_pmf(yv, mu, r):
                yv = np.asarray(yv, dtype=float)
                mu = float(mu)
                r = float(r)
                p = r / (r + mu)
                return (
                    gammaln(yv + r)
                    - gammaln(r)
                    - gammaln(yv + 1.0)
                    + r * np.log(p)
                    + xlogy(yv, 1.0 - p)
                )

            def exact_nb_quantile_bounds(mu, r, tail_probability=None):
                if tail_probability is None:
                    tail_probability = cprime_tail_probability
                tail_probability = float(np.clip(tail_probability, 1e-15, 1e-4))
                side = 0.5 * tail_probability
                p = float(r) / (float(r) + float(mu))
                lo_q = nbinom.ppf(side, float(r), p)
                hi_q = nbinom.ppf(1.0 - side, float(r), p)
                if not np.isfinite(lo_q):
                    lo_q = 0.0
                if not np.isfinite(hi_q):
                    sd = np.sqrt(float(mu) + float(mu) * float(mu) / float(r))
                    hi_q = float(mu) + 14.0 * sd + 50.0
                lo = max(0, int(np.floor(lo_q)))
                hi = max(lo, int(np.ceil(hi_q)))
                return lo, hi

            def exact_get_gamma_quad(n_quad):
                n_quad = int(n_quad)
                with _exact_gamma_quad_lock:
                    cached = _exact_gamma_quad_cache.get(n_quad)
                    if cached is not None:
                        return cached
                    nodes, weights = roots_legendre(n_quad)
                    eps = max(min(cprime_tail_probability, 1e-10), 1e-14)
                    u = 0.5 * (nodes + 1.0) * (1.0 - 2.0 * eps) + eps
                    w = 0.5 * (1.0 - 2.0 * eps) * weights
                    w = w / np.sum(w)
                    cached = (u, w)
                    _exact_gamma_quad_cache[n_quad] = cached
                    return cached

            def exact_continuum_allowed(mu, r):
                return bool(
                    np.isfinite(mu)
                    and np.isfinite(r)
                    and mu >= cprime_continuum_audit_min_mu
                    and r > 0
                    and (mu / r) >= cprime_continuum_audit_min_mu_over_r
                )

            def exact_make_continuum_arrays(mu, r, n_quad):
                u, w = exact_get_gamma_quad(n_quad)
                x = gamma_dist.ppf(u, a=float(r), scale=1.0 / float(r))
                yv = float(mu) * x
                s = half_nb_deviance(yv, float(mu), float(r))
                return np.asarray(s, dtype=float), np.log(np.asarray(w, dtype=float))

            def exact_prepare_point_state(mu, r):
                """
                Prepare one point for exact normalized-DNB gene profiling.

                Exact discrete evaluation is used whenever the initial NB support
                window is within cprime_support_max. cprime_grid_exact_width_max
                controls only whether the exact support is cached in memory or
                evaluated chunk-by-chunk; it does not choose exact versus continuum.

                Gamma continuum is reserved for initial support windows beyond
                cprime_support_max and only when the frozen continuum eligibility
                conditions are satisfied.
                """
                mu = float(mu)
                r = float(r)
                lo, hi = exact_nb_quantile_bounds(mu, r)
                width = hi - lo + 1

                if width <= cprime_grid_exact_width_max:
                    yv = np.arange(lo, hi + 1, dtype=float)
                    logp = exact_log_nb_pmf(yv, mu, r)
                    s = half_nb_deviance(yv, mu, r)
                    return {
                        "kind": "discrete_cached",
                        "mu": mu,
                        "r": r,
                        "lo": lo,
                        "hi": hi,
                        "logp": logp,
                        "s": s,
                        "logmass": float(logsumexp(logp)),
                    }

                if width <= cprime_support_max:
                    return {
                        "kind": "discrete_chunked",
                        "mu": mu,
                        "r": r,
                        "lo": lo,
                        "hi": hi,
                    }

                if exact_continuum_allowed(mu, r):
                    return {"kind": "continuum", "mu": mu, "r": r, "quad_cache": {}}

                raise RuntimeError(
                    f"Exact-theta reference window requires {width:,} points for "
                    f"mu={mu:.6g}, r={r:.6g}, beyond cprime_support_max and outside "
                    "the audited high-count continuum regime."
                )

            def exact_combine_logZ_mean(state, logw, s):
                chunk_logZ = float(logsumexp(logw))
                if not np.isfinite(chunk_logZ):
                    return state
                q = np.exp(logw - chunk_logZ)
                chunk_mean = float(np.sum(q * s))
                if state is None:
                    return [chunk_logZ, chunk_mean]
                old_logZ, old_mean = state
                new_logZ = float(np.logaddexp(old_logZ, chunk_logZ))
                a = float(np.exp(old_logZ - new_logZ))
                b = float(np.exp(chunk_logZ - new_logZ))
                return [new_logZ, a * old_mean + b * chunk_mean]

            def exact_chunked_window_state(mu, r, theta, lo, hi):
                tilted = None
                ref_logZ = None
                for start in range(int(lo), int(hi) + 1, cprime_chunk_size):
                    stop = min(int(hi) + 1, start + cprime_chunk_size)
                    yv = np.arange(start, stop, dtype=float)
                    logp = exact_log_nb_pmf(yv, mu, r)
                    s = half_nb_deviance(yv, mu, r)
                    tilted = exact_combine_logZ_mean(
                        tilted, logp - (theta - 1.0) * s, s
                    )
                    lref = float(logsumexp(logp))
                    ref_logZ = lref if ref_logZ is None else float(np.logaddexp(ref_logZ, lref))
                return tilted, ref_logZ

            def exact_eval_continuum(state, theta):
                n_fine = int(cprime_continuum_audit_quad_n)
                n_max = int(cprime_continuum_audit_quad_n_max)
                while True:
                    n_coarse = max(64, n_fine // 2)
                    for n_quad in (n_coarse, n_fine):
                        if n_quad not in state["quad_cache"]:
                            state["quad_cache"][n_quad] = exact_make_continuum_arrays(
                                state["mu"], state["r"], n_quad
                            )

                    def at_order(n_quad):
                        s, logw = state["quad_cache"][n_quad]
                        lw = logw - (theta - 1.0) * s
                        logZ = float(logsumexp(lw))
                        q = np.exp(lw - logZ)
                        mean_s = float(np.sum(q * s))
                        cprime_theta = mean_s - 0.5 / theta
                        logC = -0.5 * np.log(theta) - logZ
                        return cprime_theta, logC

                    cp_coarse, _ = at_order(n_coarse)
                    cp_fine, lc_fine = at_order(n_fine)
                    err = abs(cp_fine - cp_coarse)
                    if np.isfinite(err) and err <= cprime_continuum_audit_selfcheck_tol:
                        return cp_fine, lc_fine, f"continuum_{n_fine}", n_fine
                    if n_fine >= n_max:
                        raise RuntimeError(
                            "Exact-theta continuum self-check failed for "
                            f"mu={state['mu']:.6g}, r={state['r']:.6g}, theta={theta:.6g}: "
                            f"|fine-coarse|={err:.6g} > "
                            f"{cprime_continuum_audit_selfcheck_tol:.6g}."
                        )
                    n_fine = min(2 * n_fine, n_max)

            def exact_eval_point_state(state, theta):
                theta = float(theta)
                if not np.isfinite(theta) or theta <= 0:
                    raise ValueError("theta must be finite and positive.")

                if state["kind"] == "continuum":
                    return exact_eval_continuum(state, theta)

                mu = state["mu"]
                r = state["r"]
                lo = state["lo"]
                hi = state["hi"]

                if state["kind"] == "discrete_cached" and theta >= 1.0:
                    logp = state["logp"]
                    s = state["s"]
                    lw = logp - (theta - 1.0) * s
                    logZraw = float(logsumexp(lw))
                    q = np.exp(lw - logZraw)
                    mean_s = float(np.sum(q * s))
                    logZnorm = logZraw - state["logmass"]
                    return (
                        mean_s - 0.5 / theta,
                        -0.5 * np.log(theta) - logZnorm,
                        "discrete_cached",
                        hi - lo + 1,
                    )

                tilted, ref_logZ = exact_chunked_window_state(mu, r, theta, lo, hi)
                if tilted is None or ref_logZ is None:
                    raise RuntimeError("Initial exact-theta discrete sum failed.")

                total_points = hi - lo + 1
                if theta < 1.0:
                    step = max(1024, min(total_points // 2, 250_000))
                    stable_rounds = 0
                    mass_tol = max(cprime_tail_probability, 1e-12)
                    mean_tol = max(10.0 * cprime_tail_probability, 1e-10)

                    while stable_rounds < 2:
                        old_logZ, old_mean = tilted
                        added_tilted = None
                        added_ref_logZ = None
                        added_points = 0

                        if lo > 0:
                            new_lo = max(0, lo - step)
                            left_t, left_r = exact_chunked_window_state(
                                mu, r, theta, new_lo, lo - 1
                            )
                            if left_t is not None:
                                added_tilted = left_t
                            if left_r is not None:
                                added_ref_logZ = left_r
                            added_points += lo - new_lo
                            lo = new_lo

                        new_hi = hi + step
                        right_t, right_r = exact_chunked_window_state(
                            mu, r, theta, hi + 1, new_hi
                        )
                        if right_t is not None:
                            if added_tilted is None:
                                added_tilted = right_t
                            else:
                                lz, lm = added_tilted
                                rz, rm = right_t
                                nz = float(np.logaddexp(lz, rz))
                                added_tilted = [
                                    nz,
                                    float(np.exp(lz - nz)) * lm
                                    + float(np.exp(rz - nz)) * rm,
                                ]
                        if right_r is not None:
                            added_ref_logZ = (
                                right_r
                                if added_ref_logZ is None
                                else float(np.logaddexp(added_ref_logZ, right_r))
                            )
                        added_points += new_hi - hi
                        hi = new_hi

                        if added_tilted is None:
                            stable_rounds += 1
                            total_points += added_points
                            continue

                        add_logZ, add_mean = added_tilted
                        new_logZ = float(np.logaddexp(old_logZ, add_logZ))
                        old_weight = float(np.exp(old_logZ - new_logZ))
                        add_weight = float(np.exp(add_logZ - new_logZ))
                        new_mean = old_weight * old_mean + add_weight * add_mean
                        tilted = [new_logZ, new_mean]
                        if added_ref_logZ is not None:
                            ref_logZ = float(np.logaddexp(ref_logZ, added_ref_logZ))

                        if add_weight <= mass_tol and abs(new_mean - old_mean) <= mean_tol:
                            stable_rounds += 1
                        else:
                            stable_rounds = 0

                        total_points += added_points
                        if total_points > cprime_support_max:
                            if exact_continuum_allowed(mu, r):
                                cont_state = {
                                    "kind": "continuum",
                                    "mu": mu,
                                    "r": r,
                                    "quad_cache": {},
                                }
                                return exact_eval_continuum(cont_state, theta)
                            raise RuntimeError(
                                f"theta<1 exact discrete expansion exceeded "
                                f"{cprime_support_max:,} points."
                            )
                        step = min(step * 2, 1_000_000)

                logZraw, mean_s = tilted
                logZnorm = float(logZraw - ref_logZ)
                return (
                    float(mean_s - 0.5 / theta),
                    float(-0.5 * np.log(theta) - logZnorm),
                    "discrete_dynamic",
                    int(total_points),
                )

            def exact_prepare_model_states(mu_row, r_value):
                return [
                    exact_prepare_point_state(mu, r_value)
                    for mu in np.asarray(mu_row, dtype=float)
                ]

            def exact_eval_model(states, theta):
                cpsum = 0.0
                logCsum = 0.0
                max_work = 0
                sources = set()
                for state in states:
                    cp, lc, source, work = exact_eval_point_state(state, theta)
                    cpsum += float(cp)
                    logCsum += float(lc)
                    sources.add(str(source))
                    if np.isfinite(work):
                        max_work = max(max_work, int(work))
                return cpsum, logCsum, "+".join(sorted(sources)), max_work

            def exact_model_smin_sum(mu_row, r_value):
                mu_row = np.asarray(mu_row, dtype=float)
                fl = np.maximum(np.floor(mu_row), 0.0)
                ce = np.maximum(np.ceil(mu_row), 0.0)
                sf = half_nb_deviance(fl, mu_row, float(r_value))
                sc = half_nb_deviance(ce, mu_row, float(r_value))
                return float(np.minimum(sf, sc).sum())

            def solve_exact_precision_profile(S, a, states, smin_sum):
                S = float(S)
                a = float(a)
                cache = {}

                def evaluated(theta):
                    theta = float(theta)
                    if theta not in cache:
                        cp, lc, source, work = exact_eval_model(states, theta)
                        score = a / theta - S + cp
                        profile = a * np.log(theta) - (theta - 1.0) * S + lc
                        cache[theta] = (score, profile, cp, lc, source, work)
                    return cache[theta]

                score1, obj1, cp1, lc1, source1, work1 = evaluated(1.0)
                result = {
                    "root_status": None,
                    "theta_hat_exact": np.nan,
                    "score_at_1": float(score1),
                    "profile_at_1": float(obj1),
                    "cprime_at_1": float(cp1),
                    "logC_at_1": float(lc1),
                    "smin_sum": float(smin_sum),
                    "S_minus_smin_sum": float(S - smin_sum),
                    "score_at_hat": np.nan,
                    "profile_at_hat": np.nan,
                    "logC_at_hat": np.nan,
                    "cprime_at_hat": np.nan,
                    "local_profile_max": False,
                    "normalizer_source_at_hat": None,
                    "normalizer_work_at_hat": np.nan,
                    "n_score_evaluations": 1,
                }

                try:
                    if abs(score1) <= exact_score_zero_tol:
                        root = 1.0
                        bracket = (1.0, 1.0)
                    elif score1 > 0:
                        lo = 1.0
                        hi = 2.0
                        shi = evaluated(hi)[0]
                        while shi > 0 and hi < exact_theta_search_max:
                            lo = hi
                            hi = min(2.0 * hi, exact_theta_search_max)
                            shi = evaluated(hi)[0]
                            if hi >= exact_theta_search_max:
                                break
                        if shi > 0:
                            result["root_status"] = "root_above_search_max"
                            return result
                        root = brentq(
                            lambda th: evaluated(th)[0],
                            lo,
                            hi,
                            xtol=exact_score_root_xtol,
                            rtol=exact_score_root_rtol,
                            maxiter=200,
                        )
                        bracket = (lo, hi)
                    else:
                        hi = 1.0
                        lo = 0.5
                        slo = evaluated(lo)[0]
                        while slo < 0 and lo > exact_theta_search_min:
                            hi = lo
                            lo = max(0.5 * lo, exact_theta_search_min)
                            slo = evaluated(lo)[0]
                            if lo <= exact_theta_search_min:
                                break
                        if slo < 0:
                            result["root_status"] = "root_below_search_min"
                            return result
                        root = brentq(
                            lambda th: evaluated(th)[0],
                            lo,
                            hi,
                            xtol=exact_score_root_xtol,
                            rtol=exact_score_root_rtol,
                            maxiter=200,
                        )
                        bracket = (lo, hi)

                    scoreh, objh, cph, lch, sourceh, workh = evaluated(root)
                    result.update({
                        "root_status": "finite_root",
                        "theta_hat_exact": float(root),
                        "score_at_hat": float(scoreh),
                        "profile_at_hat": float(objh),
                        "logC_at_hat": float(lch),
                        "cprime_at_hat": float(cph),
                        "normalizer_source_at_hat": str(sourceh),
                        "normalizer_work_at_hat": float(workh),
                    })

                    left = max(exact_theta_search_min, root * np.exp(-exact_profile_local_step))
                    right = min(exact_theta_search_max, root * np.exp(exact_profile_local_step))
                    obj_left = evaluated(left)[1]
                    obj_right = evaluated(right)[1]
                    tol_obj = 1e-8 * max(1.0, abs(objh))
                    result["local_profile_max"] = bool(
                        objh >= obj_left - tol_obj and objh >= obj_right - tol_obj
                    )
                    result["root_bracket_lo"] = float(bracket[0])
                    result["root_bracket_hi"] = float(bracket[1])
                    result["n_score_evaluations"] = int(len(cache))
                    return result

                except Exception as exc:
                    result["root_status"] = "numerical_failure"
                    result["error_type"] = type(exc).__name__
                    result["error_message"] = str(exc)
                    result["n_score_evaluations"] = int(len(cache))
                    return result


            # ------------------------------------------------------------------
            # Resumable canonical M03 fit + selective exact-refinement layer
            # ------------------------------------------------------------------
            from pathlib import Path
            from concurrent.futures import ThreadPoolExecutor, as_completed
            from time import perf_counter, sleep
            import hashlib
            import json
            import os
            import pickle
            import shutil
            import uuid
            import traceback
            import threading
            import warnings
            from threadpoolctl import threadpool_limits

            if mode not in {"observed", "describe"}:
                raise ValueError("M03 supports mode='describe' or mode='observed'.")

            EXACT_MODE_ALIASES = {
                1: "default",
                2: "called",
                3: "called_buffer",
                4: "top_n",
                5: "top_fraction",
                6: "all",
                7: "gene_list",
                "1": "default",
                "2": "called",
                "3": "called_buffer",
                "4": "top_n",
                "5": "top_fraction",
                "6": "all",
                "7": "gene_list",
                "default": "default",
                "called": "called",
                "called_buffer": "called_buffer",
                "top_n": "top_n",
                "top_fraction": "top_fraction",
                "all": "all",
                "gene_list": "gene_list",
            }
            _exact_mode_key = exact_mode
            if isinstance(_exact_mode_key, str):
                _exact_mode_key = _exact_mode_key.strip().lower()
            if _exact_mode_key not in EXACT_MODE_ALIASES:
                raise ValueError(
                    "exact_mode must be one of: 'default', 'called', 'called_buffer', "
                    "'top_n', 'top_fraction', 'all', 'gene_list' (or integer 1..7)."
                )
            exact_mode_used = EXACT_MODE_ALIASES[_exact_mode_key]

            if mode == "describe":
                return {
                    "method_name": method_name,
                    "method_number": 3,
                    "r_estimator": r_estimator,
                    "theta_method": theta_method,
                    "geometry_method": geometry_method,
                    "geometry_label": "C'(1) ordinary + exact full fallback + support-feasible null",
                    "exact_full_trigger": "R_F_first <= corrected_score_tolerance",
                    "support_floor_policy": "genome-wide S0dag >= S0min",
                    "exact_refinement_modes": (
                        "default", "called", "called_buffer", "top_n",
                        "top_fraction", "all", "gene_list"
                    ),
                    "base_version": WEB_DNB_M03_BASE_VERSION,
                    "code_version": WEB_DNB_M03_VERSION,
                }

            if random_seed is None:
                raise ValueError(
                    "random_seed is required for an initial observed fit. "
                    "A refinement call may inherit it through refine_from."
                )
            if checkpoint_dir is None:
                raise ValueError(
                    "checkpoint_dir is required for observed fitting; resumability is mandatory."
                )

            if p_cutoff is not None and fdr_cutoff is not None:
                raise ValueError("p_cutoff and fdr_cutoff are mutually exclusive.")
            if p_cutoff is not None:
                p_cutoff = float(p_cutoff)
                if not np.isfinite(p_cutoff) or not (0.0 < p_cutoff <= 1.0):
                    raise ValueError("p_cutoff must be finite and lie in (0, 1].")
            if fdr_cutoff is not None:
                fdr_cutoff = float(fdr_cutoff)
                if not np.isfinite(fdr_cutoff) or not (0.0 < fdr_cutoff <= 1.0):
                    raise ValueError("fdr_cutoff must be finite and lie in (0, 1].")

            if exact_mode_used == "top_n":
                if exact_top_n is None:
                    raise ValueError("exact_top_n is required when exact_mode='top_n'.")
                if isinstance(exact_top_n, bool) or int(exact_top_n) != exact_top_n or int(exact_top_n) < 1:
                    raise ValueError("exact_top_n must be a positive integer.")
                exact_top_n = int(exact_top_n)

            if exact_mode_used == "top_fraction":
                if exact_top_fraction is None:
                    raise ValueError(
                        "exact_top_fraction is required when exact_mode='top_fraction'."
                    )
                exact_top_fraction = float(exact_top_fraction)
                if (
                    not np.isfinite(exact_top_fraction)
                    or exact_top_fraction <= 0.0
                    or exact_top_fraction > 1.0
                ):
                    raise ValueError("exact_top_fraction must lie in (0, 1].")

            if exact_mode_used == "gene_list":
                if exact_gene_ids is None:
                    raise ValueError("exact_gene_ids is required when exact_mode='gene_list'.")
                if isinstance(exact_gene_ids, (str, bytes)):
                    exact_gene_ids = [exact_gene_ids]
                else:
                    exact_gene_ids = list(exact_gene_ids)
                if len(exact_gene_ids) == 0:
                    raise ValueError("exact_gene_ids must contain at least one gene identifier.")
                exact_gene_ids = [str(x) for x in exact_gene_ids]

            checkpoint_path = Path(checkpoint_dir)

            if bool(fresh_start) and checkpoint_path.exists():
                shutil.rmtree(checkpoint_path)
            checkpoint_path.mkdir(parents=True, exist_ok=True)
            stage_dir = checkpoint_path / "stages"
            exact_dir = checkpoint_path / "exact_gene_checkpoints"
            refinement_dir = checkpoint_path / "refinement_runs"
            stage_dir.mkdir(parents=True, exist_ok=True)
            exact_dir.mkdir(parents=True, exist_ok=True)
            refinement_dir.mkdir(parents=True, exist_ok=True)

            fit_t0 = perf_counter()
            refinement_warnings = []

            def log(msg):
                if int(verbose) > 0:
                    print(msg, flush=True)

            def warn_user(msg):
                msg = str(msg)
                refinement_warnings.append(msg)
                warnings.warn(msg, RuntimeWarning, stacklevel=2)

            def robust_pickle_dump(obj, path, attempts=12):
                path = Path(path)
                path.parent.mkdir(parents=True, exist_ok=True)
                last_exc = None
                for attempt in range(int(attempts)):
                    tmp = path.parent / f".tmp_{os.getpid()}_{uuid.uuid4().hex[:8]}"
                    try:
                        with open(tmp, "wb") as handle:
                            pickle.dump(obj, handle, protocol=pickle.HIGHEST_PROTOCOL)
                            handle.flush()
                            try:
                                os.fsync(handle.fileno())
                            except OSError:
                                pass
                        os.replace(tmp, path)
                        return
                    except PermissionError as exc:
                        last_exc = exc
                        try:
                            if tmp.exists():
                                tmp.unlink()
                        except OSError:
                            pass
                        sleep(min(0.05 * (2 ** attempt), 1.0))
                    except Exception:
                        try:
                            if tmp.exists():
                                tmp.unlink()
                        except OSError:
                            pass
                        raise
                if last_exc is not None:
                    raise last_exc

            def load_pickle(path):
                with open(path, "rb") as handle:
                    return pickle.load(handle)

            def stage_load(name):
                path = stage_dir / f"{name}.pkl"
                if bool(resume) and path.exists():
                    payload = load_pickle(path)
                    if payload.get("status") == "success":
                        log(f"[resume] loaded stage {name}")
                        return payload["data"]
                return None

            def stage_save(name, data):
                robust_pickle_dump(
                    {"status": "success", "data": data},
                    stage_dir / f"{name}.pkl",
                )
                log(f"[checkpoint] saved stage {name}")

            def _hash_string_array(hasher, arr, label):
                arr = np.asarray(arr).astype(str)
                hasher.update(str(label).encode("utf-8"))
                hasher.update(str(arr.shape).encode("utf-8"))
                for value in arr:
                    hasher.update(value.encode("utf-8"))
                    hasher.update(b"\x00")

            def compute_analysis_fingerprint(y_arr, group_arr, gene_arr, sample_arr):
                y_arr = np.ascontiguousarray(np.asarray(y_arr, dtype=np.float64))
                group_arr = np.asarray(group_arr).astype(str)
                gene_arr = np.asarray(gene_arr).astype(str)
                sample_arr = np.asarray(sample_arr).astype(str)

                h = hashlib.sha256()
                h.update(WEB_DNB_M03_VERSION.encode("utf-8"))
                h.update(json.dumps(
                    numerical_settings,
                    sort_keys=True,
                    separators=(",", ":"),
                    default=str,
                ).encode("utf-8"))
                h.update(str(y_arr.shape).encode("utf-8"))
                h.update(y_arr.tobytes(order="C"))
                _hash_string_array(h, group_arr, "group")
                _hash_string_array(h, gene_arr, "gene_ids")
                _hash_string_array(h, sample_arr, "sample_names")
                return h.hexdigest()

            def input_fingerprint_if_available():
                if _input_y is None:
                    return None

                y0 = np.asarray(_input_y, dtype=float)
                group0 = np.asarray(_input_group).astype(str)
                gene0 = np.asarray(_input_gene_ids).astype(str)

                if y0.ndim != 2:
                    raise ValueError("y must be a 2D genes x samples matrix.")
                if len(gene0) != y0.shape[0]:
                    raise ValueError("gene_ids length must equal y.shape[0].")
                if len(group0) != y0.shape[1]:
                    raise ValueError("group length must equal y.shape[1].")
                if np.any(~np.isfinite(y0)) or np.any(y0 < 0):
                    raise ValueError("y contains nonfinite or negative counts.")

                keep0 = y0.sum(axis=1) > 0
                y0 = y0[keep0]
                gene0 = gene0[keep0]

                if _input_sample_names is None:
                    sample0 = np.asarray(
                        [f"sample_{j+1}" for j in range(y0.shape[1])],
                        dtype=str,
                    )
                else:
                    sample0 = np.asarray(_input_sample_names).astype(str)
                    if len(sample0) != y0.shape[1]:
                        raise ValueError("sample_names length must equal y.shape[1].")

                return compute_analysis_fingerprint(y0, group0, gene0, sample0)

            expected_input_fingerprint = input_fingerprint_if_available()

            # -----------------------------
            # Stage 1: preprocessing, r_g, means, raw deviances
            # -----------------------------
            core = stage_load("01_core")
            core_loaded_from_checkpoint = core is not None
            if core is None:
                if _input_y is None:
                    raise RuntimeError(
                        "Stage 01_core is unavailable and y was not supplied. "
                        "A refinement-only call requires the original checkpoint directory."
                    )

                y0 = np.asarray(_input_y, dtype=float)
                group0 = np.asarray(_input_group).astype(str)
                gene_ids0 = np.asarray(_input_gene_ids).astype(str)
                if y0.ndim != 2:
                    raise ValueError("y must be a 2D genes x samples matrix.")
                if len(gene_ids0) != y0.shape[0]:
                    raise ValueError("gene_ids length must equal y.shape[0].")
                if len(group0) != y0.shape[1]:
                    raise ValueError("group length must equal y.shape[1].")
                if np.any(~np.isfinite(y0)) or np.any(y0 < 0):
                    raise ValueError("y contains nonfinite or negative counts.")
                levels = np.unique(group0)
                if len(levels) != 2:
                    raise ValueError("This observed-data engine requires exactly two groups.")
                if np.any(y0.sum(axis=0) <= 0):
                    raise ValueError("Every sample must have positive total count.")

                keep = y0.sum(axis=1) > 0
                y1 = y0[keep]
                gene_ids1 = gene_ids0[keep]
                if y1.shape[0] < 200:
                    raise RuntimeError(
                        "Too few nonzero genes for the genome-wide EB/ASH fits."
                    )

                if _input_sample_names is None:
                    sample_names1 = np.asarray(
                        [f"sample_{j+1}" for j in range(y1.shape[1])],
                        dtype=str,
                    )
                else:
                    sample_names1 = np.asarray(_input_sample_names).astype(str)
                    if len(sample_names1) != y1.shape[1]:
                        raise ValueError("sample_names length must equal y.shape[1].")

                lib_size = y1.sum(axis=0)
                norm_factor = calc_tmm_factors(y1) if use_tmm else np.ones(y1.shape[1])
                effective_lib = lib_size * norm_factor
                exposure = effective_lib / np.exp(np.mean(np.log(effective_lib)))

                r_fit = estimate_r_dp(y1, exposure, group0)
                r_hat = np.asarray(r_fit["r_hat"], dtype=float)
                alpha_hat = np.asarray(r_fit["alpha_hat"], dtype=float)
                mu_F, q_by_group = fit_nb_group_means(
                    y1, exposure, group0, alpha_hat
                )
                q_0 = solve_nb_base_mean(y1, exposure, alpha_hat)
                mu_0 = q_0[:, None] * exposure[None, :]
                r_matrix = r_hat[:, None]
                S_F = half_nb_deviance(y1, mu_F, r_matrix).sum(axis=1)
                S_0 = half_nb_deviance(y1, mu_0, r_matrix).sum(axis=1)

                n = int(y1.shape[1])
                d_F = int(n - 2)
                d_0 = int(n - 1)
                if d_F <= 0:
                    raise RuntimeError(
                        "Full-model residual degrees of freedom must be positive."
                    )
                a_F = d_F / 2.0
                a_0 = d_0 / 2.0

                core = {
                    "y": y1,
                    "group": group0,
                    "gene_ids": gene_ids1,
                    "sample_names": sample_names1,
                    "exposure": exposure,
                    "r_fit": r_fit,
                    "r_hat": r_hat,
                    "alpha_hat": alpha_hat,
                    "mu_F": mu_F,
                    "q_by_group": q_by_group,
                    "mu_0": mu_0,
                    "S_F": S_F,
                    "S_0": S_0,
                    "n": n,
                    "d_F": d_F,
                    "d_0": d_0,
                    "a_F": a_F,
                    "a_0": a_0,
                }
                stage_save("01_core", core)

            # Bind stage 1 variables.
            y = core["y"]
            group = core["group"]
            gene_ids = core["gene_ids"]
            sample_names = core["sample_names"]
            exposure = core["exposure"]
            r_fit = core["r_fit"]
            r_hat = core["r_hat"]
            alpha_hat = core["alpha_hat"]
            mu_F = core["mu_F"]
            q_by_group = core["q_by_group"]
            mu_0 = core["mu_0"]
            S_F = core["S_F"]
            S_0 = core["S_0"]
            n = int(core["n"])
            d_F = int(core["d_F"])
            d_0 = int(core["d_0"])
            a_F = float(core["a_F"])
            a_0 = float(core["a_0"])
            G = len(gene_ids)

            analysis_fingerprint = compute_analysis_fingerprint(
                y, group, gene_ids, sample_names
            )

            if (
                expected_input_fingerprint is not None
                and expected_input_fingerprint != analysis_fingerprint
            ):
                raise RuntimeError(
                    "The supplied data do not match the cached M03 Stage 01 state. "
                    "Use a different checkpoint_dir or fresh_start=True."
                )

            manifest_path = checkpoint_path / "m03_manifest.pkl"
            if manifest_path.exists():
                manifest = load_pickle(manifest_path)
                prior_fp = str(manifest.get("analysis_fingerprint", ""))
                if prior_fp and prior_fp != analysis_fingerprint:
                    raise RuntimeError(
                        "Checkpoint fingerprint mismatch. Refusing to reuse M03 checkpoints "
                        "from a different analysis."
                    )
                prior_seed = manifest.get("random_seed")
                if prior_seed is not None and int(prior_seed) != int(random_seed):
                    raise RuntimeError(
                        "Checkpoint random_seed mismatch. Refusing to reuse M03 checkpoints "
                        "created under a different numerical audit seed."
                    )
            else:
                if core_loaded_from_checkpoint and bool(resume):
                    raise RuntimeError(
                        "This checkpoint directory contains pre-fingerprint M03 stages, "
                        "so their data/settings identity cannot be verified safely. "
                        "Use fresh_start=True or a new checkpoint_dir once with this "
                        "refinement-capable version."
                    )
                robust_pickle_dump(
                    {
                        "analysis_fingerprint": analysis_fingerprint,
                        "base_version": WEB_DNB_M03_BASE_VERSION,
                        "code_version": WEB_DNB_M03_VERSION,
                        "random_seed": int(random_seed),
                        "numerical_settings": numerical_settings,
                    },
                    manifest_path,
                )

            if _input_refine_from is not None:
                prior_state = _input_refine_from.get("refinement_state", {})
                prior_fp = str(prior_state.get("analysis_fingerprint", ""))
                if not prior_fp:
                    raise ValueError(
                        "refine_from is missing its analysis_fingerprint."
                    )
                if prior_fp != analysis_fingerprint:
                    raise RuntimeError(
                        "refine_from belongs to a different M03 analysis. "
                        "Refusing to mix exact results across analyses."
                    )

            # -----------------------------
            # Stage 2: exact-when-feasible C'(1), full DEB on ordinary-admissible genes
            # -----------------------------
            full_stage = stage_load("02_full_firstorder")
            if full_stage is None:
                if _input_y is None:
                    raise RuntimeError(
                        "Stage 02_full_firstorder is unavailable. "
                        "A refinement-only call cannot rebuild it without the original data."
                    )
                cprime_t0 = perf_counter()
                cprime = build_cprime_lookup(mu_F, mu_0, r_hat, 1)
                cprime_build_seconds = float(perf_counter() - cprime_t0)
                C_F = np.asarray(cprime["C_F"], dtype=float)
                C_0 = np.asarray(cprime["C_0"], dtype=float)
                R_F = S_F - C_F
                full_theta_fit = fit_deb_masked(
                    R_F, a_F, f"{method_name} full DEB theta"
                )
                theta_F_first = np.asarray(full_theta_fit["theta"], dtype=float)
                mask_Rpos = np.isfinite(R_F) & (
                    R_F > corrected_score_tolerance
                )
                mask_Rnonpos = np.isfinite(R_F) & ~mask_Rpos
                delta = S_0 - S_F
                nesting_tol = 1e-8 * max(
                    1.0, float(np.nanmedian(np.abs(S_F)))
                )
                if np.any(np.isfinite(delta) & (delta < -nesting_tol)):
                    raise RuntimeError(
                        f"{method_name}: raw nesting failed; "
                        f"min S0-SF={float(np.nanmin(delta)):.6g}."
                    )
                delta = np.maximum(delta, 0.0)
                full_stage = {
                    "C_F": C_F,
                    "C_0": C_0,
                    "R_F": R_F,
                    "full_theta_fit": full_theta_fit,
                    "theta_F_first": theta_F_first,
                    "mask_Rpos": mask_Rpos,
                    "mask_Rnonpos": mask_Rnonpos,
                    "delta": delta,
                    "cprime_diagnostics": {
                        "audit_max_abs_error": float(
                            cprime.get("audit_max_abs_error", np.nan)
                        ),
                        "interpolation_method": str(
                            cprime.get("interpolation_method", "")
                        ),
                        "emergency_refinements_used": int(
                            cprime.get("emergency_refinements_used", 0)
                        ),
                        "build_seconds": float(cprime_build_seconds),
                        "n_mu": int(cprime.get("n_mu", -1)),
                        "n_r": int(cprime.get("n_r", -1)),
                        "support_sum_count": int(
                            cprime.get("support_sum_count", 0)
                        ),
                        "gamma_continuum_count": int(
                            cprime.get("gamma_continuum_count", 0)
                        ),
                        "wide_exact_fallback_count": int(
                            cprime.get("wide_exact_fallback_count", 0)
                        ),
                        "max_discrete_support_width": int(
                            cprime.get("max_discrete_support_width", 0)
                        ),
                        "max_support_y": float(
                            cprime.get("max_support_y", np.nan)
                        ),
                        "grid_continuum_policy": str(
                            cprime.get("grid_continuum_policy", "")
                        ),
                        "n_mixed_exact_gamma_cells": int(
                            cprime.get("n_mixed_exact_gamma_cells", 0)
                        ),
                        "audit_mixed_cell_point_count": int(
                            cprime.get("audit_mixed_cell_point_count", 0)
                        ),
                        "audit_seam_branch_interp_count": int(
                            cprime.get("audit_seam_branch_interp_count", 0)
                        ),
                        "audit_seam_direct_fallback_count": int(
                            cprime.get("audit_seam_direct_fallback_count", 0)
                        ),
                        "production_mixed_cell_point_count": int(
                            cprime.get("production_mixed_cell_point_count", 0)
                        ),
                        "production_seam_branch_interp_count": int(
                            cprime.get("production_seam_branch_interp_count", 0)
                        ),
                        "production_seam_direct_fallback_count": int(
                            cprime.get("production_seam_direct_fallback_count", 0)
                        ),
                    },
                }
                stage_save("02_full_firstorder", full_stage)

            C_F = full_stage["C_F"]
            C_0 = full_stage["C_0"]
            R_F = full_stage["R_F"]
            full_theta_fit = full_stage["full_theta_fit"]
            theta_F_first = full_stage["theta_F_first"]
            mask_Rpos = full_stage["mask_Rpos"]
            mask_Rnonpos = full_stage["mask_Rnonpos"]
            delta = full_stage["delta"]

            # -----------------------------
            # Stage 3: canonical ASH/kappa_c + support-feasible null geometry
            # -----------------------------
            geom_stage = stage_load("03_geometry_support")
            if geom_stage is None:
                if _input_y is None:
                    raise RuntimeError(
                        "Stage 03_geometry_support is unavailable. "
                        "A refinement-only call cannot rebuild it without the original data."
                    )

                shared_valid = (
                    mask_Rpos
                    & np.isfinite(theta_F_first)
                    & (theta_F_first > 0)
                    & np.isfinite(delta)
                )
                if shared_valid.sum() < 20:
                    raise RuntimeError(
                        f"{method_name}: too few usable genes for geometry."
                    )

                effect = nb_effect_and_se(
                    mu_F, q_by_group, group, alpha_hat, theta_F_first
                )
                ash = fit_ash_normal_mixture(effect["beta"], effect["se"])
                weights = np.nan_to_num(
                    np.asarray(ash["posterior_null"], dtype=float),
                    nan=0.0,
                )
                geom_valid = (
                    shared_valid
                    & np.isfinite(weights)
                    & (weights >= 0)
                )
                if geom_valid.sum() < 20:
                    raise RuntimeError(
                        f"{method_name}: too few usable genes for observed kappa_c."
                    )
                w = weights[geom_valid]
                if float(np.sum(w)) <= 0:
                    raise RuntimeError(
                        f"{method_name}: observed ASH geometry weights sum to zero."
                    )
                Q_c_valid = (
                    2.0
                    * theta_F_first[geom_valid]
                    * delta[geom_valid]
                )
                kappa_F_used = 1.0
                kappa_c = float(
                    np.sum(w * Q_c_valid) / np.sum(w)
                )
                lambda_c = float(kappa_c)
                if not np.isfinite(kappa_c) or kappa_c <= 0:
                    raise RuntimeError(
                        f"{method_name}: invalid kappa_c={kappa_c}."
                    )

                current_S0dag = S_F + delta / lambda_c

                mu0_floor = np.maximum(np.floor(mu_0), 0.0)
                mu0_ceil = np.maximum(np.ceil(mu_0), 0.0)
                s_floor = half_nb_deviance(
                    mu0_floor, mu_0, r_hat[:, None]
                )
                s_ceil = half_nb_deviance(
                    mu0_ceil, mu_0, r_hat[:, None]
                )
                S0_min = np.minimum(s_floor, s_ceil).sum(axis=1)

                support_violation = (
                    np.isfinite(current_S0dag)
                    & np.isfinite(S0_min)
                    & (
                        current_S0dag
                        < S0_min - support_floor_tolerance
                    )
                )
                B = np.maximum(S_F, S0_min)
                shrinkable = np.maximum(S_0 - B, 0.0)
                feasible_S0dag = B + shrinkable / lambda_c
                S0dag_final = np.where(
                    support_violation,
                    feasible_S0dag,
                    current_S0dag,
                )
                if np.any(
                    S0dag_final
                    < S0_min - support_floor_tolerance
                ):
                    raise RuntimeError(
                        "Support-feasible correction failed to restore "
                        "the null support floor."
                    )

                geom_stage = {
                    "weights": weights,
                    "ash_pi0": float(ash["pi0"]),
                    "shared_valid": shared_valid,
                    "geom_valid": geom_valid,
                    "geometry_weight_sum": float(np.sum(w)),
                    "kappa_F_used": kappa_F_used,
                    "kappa_c": kappa_c,
                    "lambda_c": lambda_c,
                    "current_S0dag": current_S0dag,
                    "S0_min": S0_min,
                    "support_violation": support_violation,
                    "feasible_S0dag": feasible_S0dag,
                    "S0dag_final": S0dag_final,
                }
                stage_save("03_geometry_support", geom_stage)

            weights = geom_stage["weights"]
            shared_valid = geom_stage["shared_valid"]
            geom_valid = geom_stage["geom_valid"]
            kappa_F_used = float(geom_stage["kappa_F_used"])
            kappa_c = float(geom_stage["kappa_c"])
            lambda_c = float(geom_stage["lambda_c"])
            current_S0dag = geom_stage["current_S0dag"]
            S0_min = geom_stage["S0_min"]
            support_violation = geom_stage["support_violation"]
            feasible_S0dag = geom_stage["feasible_S0dag"]
            S0dag_final = geom_stage["S0dag_final"]

            # -----------------------------
            # Stage 4: first-order null DEB on frozen support-feasible geometry
            # -----------------------------
            null_stage = stage_load("04_null_deb")
            if null_stage is None:
                if _input_y is None:
                    raise RuntimeError(
                        "Stage 04_null_deb is unavailable. "
                        "A refinement-only call cannot rebuild it without the original data."
                    )
                R_0_dagger_first = S0dag_final - C_0
                null_theta_fit = fit_deb_masked(
                    R_0_dagger_first,
                    a_0,
                    f"{method_name} support-feasible null DEB theta",
                )
                theta_0_first = np.asarray(
                    null_theta_fit["theta"],
                    dtype=float,
                )
                test_first = normalized_statistic(
                    S_F,
                    S0dag_final,
                    C_F,
                    C_0,
                    theta_F_first,
                    theta_0_first,
                    n,
                )
                null_stage = {
                    "R_0_dagger_first": R_0_dagger_first,
                    "null_theta_fit": null_theta_fit,
                    "theta_0_first": theta_0_first,
                    "test_first": test_first,
                }
                stage_save("04_null_deb", null_stage)

            R_0_dagger_first = null_stage["R_0_dagger_first"]
            null_theta_fit = null_stage["null_theta_fit"]
            theta_0_first = null_stage["theta_0_first"]
            test_first = null_stage["test_first"]

            # ------------------------------------------------------------------
            # Reusable per-gene exact engine.  Exact results are independent of
            # selection reason and are cached once per gene + analysis fingerprint.
            # ------------------------------------------------------------------
            def exact_checkpoint_path(i):
                return exact_dir / f"gene_{int(i):07d}.pkl"

            def _payload_status_kind(payload):
                status = str(payload.get("status", "failure"))
                if status.startswith("success"):
                    return "success"
                if status.startswith("statistical_failure"):
                    return "statistical_failure"
                return "failure"

            def _payload_output_status(payload):
                kind = _payload_status_kind(payload)
                if kind == "success":
                    if bool(payload.get("negative_T", False)):
                        return "success_negative_T_truncated"
                    return "success"
                if kind == "statistical_failure":
                    stage = payload.get("failure_stage")
                    return (
                        f"statistical_failure:{stage}"
                        if stage
                        else "statistical_failure"
                    )
                error_type = payload.get("error_type")
                return (
                    f"failure:{error_type}"
                    if error_type
                    else "failure"
                )

            def exact_gene_job(i):
                i = int(i)
                gid = str(gene_ids[i])
                try:
                    full_states = exact_prepare_model_states(
                        mu_F[i], r_hat[i]
                    )
                    smin_F = exact_model_smin_sum(
                        mu_F[i], r_hat[i]
                    )
                    full_fit = solve_exact_precision_profile(
                        S_F[i],
                        a_F,
                        full_states,
                        smin_F,
                    )

                    payload = {
                        "status": "success",
                        "analysis_fingerprint": analysis_fingerprint,
                        "gene_index": i,
                        "gene_id": gid,
                        "full_fit": full_fit,
                        "null_fit": None,
                        "T_raw": np.nan,
                        "T": np.nan,
                        "p_value": np.nan,
                    }

                    if full_fit.get("root_status") != "finite_root":
                        payload["status"] = "statistical_failure"
                        payload["failure_stage"] = "full_exact_root"
                        return payload

                    null_states = exact_prepare_model_states(
                        mu_0[i], r_hat[i]
                    )
                    null_fit = solve_exact_precision_profile(
                        S0dag_final[i],
                        a_0,
                        null_states,
                        S0_min[i],
                    )
                    payload["null_fit"] = null_fit

                    if null_fit.get("root_status") != "finite_root":
                        payload["status"] = "statistical_failure"
                        payload["failure_stage"] = "null_exact_root"
                        return payload

                    thF = float(full_fit["theta_hat_exact"])
                    th0 = float(null_fit["theta_hat_exact"])
                    logCF = float(full_fit["logC_at_hat"])
                    logC0 = float(null_fit["logC_at_hat"])

                    T_raw_exact = (
                        2.0
                        * (
                            th0 * S0dag_final[i]
                            - thF * S_F[i]
                        )
                        + float(n) * np.log(thF / th0)
                        + 2.0 * (logCF - logC0)
                    )
                    T_exact = max(float(T_raw_exact), 0.0)
                    pval = float(chi2.sf(T_exact, 1))

                    payload.update({
                        "T_raw": float(T_raw_exact),
                        "T": float(T_exact),
                        "p_value": pval,
                        "theta_F_exact": thF,
                        "theta_0_exact": th0,
                        "negative_T": bool(T_raw_exact < 0),
                    })
                    return payload

                except Exception as exc:
                    return {
                        "status": "failure",
                        "analysis_fingerprint": analysis_fingerprint,
                        "gene_index": i,
                        "gene_id": gid,
                        "error_type": type(exc).__name__,
                        "error_message": str(exc),
                        "traceback": traceback.format_exc(),
                    }

            # Active exact results are only the mandatory M03 fallbacks, exact results
            # carried forward through refine_from, and genes selected in THIS refinement
            # path. Merely having another cached exact result on disk never changes output.
            exact_payloads_active = {}
            exact_reason_map = {}
            exact_round_map = {}
            selected_this_call = set()
            computed_this_call = set()

            REASON_ORDER = {
                "mandatory_R_F_nonpositive": 0,
                "called_initial": 1,
                "called_buffer": 2,
                "called_closure": 3,
                "top_n": 4,
                "top_fraction": 5,
                "user_gene_list": 6,
                "all": 7,
                "prior_refinement": 8,
            }

            def _add_reason(i, reason):
                i = int(i)
                if i not in exact_reason_map:
                    exact_reason_map[i] = set()
                exact_reason_map[i].add(str(reason))

            def _set_round(i, round_id):
                i = int(i)
                round_id = int(round_id)
                if i not in exact_round_map:
                    exact_round_map[i] = round_id
                else:
                    exact_round_map[i] = min(
                        int(exact_round_map[i]),
                        round_id,
                    )

            def _surrogate_payload_from_values(
                i,
                *,
                status,
                p_value=np.nan,
                T_raw=np.nan,
                T=np.nan,
                theta_F_exact=np.nan,
                theta_0_exact=np.nan,
                negative_T=False,
                failure_stage=None,
                error_type=None,
            ):
                status_kind = str(status)
                if status_kind.startswith("success"):
                    status_kind = "success"
                elif status_kind.startswith("statistical_failure"):
                    status_kind = "statistical_failure"
                else:
                    status_kind = "failure"

                payload = {
                    "status": status_kind,
                    "analysis_fingerprint": analysis_fingerprint,
                    "gene_index": int(i),
                    "gene_id": str(gene_ids[int(i)]),
                    "p_value": float(p_value) if np.isfinite(p_value) else np.nan,
                    "T_raw": float(T_raw) if np.isfinite(T_raw) else np.nan,
                    "T": float(T) if np.isfinite(T) else np.nan,
                    "theta_F_exact": (
                        float(theta_F_exact)
                        if np.isfinite(theta_F_exact)
                        else np.nan
                    ),
                    "theta_0_exact": (
                        float(theta_0_exact)
                        if np.isfinite(theta_0_exact)
                        else np.nan
                    ),
                    "negative_T": bool(negative_T),
                }
                if failure_stage:
                    payload["failure_stage"] = str(failure_stage)
                if error_type:
                    payload["error_type"] = str(error_type)
                return payload

            def _load_valid_cached_payload(i):
                path = exact_checkpoint_path(i)
                if not bool(resume) or not path.exists():
                    return None

                old = load_pickle(path)
                if (
                    str(old.get("analysis_fingerprint", ""))
                    != analysis_fingerprint
                ):
                    log(
                        f"[exact cache] ignoring stale fingerprint for "
                        f"gene index {int(i)}"
                    )
                    return None

                kind = _payload_status_kind(old)
                if kind in {"success", "statistical_failure"}:
                    return old
                if (
                    kind == "failure"
                    and not bool(retry_failed_on_resume)
                ):
                    return old
                return None

            def ensure_exact_indices(indices, reason, round_id):
                indices = sorted({
                    int(i)
                    for i in np.asarray(list(indices), dtype=int)
                    if 0 <= int(i) < G
                })
                if not indices:
                    return

                pending = []
                before = set(exact_payloads_active)

                for i in indices:
                    _add_reason(i, reason)
                    _set_round(i, round_id)
                    if i not in before:
                        selected_this_call.add(i)

                    existing = exact_payloads_active.get(i)
                    if existing is not None:
                        kind = _payload_status_kind(existing)
                        if (
                            kind == "failure"
                            and bool(retry_failed_on_resume)
                        ):
                            exact_payloads_active.pop(i, None)
                        else:
                            continue

                    cached = _load_valid_cached_payload(i)
                    if cached is not None:
                        exact_payloads_active[i] = cached
                        continue

                    pending.append(i)

                log(
                    f"[exact refinement] reason={reason} | requested={len(indices):,} | "
                    f"new computations={len(pending):,}"
                )

                if not pending:
                    return

                with threadpool_limits(limits=1):
                    with ThreadPoolExecutor(
                        max_workers=max(1, int(n_jobs_exact))
                    ) as ex:
                        future_map = {
                            ex.submit(exact_gene_job, i): i
                            for i in pending
                        }
                        done = 0
                        for fut in as_completed(future_map):
                            i = future_map[fut]
                            payload = fut.result()
                            robust_pickle_dump(
                                payload,
                                exact_checkpoint_path(i),
                            )
                            exact_payloads_active[i] = payload
                            computed_this_call.add(i)
                            done += 1
                            if (
                                int(verbose) > 0
                                and (
                                    done % 100 == 0
                                    or done == len(pending)
                                )
                            ):
                                log(
                                    f"[exact refinement] completed "
                                    f"{done:,}/{len(pending):,} "
                                    f"for reason={reason}"
                                )

            # ------------------------------------------------------------------
            # Canonical M03 base result: mandatory exact fallback ONLY.
            # This is option 1 and is frozen before any optional refinement.
            # ------------------------------------------------------------------
            base_cached = stage_load("05_base_m03_result")

            mandatory_indices = np.flatnonzero(mask_Rnonpos)

            if base_cached is not None:
                # Seed the mandatory exact payloads from the saved canonical base so
                # successful old exact work does not need to be recomputed merely because
                # the per-gene exact cache is absent.
                base_status = np.asarray(
                    base_cached["exact_status_m03"],
                    dtype=object,
                )
                base_neg = np.asarray(
                    base_cached["exact_negative_T_m03"],
                    dtype=bool,
                )
                base_p_saved = np.asarray(
                    base_cached["p_value_m03"],
                    dtype=float,
                )
                base_Traw_saved = np.asarray(
                    base_cached["T_raw_m03"],
                    dtype=float,
                )
                base_T_saved = np.asarray(
                    base_cached["T_m03"],
                    dtype=float,
                )
                base_thF_saved = np.asarray(
                    base_cached["theta_F_m03"],
                    dtype=float,
                )
                base_th0_saved = np.asarray(
                    base_cached["theta_0_m03"],
                    dtype=float,
                )

                for i in mandatory_indices:
                    status_i = str(base_status[i])
                    failure_stage = None
                    if status_i.startswith("statistical_failure:"):
                        failure_stage = status_i.split(":", 1)[1]
                    error_type = None
                    if status_i.startswith("failure:"):
                        error_type = status_i.split(":", 1)[1]
                    exact_payloads_active[int(i)] = (
                        _surrogate_payload_from_values(
                            i,
                            status=status_i,
                            p_value=base_p_saved[i],
                            T_raw=base_Traw_saved[i],
                            T=base_T_saved[i],
                            theta_F_exact=base_thF_saved[i],
                            theta_0_exact=base_th0_saved[i],
                            negative_T=base_neg[i],
                            failure_stage=failure_stage,
                            error_type=error_type,
                        )
                    )

            for i in mandatory_indices:
                _add_reason(i, "mandatory_R_F_nonpositive")
                _set_round(i, 0)

            # This also retries prior numerical failures if requested.
            ensure_exact_indices(
                mandatory_indices,
                "mandatory_R_F_nonpositive",
                0,
            )

            p_m03 = np.asarray(test_first["p"], dtype=float).copy()
            T_raw_m03 = np.asarray(
                test_first["T_raw"],
                dtype=float,
            ).copy()
            T_m03 = np.asarray(
                test_first["T"],
                dtype=float,
            ).copy()
            theta_F_m03 = np.asarray(
                theta_F_first,
                dtype=float,
            ).copy()
            theta_0_m03 = np.asarray(
                theta_0_first,
                dtype=float,
            ).copy()
            branch_m03 = np.full(
                G,
                "first_order_DEB",
                dtype=object,
            )
            exact_status_m03 = np.full(
                G,
                "not_needed",
                dtype=object,
            )
            exact_negative_T_m03 = np.zeros(G, dtype=bool)

            p_m03[mask_Rnonpos] = np.nan
            T_raw_m03[mask_Rnonpos] = np.nan
            T_m03[mask_Rnonpos] = np.nan
            theta_F_m03[mask_Rnonpos] = np.nan
            theta_0_m03[mask_Rnonpos] = np.nan
            branch_m03[mask_Rnonpos] = (
                "exact_profile_R_F_nonpositive"
            )

            for i in mandatory_indices:
                payload = exact_payloads_active.get(int(i))
                if payload is None:
                    exact_status_m03[i] = "missing_exact_result"
                    continue

                exact_status_m03[i] = _payload_output_status(payload)
                kind = _payload_status_kind(payload)

                if (
                    kind == "success"
                    and np.isfinite(payload.get("p_value", np.nan))
                ):
                    p_m03[i] = float(payload["p_value"])
                    T_raw_m03[i] = float(payload["T_raw"])
                    T_m03[i] = float(payload["T"])
                    theta_F_m03[i] = float(
                        payload["theta_F_exact"]
                    )
                    theta_0_m03[i] = float(
                        payload["theta_0_exact"]
                    )
                    exact_negative_T_m03[i] = bool(
                        payload.get("negative_T", False)
                    )

            q_m03 = bh_adjust(p_m03)

            base_stage = {
                "analysis_fingerprint": analysis_fingerprint,
                "p_value_m03": p_m03,
                "q_value_m03": q_m03,
                "T_raw_m03": T_raw_m03,
                "T_m03": T_m03,
                "theta_F_m03": theta_F_m03,
                "theta_0_m03": theta_0_m03,
                "branch_m03": branch_m03,
                "exact_status_m03": exact_status_m03,
                "exact_negative_T_m03": exact_negative_T_m03,
            }
            stage_save("05_base_m03_result", base_stage)

            # ------------------------------------------------------------------
            # Carry forward a previous refinement result without recomputing genes.
            # ------------------------------------------------------------------
            if _input_refine_from is not None:
                prior_gr = _input_refine_from.get("gene_results")
                if not isinstance(prior_gr, pd.DataFrame):
                    raise ValueError(
                        "refine_from['gene_results'] must be a pandas DataFrame."
                    )
                required_prior_cols = {
                    "gene_id",
                    "exact_attempted",
                    "exact_status",
                    "p_value",
                    "T_raw",
                    "T",
                    "theta_F",
                    "theta_0",
                }
                missing_prior_cols = sorted(
                    required_prior_cols.difference(prior_gr.columns)
                )
                if missing_prior_cols:
                    raise ValueError(
                        "refine_from gene_results is missing required columns: "
                        + ", ".join(missing_prior_cols)
                    )

                prior_lookup = {
                    str(g): j
                    for j, g in enumerate(
                        prior_gr["gene_id"].astype(str).tolist()
                    )
                }
                current_lookup = {
                    str(g): i
                    for i, g in enumerate(gene_ids.astype(str))
                }
                if set(prior_lookup) != set(current_lookup):
                    raise RuntimeError(
                        "refine_from gene IDs do not match the current cached analysis."
                    )

                for gid, current_i in current_lookup.items():
                    row = prior_gr.iloc[prior_lookup[gid]]
                    if not bool(row["exact_attempted"]):
                        continue

                    status_i = str(row["exact_status"])
                    p_exact_i = (
                        row["p_value_exact"]
                        if "p_value_exact" in prior_gr.columns
                        else row["p_value"]
                    )
                    Traw_exact_i = (
                        row["exact_T_raw"]
                        if "exact_T_raw" in prior_gr.columns
                        else row["T_raw"]
                    )
                    T_exact_i = (
                        row["exact_T"]
                        if "exact_T" in prior_gr.columns
                        else row["T"]
                    )
                    thF_exact_i = (
                        row["theta_F_exact"]
                        if "theta_F_exact" in prior_gr.columns
                        else row["theta_F"]
                    )
                    th0_exact_i = (
                        row["theta_0_exact"]
                        if "theta_0_exact" in prior_gr.columns
                        else row["theta_0"]
                    )
                    neg_i = (
                        bool(row["exact_negative_T"])
                        if "exact_negative_T" in prior_gr.columns
                        else False
                    )

                    failure_stage = None
                    if status_i.startswith("statistical_failure:"):
                        failure_stage = status_i.split(":", 1)[1]
                    error_type = None
                    if status_i.startswith("failure:"):
                        error_type = status_i.split(":", 1)[1]

                    exact_payloads_active[current_i] = (
                        _surrogate_payload_from_values(
                            current_i,
                            status=status_i,
                            p_value=p_exact_i,
                            T_raw=Traw_exact_i,
                            T=T_exact_i,
                            theta_F_exact=thF_exact_i,
                            theta_0_exact=th0_exact_i,
                            negative_T=neg_i,
                            failure_stage=failure_stage,
                            error_type=error_type,
                        )
                    )

                    if "exact_reason" in prior_gr.columns:
                        reason_text = str(row["exact_reason"])
                        if (
                            reason_text
                            and reason_text.lower() != "nan"
                        ):
                            for reason in reason_text.split(";"):
                                reason = reason.strip()
                                if reason:
                                    _add_reason(current_i, reason)
                    if current_i not in exact_reason_map:
                        _add_reason(current_i, "prior_refinement")

                    if "exact_round" in prior_gr.columns:
                        prior_round = row["exact_round"]
                        if pd.notna(prior_round):
                            _set_round(
                                current_i,
                                int(prior_round),
                            )

            # ------------------------------------------------------------------
            # Assemble the current p-vector from canonical M03 + active exact results.
            # Failed exact attempts remain explicit NA; they are never replaced by p=1
            # or silently reverted to the approximate first-order p-value.
            # ------------------------------------------------------------------
            def assemble_current_state():
                p = p_m03.copy()
                T_raw = T_raw_m03.copy()
                T = T_m03.copy()
                theta_F_final = theta_F_m03.copy()
                theta_0_final = theta_0_m03.copy()
                branch = branch_m03.copy()

                exact_attempted = np.zeros(G, dtype=bool)
                exact_refined = np.zeros(G, dtype=bool)
                exact_status = np.full(
                    G,
                    "not_attempted",
                    dtype=object,
                )
                exact_negative_T = np.zeros(G, dtype=bool)
                p_value_exact = np.full(G, np.nan)
                exact_T_raw = np.full(G, np.nan)
                exact_T = np.full(G, np.nan)
                theta_F_exact = np.full(G, np.nan)
                theta_0_exact = np.full(G, np.nan)

                for i, payload in exact_payloads_active.items():
                    i = int(i)
                    exact_attempted[i] = True
                    exact_status[i] = _payload_output_status(payload)
                    kind = _payload_status_kind(payload)

                    if (
                        kind == "success"
                        and np.isfinite(
                            payload.get("p_value", np.nan)
                        )
                    ):
                        exact_refined[i] = True
                        p_exact = float(payload["p_value"])
                        tr_exact = float(payload["T_raw"])
                        t_exact = float(payload["T"])
                        tf_exact = float(
                            payload["theta_F_exact"]
                        )
                        t0_exact = float(
                            payload["theta_0_exact"]
                        )

                        p[i] = p_exact
                        T_raw[i] = tr_exact
                        T[i] = t_exact
                        theta_F_final[i] = tf_exact
                        theta_0_final[i] = t0_exact

                        p_value_exact[i] = p_exact
                        exact_T_raw[i] = tr_exact
                        exact_T[i] = t_exact
                        theta_F_exact[i] = tf_exact
                        theta_0_exact[i] = t0_exact
                        exact_negative_T[i] = bool(
                            payload.get("negative_T", False)
                        )

                        if mask_Rnonpos[i]:
                            branch[i] = (
                                "exact_profile_R_F_nonpositive"
                            )
                        else:
                            branch[i] = "exact_profile_refined"
                    else:
                        # Exact was explicitly requested/required but was not obtainable.
                        p[i] = np.nan
                        T_raw[i] = np.nan
                        T[i] = np.nan
                        theta_F_final[i] = np.nan
                        theta_0_final[i] = np.nan
                        branch[i] = "exact_profile_failure"

                q = bh_adjust(p)

                return {
                    "p": p,
                    "q": q,
                    "T_raw": T_raw,
                    "T": T,
                    "theta_F": theta_F_final,
                    "theta_0": theta_0_final,
                    "branch": branch,
                    "exact_attempted": exact_attempted,
                    "exact_refined": exact_refined,
                    "exact_status": exact_status,
                    "exact_negative_T": exact_negative_T,
                    "p_value_exact": p_value_exact,
                    "exact_T_raw": exact_T_raw,
                    "exact_T": exact_T,
                    "theta_F_exact": theta_F_exact,
                    "theta_0_exact": theta_0_exact,
                }

            def call_mask_from_state(state):
                if p_cutoff is not None:
                    return (
                        np.isfinite(state["p"])
                        & (state["p"] <= p_cutoff)
                    )
                if fdr_cutoff is not None:
                    return (
                        np.isfinite(state["q"])
                        & (state["q"] <= fdr_cutoff)
                    )
                return np.zeros(G, dtype=bool)

            def next_exact_round():
                if not exact_round_map:
                    return 1
                return max(
                    int(v)
                    for v in exact_round_map.values()
                ) + 1

            def run_called_closure(round_id):
                round_id = int(round_id)
                while True:
                    state = assemble_current_state()
                    called_now = call_mask_from_state(state)
                    new_called = [
                        int(i)
                        for i in np.flatnonzero(called_now)
                        if int(i) not in exact_payloads_active
                    ]
                    if not new_called:
                        return round_id, state

                    ensure_exact_indices(
                        new_called,
                        "called_closure",
                        round_id,
                    )
                    round_id += 1

            refinement_diagnostics = {
                "exact_mode": exact_mode_used,
                "p_cutoff": p_cutoff,
                "fdr_cutoff": fdr_cutoff,
                "exact_top_n": exact_top_n,
                "exact_top_fraction": exact_top_fraction,
                "n_prior_exact_attempted": int(
                    np.sum(
                        _input_refine_from["gene_results"]["exact_attempted"].astype(bool)
                    )
                ) if _input_refine_from is not None else 0,
                "n_called_initial": 0,
                "n_called_final": 0,
                "buffer_mean_abs_p_change": np.nan,
                "buffer_sd_abs_p_change": np.nan,
                "buffer_call_boundary": np.nan,
                "buffer_raw_boundary": np.nan,
                "buffer_final_boundary": np.nan,
                "buffer_p_boundary_cap_hit": False,
                "buffer_unit_interval_cap_hit": False,
                "buffer_count_cap_hit": False,
                "buffer_n_eligible": 0,
                "buffer_n_selected": 0,
                "buffer_n_change_estimates": 0,
            }

            state_before_optional = assemble_current_state()
            initial_called_mask = call_mask_from_state(
                state_before_optional
            )
            refinement_diagnostics["n_called_initial"] = int(
                np.sum(initial_called_mask)
            )

            round_id = next_exact_round()

            if exact_mode_used in {"called", "called_buffer"}:
                if p_cutoff is None and fdr_cutoff is None:
                    warn_user(
                        f"exact_mode='{exact_mode_used}' was requested without "
                        "p_cutoff or fdr_cutoff; no optional exact refinement "
                        "was performed."
                    )
                else:
                    initial_called_indices = np.flatnonzero(
                        initial_called_mask
                    )
                    ensure_exact_indices(
                        initial_called_indices,
                        "called_initial",
                        round_id,
                    )
                    round_id += 1

                    if exact_mode_used == "called_buffer":
                        n_called = int(len(initial_called_indices))

                        if n_called == 0:
                            warn_user(
                                "called_buffer found zero called genes, so the "
                                "adaptive p-value-change buffer could not be estimated. "
                                "No buffer genes were added."
                            )
                        else:
                            if n_called <= 3:
                                warn_user(
                                    "called_buffer is estimating its p-value-change "
                                    f"distribution from only {n_called} called gene(s). "
                                    "The predefined 2x p-boundary and 2x call-count "
                                    "caps remain active."
                                )

                            state_after_initial = assemble_current_state()

                            change_indices = []
                            changes = []
                            for i in initial_called_indices:
                                i = int(i)
                                if not mask_Rpos[i]:
                                    # Mandatory exact genes are already part of canonical M03
                                    # and do not reveal the effect of optional exact refinement.
                                    continue
                                payload = exact_payloads_active.get(i)
                                if payload is None:
                                    continue
                                if _payload_status_kind(payload) != "success":
                                    continue
                                if (
                                    not np.isfinite(p_m03[i])
                                    or not np.isfinite(
                                        state_after_initial["p"][i]
                                    )
                                ):
                                    continue
                                change_indices.append(i)
                                changes.append(
                                    abs(
                                        float(state_after_initial["p"][i])
                                        - float(p_m03[i])
                                    )
                                )

                            changes = np.asarray(changes, dtype=float)
                            refinement_diagnostics[
                                "buffer_n_change_estimates"
                            ] = int(len(changes))

                            if len(changes) == 0:
                                warn_user(
                                    "called_buffer had called genes but no usable "
                                    "ordinary first-order called genes with successful "
                                    "exact p-values, so the adaptive buffer could not "
                                    "be estimated."
                                )
                            else:
                                mean_abs_change = float(
                                    np.mean(changes)
                                )
                                if len(changes) == 1:
                                    sd_abs_change = 0.0
                                elif len(changes) <= 3:
                                    sd_abs_change = float(
                                        np.std(changes, ddof=0)
                                    )
                                else:
                                    sd_abs_change = float(
                                        np.std(changes, ddof=1)
                                    )

                                if p_cutoff is not None:
                                    call_boundary = float(p_cutoff)
                                else:
                                    called_start_p = (
                                        state_before_optional["p"][
                                            initial_called_mask
                                        ]
                                    )
                                    called_start_p = called_start_p[
                                        np.isfinite(called_start_p)
                                    ]
                                    if len(called_start_p) == 0:
                                        call_boundary = np.nan
                                    else:
                                        call_boundary = float(
                                            np.max(called_start_p)
                                        )

                                if not np.isfinite(call_boundary):
                                    warn_user(
                                        "called_buffer could not determine a finite "
                                        "raw-p rejection boundary, so no buffer genes "
                                        "were added."
                                    )
                                else:
                                    raw_boundary = float(
                                        call_boundary
                                        + mean_abs_change
                                        + sd_abs_change
                                    )
                                    p_cap = float(
                                        min(2.0 * call_boundary, 1.0)
                                    )
                                    final_boundary = float(
                                        min(raw_boundary, p_cap, 1.0)
                                    )

                                    p_cap_hit = bool(
                                        raw_boundary > 2.0 * call_boundary
                                    )
                                    unit_cap_hit = bool(
                                        raw_boundary > 1.0
                                    )

                                    refinement_diagnostics.update({
                                        "buffer_mean_abs_p_change": mean_abs_change,
                                        "buffer_sd_abs_p_change": sd_abs_change,
                                        "buffer_call_boundary": call_boundary,
                                        "buffer_raw_boundary": raw_boundary,
                                        "buffer_final_boundary": final_boundary,
                                        "buffer_p_boundary_cap_hit": p_cap_hit,
                                        "buffer_unit_interval_cap_hit": unit_cap_hit,
                                    })

                                    if p_cap_hit:
                                        warn_user(
                                            "called_buffer hit the predefined 2x "
                                            "raw-p boundary cap: "
                                            f"uncapped={raw_boundary:.6g}, "
                                            f"2x boundary={2.0 * call_boundary:.6g}, "
                                            f"used={final_boundary:.6g}."
                                        )
                                    if unit_cap_hit:
                                        warn_user(
                                            "called_buffer's uncapped boundary exceeded 1.0; "
                                            f"uncapped={raw_boundary:.6g}, "
                                            f"used={final_boundary:.6g}."
                                        )

                                    active_now = set(
                                        exact_payloads_active.keys()
                                    )
                                    eligible = [
                                        int(i)
                                        for i in range(G)
                                        if (
                                            not initial_called_mask[i]
                                            and i not in active_now
                                            and np.isfinite(
                                                state_before_optional["p"][i]
                                            )
                                            and (
                                                state_before_optional["p"][i]
                                                > call_boundary
                                            )
                                            and (
                                                state_before_optional["p"][i]
                                                <= final_boundary
                                            )
                                        )
                                    ]
                                    eligible.sort(
                                        key=lambda i: (
                                            float(
                                                state_before_optional["p"][i]
                                            ),
                                            int(i),
                                        )
                                    )

                                    count_cap = int(
                                        2 * n_called
                                    )
                                    count_cap_hit = bool(
                                        len(eligible) > count_cap
                                    )
                                    selected_buffer = eligible[
                                        :count_cap
                                    ]

                                    refinement_diagnostics.update({
                                        "buffer_count_cap_hit": count_cap_hit,
                                        "buffer_n_eligible": int(
                                            len(eligible)
                                        ),
                                        "buffer_n_selected": int(
                                            len(selected_buffer)
                                        ),
                                    })

                                    if count_cap_hit:
                                        warn_user(
                                            "called_buffer hit the predefined 2x "
                                            "call-count cap: "
                                            f"eligible={len(eligible):,}, "
                                            f"called={n_called:,}, "
                                            f"selected={len(selected_buffer):,}."
                                        )

                                    ensure_exact_indices(
                                        selected_buffer,
                                        "called_buffer",
                                        round_id,
                                    )
                                    round_id += 1

                    # Every final called gene must have been sent through exact profiling.
                    round_id, _ = run_called_closure(
                        round_id
                    )

            elif exact_mode_used == "top_n":
                nonmandatory_rankable = [
                    int(i)
                    for i in range(G)
                    if (
                        not mask_Rnonpos[i]
                        and np.isfinite(p_m03[i])
                    )
                ]
                nonmandatory_rankable.sort(
                    key=lambda i: (
                        float(p_m03[i]),
                        int(i),
                    )
                )

                if exact_top_n > len(nonmandatory_rankable):
                    warn_user(
                        "exact_top_n exceeds the number of non-mandatory genes "
                        "with finite canonical M03 p-values. All rankable "
                        "non-mandatory genes will be exacted."
                    )

                selected = nonmandatory_rankable[
                    :min(exact_top_n, len(nonmandatory_rankable))
                ]
                ensure_exact_indices(
                    selected,
                    "top_n",
                    round_id,
                )

            elif exact_mode_used == "top_fraction":
                requested_n = int(
                    np.ceil(float(exact_top_fraction) * G)
                )
                nonmandatory_rankable = [
                    int(i)
                    for i in range(G)
                    if (
                        not mask_Rnonpos[i]
                        and np.isfinite(p_m03[i])
                    )
                ]
                nonmandatory_rankable.sort(
                    key=lambda i: (
                        float(p_m03[i]),
                        int(i),
                    )
                )

                if requested_n > len(nonmandatory_rankable):
                    warn_user(
                        "The requested top fraction corresponds to more genes than "
                        "the number of non-mandatory genes with finite canonical "
                        "M03 p-values. All rankable non-mandatory genes will be exacted."
                    )

                selected = nonmandatory_rankable[
                    :min(requested_n, len(nonmandatory_rankable))
                ]
                ensure_exact_indices(
                    selected,
                    "top_fraction",
                    round_id,
                )

            elif exact_mode_used == "all":
                ensure_exact_indices(
                    range(G),
                    "all",
                    round_id,
                )

            elif exact_mode_used == "gene_list":
                gene_to_indices = {}
                for i, g in enumerate(gene_ids.astype(str)):
                    gene_to_indices.setdefault(str(g), []).append(int(i))

                unknown = sorted({
                    gid
                    for gid in exact_gene_ids
                    if gid not in gene_to_indices
                })
                if unknown:
                    preview = ", ".join(unknown[:20])
                    extra = (
                        f" (+{len(unknown)-20} more)"
                        if len(unknown) > 20
                        else ""
                    )
                    raise ValueError(
                        "The following exact_gene_ids were not found in the analyzed "
                        f"gene IDs: {preview}{extra}"
                    )

                ambiguous = sorted({
                    gid
                    for gid in exact_gene_ids
                    if (
                        gid in gene_to_indices
                        and len(gene_to_indices[gid]) != 1
                    )
                })
                if ambiguous:
                    preview = ", ".join(ambiguous[:20])
                    extra = (
                        f" (+{len(ambiguous)-20} more)"
                        if len(ambiguous) > 20
                        else ""
                    )
                    raise ValueError(
                        "The following requested gene IDs are not unique in the "
                        "analyzed data and therefore cannot be selected unambiguously: "
                        f"{preview}{extra}"
                    )

                selected = [
                    gene_to_indices[gid][0]
                    for gid in dict.fromkeys(exact_gene_ids)
                ]
                ensure_exact_indices(
                    selected,
                    "user_gene_list",
                    round_id,
                )

            # default: canonical M03 plus any prior refine_from exact state, with no new
            # optional genes selected.

            state_final = assemble_current_state()
            p = state_final["p"]
            q = state_final["q"]
            T_raw = state_final["T_raw"]
            T = state_final["T"]
            theta_F_final = state_final["theta_F"]
            theta_0_final = state_final["theta_0"]
            branch = state_final["branch"]
            exact_attempted = state_final["exact_attempted"]
            exact_refined = state_final["exact_refined"]
            exact_status = state_final["exact_status"]
            exact_negative_T = state_final["exact_negative_T"]
            p_value_exact = state_final["p_value_exact"]
            exact_T_raw = state_final["exact_T_raw"]
            exact_T = state_final["exact_T"]
            theta_F_exact = state_final["theta_F_exact"]
            theta_0_exact = state_final["theta_0_exact"]

            final_called_mask = call_mask_from_state(
                state_final
            )
            refinement_diagnostics["n_called_final"] = int(
                np.sum(final_called_mask)
            )

            # Warn when newly activated exact profiling fails. A failed exact
            # request remains explicit NA; it is not replaced by p=1 and is not
            # silently reverted to the approximate M03 p-value.
            failed_new = [
                int(i)
                for i in selected_this_call
                if (
                    i in exact_payloads_active
                    and _payload_status_kind(
                        exact_payloads_active[i]
                    ) != "success"
                )
            ]
            if failed_new:
                failed_preview = ", ".join(
                    str(gene_ids[i])
                    for i in failed_new[:10]
                )
                failed_extra = (
                    f" (+{len(failed_new) - 10} more)"
                    if len(failed_new) > 10
                    else ""
                )
                warn_user(
                    "Exact profiling failed for "
                    f"{len(failed_new):,} newly activated gene(s). "
                    "Their final exact-refined p-values are NA rather than p=1 "
                    "or silently reverted approximate values. "
                    f"Genes: {failed_preview}{failed_extra}. "
                    "See exact_status in gene_results."
                )

            # Warn when a newly activated exact result produces a negative raw LRT.
            negative_new = [
                int(i)
                for i in selected_this_call
                if (
                    i in exact_payloads_active
                    and _payload_status_kind(
                        exact_payloads_active[i]
                    ) == "success"
                    and bool(
                        exact_payloads_active[i].get(
                            "negative_T",
                            False,
                        )
                    )
                )
            ]
            if negative_new:
                warn_user(
                    "Exact profiling produced negative raw LRT statistics for "
                    f"{len(negative_new):,} newly activated gene(s). "
                    "Those statistics were truncated to T=0 and p=1. "
                    "See exact_negative_T and exact_T_raw in gene_results."
                )

            # Canonical and final exact diagnostics.
            n_exact_attempted_total = int(
                np.sum(exact_attempted)
            )
            n_exact_success_total = int(
                np.sum(exact_refined)
            )
            n_exact_failure_total = int(
                np.sum(
                    exact_attempted
                    & ~exact_refined
                )
            )
            n_exact_negative_total = int(
                np.sum(exact_negative_T)
            )

            n_finite_p = int(np.isfinite(p).sum())
            n_na_p = int(G - n_finite_p)
            all_support_feasible = bool(
                np.all(
                    S0dag_final
                    >= S0_min - support_floor_tolerance
                )
            )

            Q_F = np.full(G, np.nan, dtype=float)
            Q_F[shared_valid] = (
                2.0
                * theta_F_first[shared_valid]
                * R_F[shared_valid]
            )

            Q_c = np.full(G, np.nan, dtype=float)
            valid_final_theta = (
                np.isfinite(theta_F_final)
                & (theta_F_final > 0)
            )
            Q_c[valid_final_theta] = (
                2.0
                * theta_F_final[valid_final_theta]
                * delta[valid_final_theta]
            )

            fc_levels = list(np.unique(group))
            if len(fc_levels) != 2:
                raise RuntimeError(
                    "M03 log2FC output requires exactly two groups."
                )
            fc_A = np.maximum(
                q_by_group[str(fc_levels[0])],
                1e-300,
            )
            fc_B = np.maximum(
                q_by_group[str(fc_levels[1])],
                1e-300,
            )
            m03_log2FC = (
                np.log(fc_B) - np.log(fc_A)
            ) / np.log(2.0)

            def reason_string(i):
                reasons = exact_reason_map.get(int(i), set())
                if not reasons:
                    return ""
                return ";".join(
                    sorted(
                        reasons,
                        key=lambda x: (
                            REASON_ORDER.get(x, 999),
                            str(x),
                        ),
                    )
                )

            exact_reason = np.asarray(
                [reason_string(i) for i in range(G)],
                dtype=object,
            )
            exact_round = np.full(G, np.nan)
            for i, value in exact_round_map.items():
                exact_round[int(i)] = int(value)

            p_value_change = np.full(G, np.nan)
            both_finite = np.isfinite(p) & np.isfinite(p_m03)
            p_value_change[both_finite] = (
                p[both_finite] - p_m03[both_finite]
            )
            abs_p_value_change = np.abs(p_value_change)

            gene_results = pd.DataFrame({
                "gene_id": gene_ids,
                "log2FC": m03_log2FC,

                # Frozen canonical M03 values.
                "p_value_m03": p_m03,
                "q_value_m03": q_m03,

                # Current values after any exact refinement.
                "p_value": p,
                "q_value": q,

                # Exact provenance.
                "exact_attempted": exact_attempted,
                "exact_refined": exact_refined,
                "exact_status": exact_status,
                "exact_reason": exact_reason,
                "exact_round": exact_round,
                "exact_negative_T": exact_negative_T,
                "p_value_exact": p_value_exact,
                "p_value_change": p_value_change,
                "abs_p_value_change": abs_p_value_change,
                "exact_T_raw": exact_T_raw,
                "exact_T": exact_T,
                "theta_F_exact": theta_F_exact,
                "theta_0_exact": theta_0_exact,

                # M03 branch/result details.
                "branch_m03": branch_m03,
                "branch": branch,
                "r_hat": r_hat,
                "theta_F": theta_F_final,
                "theta_0": theta_0_final,
                "theta_F_firstorder_DEB": theta_F_first,
                "theta_0_firstorder_DEB": theta_0_first,
                "ASH_null_weight": weights,
                "S_F": S_F,
                "S_0": S_0,
                "C_F_1": C_F,
                "C_0_1": C_0,
                "R_F_first": R_F,
                "S_0_dagger_current": current_S0dag,
                "S_0_min": S0_min,
                "support_floor_violation": support_violation,
                "S_0_dagger_final": S0dag_final,
                "R_0_dagger_first": R_0_dagger_first,
                "T_raw": T_raw,
                "T": T,
                "Q_F": Q_F,
                "Q_c": Q_c,
            })

            mandatory_success = int(
                np.sum(
                    mask_Rnonpos
                    & np.isfinite(p_m03)
                )
            )
            mandatory_failure = int(
                np.sum(mask_Rnonpos) - mandatory_success
            )
            mandatory_negative = int(
                np.sum(
                    exact_negative_T_m03
                    & mask_Rnonpos
                )
            )

            observed_diagnostics = {
                "method_name": method_name,
                "base_version": WEB_DNB_M03_BASE_VERSION,
                "code_version": WEB_DNB_M03_VERSION,
                "analysis_fingerprint": analysis_fingerprint,
                "r_estimator": r_estimator,
                "theta_method": theta_method,
                "geometry_method": geometry_method,
                "n_genes": int(G),
                "n_samples": int(n),
                "d_F": int(d_F),
                "d_0": int(d_0),
                "n_R_F_positive": int(mask_Rpos.sum()),
                "n_R_F_nonpositive": int(mask_Rnonpos.sum()),
                "n_m03_mandatory_exact_success": mandatory_success,
                "n_m03_mandatory_exact_failure": mandatory_failure,
                "n_m03_mandatory_exact_negative_T": mandatory_negative,
                "n_exact_attempted_total": n_exact_attempted_total,
                "n_exact_success_total": n_exact_success_total,
                "n_exact_failure_total": n_exact_failure_total,
                "n_exact_negative_T_total": n_exact_negative_total,
                "n_exact_computed_this_call": int(
                    len(computed_this_call)
                ),
                "n_exact_selected_this_call": int(
                    len(selected_this_call)
                ),
                "n_support_floor_violations": int(
                    support_violation.sum()
                ),
                "n_support_floor_violations_R_positive": int(
                    (support_violation & mask_Rpos).sum()
                ),
                "n_support_floor_violations_R_nonpositive": int(
                    (support_violation & mask_Rnonpos).sum()
                ),
                "all_support_feasible_after_correction": all_support_feasible,
                "n_finite_p": n_finite_p,
                "n_na_p": n_na_p,
                "kappa_F_source": "cprime1_identity_from_theta_F",
                "kappa_F_used": 1.0,
                "kappa_c": float(kappa_c),
                "lambda_c": float(lambda_c),
                "ASH_pi0": float(geom_stage["ash_pi0"]),
                "ASH_mean_null_weight": float(
                    np.mean(weights)
                ),
                "geometry_weight_sum": float(
                    geom_stage["geometry_weight_sum"]
                ),
                "full_theta_DEB_alpha0": float(
                    full_theta_fit.get("alpha0", np.nan)
                ),
                "full_theta_DEB_prior_mean": float(
                    full_theta_fit.get(
                        "prior_mean_theta",
                        np.nan,
                    )
                ),
                "null_theta_DEB_alpha0": float(
                    null_theta_fit.get("alpha0", np.nan)
                ),
                "null_theta_DEB_prior_mean": float(
                    null_theta_fit.get(
                        "prior_mean_theta",
                        np.nan,
                    )
                ),
                "firstorder_negative_T_fraction_Rpos": (
                    float(
                        np.mean(
                            np.asarray(
                                test_first["T_raw"]
                            )[mask_Rpos]
                            < 0
                        )
                    )
                    if np.any(mask_Rpos)
                    else np.nan
                ),
                "exact_negative_T_fraction_Rnonpos": (
                    float(
                        mandatory_negative
                        / max(mandatory_success, 1)
                    )
                ),
                "cprime_audit_max_abs_error": float(
                    full_stage["cprime_diagnostics"][
                        "audit_max_abs_error"
                    ]
                ),
                "cprime_build_seconds": float(
                    full_stage["cprime_diagnostics"].get(
                        "build_seconds",
                        np.nan,
                    )
                ),
                "cprime_grid_n_mu": int(
                    full_stage["cprime_diagnostics"].get(
                        "n_mu",
                        -1,
                    )
                ),
                "cprime_grid_n_r": int(
                    full_stage["cprime_diagnostics"].get(
                        "n_r",
                        -1,
                    )
                ),
                "cprime_grid_exact_node_count": int(
                    full_stage["cprime_diagnostics"].get(
                        "support_sum_count",
                        0,
                    )
                ),
                "cprime_grid_gamma_node_count": int(
                    full_stage["cprime_diagnostics"].get(
                        "gamma_continuum_count",
                        0,
                    )
                ),
                "cprime_grid_wide_exact_fallback_count": int(
                    full_stage["cprime_diagnostics"].get(
                        "wide_exact_fallback_count",
                        0,
                    )
                ),
                "cprime_grid_max_discrete_support_width": int(
                    full_stage["cprime_diagnostics"].get(
                        "max_discrete_support_width",
                        0,
                    )
                ),
                "cprime_grid_max_support_y": float(
                    full_stage["cprime_diagnostics"].get(
                        "max_support_y",
                        np.nan,
                    )
                ),
                "cprime_grid_continuum_policy": str(
                    full_stage["cprime_diagnostics"].get(
                        "grid_continuum_policy",
                        ""
                    )
                ),
                "cprime_grid_exact_when_feasible": True,
                "cprime_grid_gamma_only_beyond_support_cap": True,
                "cprime_grid_seam_safe_interpolation": True,
                "cprime_grid_mixed_exact_gamma_cells": int(
                    full_stage["cprime_diagnostics"].get(
                        "n_mixed_exact_gamma_cells",
                        0,
                    )
                ),
                "cprime_audit_mixed_cell_point_count": int(
                    full_stage["cprime_diagnostics"].get(
                        "audit_mixed_cell_point_count",
                        0,
                    )
                ),
                "cprime_audit_seam_branch_interp_count": int(
                    full_stage["cprime_diagnostics"].get(
                        "audit_seam_branch_interp_count",
                        0,
                    )
                ),
                "cprime_audit_seam_direct_fallback_count": int(
                    full_stage["cprime_diagnostics"].get(
                        "audit_seam_direct_fallback_count",
                        0,
                    )
                ),
                "cprime_production_mixed_cell_point_count": int(
                    full_stage["cprime_diagnostics"].get(
                        "production_mixed_cell_point_count",
                        0,
                    )
                ),
                "cprime_production_seam_branch_interp_count": int(
                    full_stage["cprime_diagnostics"].get(
                        "production_seam_branch_interp_count",
                        0,
                    )
                ),
                "cprime_production_seam_direct_fallback_count": int(
                    full_stage["cprime_diagnostics"].get(
                        "production_seam_direct_fallback_count",
                        0,
                    )
                ),
                "refinement_warnings": tuple(
                    refinement_warnings
                ),
                "fit_seconds": float(
                    perf_counter() - fit_t0
                ),
            }
            observed_diagnostics.update(
                refinement_diagnostics
            )

            refinement_state = {
                "analysis_fingerprint": analysis_fingerprint,
                "checkpoint_dir": str(checkpoint_path.resolve()),
                "base_version": WEB_DNB_M03_BASE_VERSION,
                "code_version": WEB_DNB_M03_VERSION,
                "random_seed": int(random_seed),
                "numerical_settings": dict(
                    numerical_settings
                ),
                "n_jobs_exact": int(n_jobs_exact),
                "exact_mode": exact_mode_used,
                "p_cutoff": p_cutoff,
                "fdr_cutoff": fdr_cutoff,
                "exact_top_n": exact_top_n,
                "exact_top_fraction": exact_top_fraction,
                "exact_gene_ids": (
                    tuple(exact_gene_ids)
                    if exact_gene_ids is not None
                    else None
                ),
                "n_exact_attempted_total": n_exact_attempted_total,
                "n_exact_success_total": n_exact_success_total,
                "max_exact_round": (
                    int(np.nanmax(exact_round))
                    if np.any(np.isfinite(exact_round))
                    else 0
                ),
            }

            result = {
                "method_name": method_name,
                "gene_results": gene_results,
                "kappa_F_source": "cprime1_identity_from_theta_F",
                "kappa_F_used": 1.0,
                "kappa_c": float(kappa_c),
                "lambda_c": float(lambda_c),
                "exact_mode": exact_mode_used,
                "diagnostics": observed_diagnostics,
                "refinement_state": refinement_state,
            }

            # Save a cheap snapshot for provenance only. It is NOT a global final-result
            # checkpoint and is never used to override a later refinement request.
            snapshot_key = hashlib.sha256(
                json.dumps(
                    {
                        "mode": exact_mode_used,
                        "p_cutoff": p_cutoff,
                        "fdr_cutoff": fdr_cutoff,
                        "exact_top_n": exact_top_n,
                        "exact_top_fraction": exact_top_fraction,
                        "exact_gene_ids": (
                            exact_gene_ids
                            if exact_gene_ids is not None
                            else None
                        ),
                        "n_exact_attempted": n_exact_attempted_total,
                        "fingerprint": analysis_fingerprint,
                    },
                    sort_keys=True,
                    default=str,
                ).encode("utf-8")
            ).hexdigest()[:16]

            robust_pickle_dump(
                {
                    "status": "success",
                    "data": {
                        "refinement_state": refinement_state,
                        "diagnostics": observed_diagnostics,
                        "exact_gene_ids": tuple(
                            gene_results.loc[
                                gene_results["exact_attempted"],
                                "gene_id",
                            ].astype(str)
                        ),
                    },
                },
                refinement_dir / f"refinement_{snapshot_key}.pkl",
            )

            return result

        return _run()
    finally:
        _ended = datetime.now().astimezone()
        print(f"[end]   {_ended.strftime('%Y-%m-%d %I:%M:%S %p %Z')}", flush=True)
